In [ ]:
# ============================================================
# ALBARAKA TÜRK
# FINANSMAN ACCESS TEST V1
#
# Amaç:
# Ana finansman sayfası / detay / alternatif erişim
# yöntemlerinden hangisi çalışıyor görmek.
# ============================================================

!pip -q install requests==2.32.4 beautifulsoup4

import requests
from bs4 import BeautifulSoup


# ============================================================
# TEST URLS
# ============================================================

TEST_URLS = {

    # --------------------------------------------------------
    # MAIN
    # --------------------------------------------------------

    "direct_main":
        (
            "https://www.albaraka.com.tr/"
            "tr/bireysel/finansmanlar"
        ),

    "jina_main":
        (
            "https://r.jina.ai/https://"
            "www.albaraka.com.tr/"
            "tr/bireysel/finansmanlar"
        ),

    # --------------------------------------------------------
    # DETAIL
    # --------------------------------------------------------

    "direct_konut":
        (
            "https://www.albaraka.com.tr/"
            "tr/bireysel/finansmanlar/"
            "konut-finansmani/"
            "konut-finansmani"
        ),

    "direct_tasit":
        (
            "https://www.albaraka.com.tr/"
            "tr/bireysel/finansmanlar/"
            "tasit-finansmani/"
            "tasit-finansmani"
        ),

    "direct_egitim":
        (
            "https://www.albaraka.com.tr/"
            "tr/bireysel/finansmanlar/"
            "ihtiyac/"
            "egitim-finansmani"
        ),

    "direct_online_alisveris":
        (
            "https://www.albaraka.com.tr/"
            "tr/bireysel/finansmanlar/"
            "ihtiyac/"
            "online-alisveris-finansmani"
        ),

    "direct_pratik_kart":
        (
            "https://www.albaraka.com.tr/"
            "tr/bireysel/finansmanlar/"
            "ihtiyac/"
            "pratik-finansman-kart"
        ),

    # --------------------------------------------------------
    # JET FINANSMAN SUBDOMAIN
    # --------------------------------------------------------

    "jet_finansman":
        (
            "https://basvur.albaraka.com.tr/"
            "jet-finansman"
        ),
}


# ============================================================
# SESSION
# ============================================================

session = requests.Session()

session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 "
        "(Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/151.0.0.0 "
        "Safari/537.36"
    ),
    "Accept-Language":
        "tr-TR,tr;q=0.9,en;q=0.8",
})


# ============================================================
# WAF CHECK
# ============================================================

WAF_MARKERS = [
    "request rejected",
    "the requested url was rejected",
    "support id",
    "access denied",
    "please consult with your administrator",
]


def clean_text(value):

    return " ".join(
        str(value or "")
        .replace("\xa0", " ")
        .split()
    )


def test_url(
    name,
    url
):

    print()
    print("=" * 115)
    print(name.upper())
    print("=" * 115)

    print(
        "URL    :",
        url
    )


    try:

        response = session.get(
            url,
            timeout=40,
            allow_redirects=True
        )


        html = response.text


        soup = BeautifulSoup(
            html,
            "html.parser"
        )


        text = clean_text(
            soup.get_text(
                " ",
                strip=True
            )
        )


        text_lower = (
            text.casefold()
        )


        waf = any(
            marker
            in text_lower
            for marker
            in WAF_MARKERS
        )


        title = ""

        if soup.title:

            title = clean_text(
                soup.title.get_text(
                    " ",
                    strip=True
                )
            )


        h1 = soup.find(
            "h1"
        )


        h1_text = (
            clean_text(
                h1.get_text(
                    " ",
                    strip=True
                )
            )
            if h1
            else ""
        )


        finance_markers = {
            "finansman":
                (
                    "finansman"
                    in text_lower
                ),

            "konut":
                (
                    "konut"
                    in text_lower
                ),

            "taşıt":
                (
                    "taşıt"
                    in text_lower
                    or
                    "tasit"
                    in text_lower
                ),

            "eğitim":
                (
                    "eğitim"
                    in text_lower
                    or
                    "egitim"
                    in text_lower
                ),

            "pratik":
                (
                    "pratik finansman"
                    in text_lower
                ),

            "jet":
                (
                    "jet finansman"
                    in text_lower
                ),
        }


        success = (
            response.status_code == 200
            and
            not waf
            and
            len(text) >= 150
            and
            any(
                finance_markers.values()
            )
        )


        print(
            "HTTP   :",
            response.status_code
        )

        print(
            "Length :",
            len(html)
        )

        print(
            "Text   :",
            len(text)
        )

        print(
            "Final  :",
            response.url
        )

        print(
            "Title  :",
            title[:150]
        )

        print(
            "H1     :",
            h1_text[:150]
        )

        print(
            "WAF    :",
            waf
        )

        print(
            "Markers:",
            finance_markers
        )

        print(
            "RESULT :",
            (
                "✅ ÇALIŞIYOR"
                if success
                else
                "❌ UYGUN DEĞİL"
            )
        )


        return {
            "name":
                name,

            "status":
                response.status_code,

            "waf":
                waf,

            "success":
                success,

            "html_length":
                len(html),

            "text_length":
                len(text),

            "title":
                title,

            "h1":
                h1_text,
        }


    except Exception as error:

        print(
            "ERROR  :",
            type(error).__name__,
            error
        )

        print(
            "RESULT : ❌"
        )


        return {
            "name":
                name,

            "status":
                None,

            "waf":
                False,

            "success":
                False,

            "error":
                str(error),
        }


# ============================================================
# RUN
# ============================================================

results = []


for name, url in (
    TEST_URLS.items()
):

    result = test_url(
        name,
        url
    )

    results.append(
        result
    )


# ============================================================
# SUMMARY
# ============================================================

print()
print("=" * 115)
print("ALBARAKA TÜRK - ACCESS TEST ÖZETİ")
print("=" * 115)


working = []


for result in results:

    symbol = (
        "✅"
        if result[
            "success"
        ]
        else "❌"
    )

    print(
        f"{result['name']:<25}: "
        f"{symbol} "
        f"| HTTP={result.get('status')} "
        f"| WAF={result.get('waf')}"
    )


    if result[
        "success"
    ]:

        working.append(
            result[
                "name"
            ]
        )


print()
print(
    "Çalışan yöntemler:",
    working
)


print()
print("=" * 115)


if working:

    print(
        "SONUÇ: EN AZ BİR "
        "ALBARAKA ERİŞİM YOLU VAR ✅"
    )

else:

    print(
        "SONUÇ: ANA SİTE "
        "HTTP YÖNTEMLERİ KAPALI ⚠️"
    )


print("=" * 115)


DIRECT_MAIN
URL    : https://www.albaraka.com.tr/tr/bireysel/finansmanlar
HTTP   : 200
Length : 196410
Text   : 8440
Final  : https://www.albaraka.com.tr/tr/bireysel/finansmanlar
Title  : Finansmanlar | Albaraka Türk Katılım Bankası
H1     : Finansmanlar
WAF    : False
Markers: {'finansman': True, 'konut': True, 'taşıt': True, 'eğitim': True, 'pratik': True, 'jet': True}
RESULT : ✅ ÇALIŞIYOR

JINA_MAIN
URL    : https://r.jina.ai/https://www.albaraka.com.tr/tr/bireysel/finansmanlar
HTTP   : 403
Length : 5876
Text   : 58
Final  : https://r.jina.ai/https://www.albaraka.com.tr/tr/bireysel/finansmanlar
Title  : Just a moment...
H1     : 
WAF    : False
Markers: {'finansman': False, 'konut': False, 'taşıt': False, 'eğitim': False, 'pratik': False, 'jet': False}
RESULT : ❌ UYGUN DEĞİL

DIRECT_KONUT
URL    : https://www.albaraka.com.tr/tr/bireysel/finansmanlar/konut-finansmani/konut-finansmani
HTTP   : 200
Length : 169022
Text   : 18563
Final  : https://www.albaraka.com.tr/tr/bireysel/finansm

In [ ]:
# ============================================================
# ALBARAKA TÜRK
# FINANSMAN RAW SCRAPER V1
#
# Input:
#   https://www.albaraka.com.tr/tr/bireysel/finansmanlar
#
# Output:
#   /content/albaraka_turk_finansmanlar_raw.json
#
# Beklenen ürün sayısı: 17
# ============================================================

!pip -q install requests==2.32.4 beautifulsoup4

import json
import re
import time

import requests

from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from google.colab import files


# ============================================================
# AYARLAR
# ============================================================

BASE = "https://www.albaraka.com.tr"

MAIN_URL = (
    BASE
    + "/tr/bireysel/finansmanlar"
)

OUTPUT_FILE = (
    "/content/"
    "albaraka_turk_finansmanlar_raw.json"
)

EXPECTED_COUNT = 17


# ============================================================
# RESMİ ANA SAYFADAKİ GÜNCEL ÜRÜNLER
#
# URL'LERİ elle vermiyoruz.
# Sayfadaki anchor'lardan otomatik buluyoruz.
# ============================================================

EXPECTED_PRODUCTS = {

    "Konut Finansmanı":
        "Konut Finansmanı",

    "Taşıt Finansmanı":
        "Taşıt Finansmanı",

    "Togg Finansmanı":
        "Taşıt Finansmanı",

    "Dijital Araç Finansmanı":
        "Taşıt Finansmanı",

    "Deniz Taşıtları Finansmanı":
        "Taşıt Finansmanı",

    "Taşıt Kiralama Finansmanı":
        "Taşıt Finansmanı",

    "İş Yeri Finansmanı":
        "Gayrimenkul Finansmanı",

    "Arsa Finansmanı":
        "Gayrimenkul Finansmanı",

    "2B Arazi Finansmanı":
        "Gayrimenkul Finansmanı",

    "Bayide Finansman":
        "Bayide Finansman",

    "Pratik Finansman Kart":
        "İhtiyaç Finansmanı",

    "SMS'li Finansman":
        "İhtiyaç Finansmanı",

    "Şubesiz Umre Finansmanı":
        "İhtiyaç Finansmanı",

    "Jet Finansman":
        "İhtiyaç Finansmanı",

    "Motosiklet, ATV, Bisiklet":
        "İhtiyaç Finansmanı",

    "Eğitim Finansmanı":
        "İhtiyaç Finansmanı",

    "BES Teminatlı Finansman":
        "BES Teminatlı Finansman",
}


# ============================================================
# SESSION
# ============================================================

session = requests.Session()

session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 "
        "(Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/151.0.0.0 "
        "Safari/537.36"
    ),
    "Accept-Language":
        "tr-TR,tr;q=0.9,en;q=0.8",
})


# ============================================================
# HELPERS
# ============================================================

def clean_text(value):

    value = str(
        value or ""
    )

    value = (
        value
        .replace("\xa0", " ")
        .replace("’", "'")
        .replace("‘", "'")
        .replace("–", "-")
        .replace("—", "-")
    )

    value = re.sub(
        r"\s+",
        " ",
        value
    )

    return value.strip()


def normalize(value):

    return (
        clean_text(value)
        .replace("İ", "i")
        .replace("I", "ı")
        .casefold()
    )


def normalize_product_name(value):

    value = clean_text(value)

    # Site varyasyonları
    value = re.sub(
        r"\s*'\s*",
        "'",
        value
    )

    value = re.sub(
        r"\s*,\s*",
        ", ",
        value
    )

    return value


def get_page(url):

    try:

        response = session.get(
            url,
            timeout=40,
            allow_redirects=True
        )

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        return response, soup

    except Exception as error:

        print(
            "REQUEST ERROR:",
            url,
            type(error).__name__,
            error
        )

        return None, None


# ============================================================
# PAGE CLEANER
# ============================================================

def extract_main_text(soup):

    clone = BeautifulSoup(
        str(soup),
        "html.parser"
    )

    for selector in [
        "script",
        "style",
        "noscript",
        "svg",
        "header",
        "footer",
        "nav",
        "form",
    ]:

        for tag in clone.select(
            selector
        ):

            tag.decompose()


    main = clone.find(
        "main"
    )


    if main is None:

        h1 = clone.find(
            "h1"
        )

        if h1:

            current = h1

            for _ in range(8):

                if current is None:
                    break

                text = clean_text(
                    current.get_text(
                        "\n",
                        strip=True
                    )
                )

                if len(text) >= 500:

                    main = current
                    break

                current = current.parent


    if main is None:

        main = clone.body


    if main is None:

        return ""


    return clean_text(
        main.get_text(
            "\n",
            strip=True
        )
    )


# ============================================================
# PRODUCT NAME MATCH
# ============================================================

ALIASES = {

    "sms'li finansman": [
        "sms'li finansman",
        "sms li finansman",
        "sms’li finansman",
        "sms’ li finansman",
        "sms' li finansman",
    ],

    "motosiklet, atv, bisiklet": [
        "motosiklet, atv, bisiklet",
        "motosiklet, atv , bisiklet",
        "motosiklet atv bisiklet",
    ],
}


def product_name_matches(
    expected,
    anchor_text
):

    e = normalize(
        expected
    )

    a = normalize(
        anchor_text
    )


    if e == a:
        return True


    aliases = ALIASES.get(
        e,
        []
    )


    return a in [
        normalize(x)
        for x in aliases
    ]


# ============================================================
# URL SCORING
#
# Aynı ürün navigation'da birden çok kez geçerse
# en uygun bireysel detail URL seçilir.
# ============================================================

def score_url(
    product_name,
    url
):

    parsed = urlparse(
        url
    )

    path = (
        parsed.path
        .rstrip("/")
    )


    score = 0


    if (
        parsed.netloc
        in {
            "albaraka.com.tr",
            "www.albaraka.com.tr",
        }
    ):

        score += 20


    if path.startswith(
        "/tr/bireysel/finansmanlar/"
    ):

        score += 100


    # Daha derin path genelde detail page.
    score += (
        path.count("/")
        * 3
    )


    # Category landing page'leri azalt.
    generic_paths = {
        "/tr/bireysel/finansmanlar/ihtiyac",
        "/tr/bireysel/finansmanlar/tasit-finansmani",
        "/tr/bireysel/finansmanlar/gayrimenkul-finansmani",
    }


    if path in generic_paths:

        score -= 80


    # Application URL olmasın.
    if (
        "basvur.albaraka.com.tr"
        in url
    ):

        score -= 100


    # Findeks vb. yok.
    if (
        "findeks"
        in path.casefold()
    ):

        score -= 100


    return score


# ============================================================
# FETCH MAIN
# ============================================================

print("=" * 118)
print(
    "ALBARAKA TÜRK - "
    "FINANSMAN RAW SCRAPER V1"
)
print("=" * 118)


response, soup = get_page(
    MAIN_URL
)


if (
    response is None
    or
    response.status_code != 200
):

    raise RuntimeError(
        "Ana finansman sayfası alınamadı."
    )


print(
    "MAIN HTTP:",
    response.status_code
)

print(
    "HTML len :",
    len(
        response.text
    )
)


# ============================================================
# DISCOVER ALL MATCHING ANCHORS
# ============================================================

all_anchors = []


for a in soup.find_all(
    "a",
    href=True
):

    text = normalize_product_name(
        a.get_text(
            " ",
            strip=True
        )
    )


    if not text:
        continue


    url = urljoin(
        MAIN_URL,
        a.get(
            "href",
            ""
        )
    )


    all_anchors.append(
        (
            text,
            url
        )
    )


# ============================================================
# CHOOSE URL FOR EACH PRODUCT
# ============================================================

inventory = []

missing = []


for product_name, category in (
    EXPECTED_PRODUCTS.items()
):

    candidates = []


    for anchor_text, url in all_anchors:

        if product_name_matches(
            product_name,
            anchor_text
        ):

            candidates.append(
                url
            )


    # Dedupe
    candidates = list(
        dict.fromkeys(
            candidates
        )
    )


    candidates.sort(
        key=lambda url:
            score_url(
                product_name,
                url
            ),
        reverse=True
    )


    if not candidates:

        missing.append(
            product_name
        )

        print()
        print(
            "❌ URL BULUNAMADI:",
            product_name
        )

        continue


    chosen = candidates[0]


    inventory.append({
        "urun_adi":
            product_name,

        "urun_kategorisi":
            category,

        "kaynak_url":
            chosen,

        "url_adaylari":
            candidates,
    })


# ============================================================
# INVENTORY PRINT
# ============================================================

print()
print("=" * 118)
print("ÜRÜN ENVANTERİ")
print("=" * 118)


for index, item in enumerate(
    inventory,
    start=1
):

    print(
        f"[{index:02d}] "
        f"{item['urun_adi']}"
    )

    print(
        "     Kategori:",
        item[
            "urun_kategorisi"
        ]
    )

    print(
        "     URL     :",
        item[
            "kaynak_url"
        ]
    )


print()
print(
    "Bulunan ürün:",
    len(inventory),
    "/",
    EXPECTED_COUNT
)

print(
    "Eksik ürün :",
    missing
)


# ============================================================
# FETCH DETAILS
# ============================================================

print()
print("=" * 118)
print("DETAY SCRAPE")
print("=" * 118)


records = []

failed = []

redirected = []


for index, item in enumerate(
    inventory,
    start=1
):

    product_name = item[
        "urun_adi"
    ]

    category = item[
        "urun_kategorisi"
    ]

    source_url = item[
        "kaynak_url"
    ]


    print()
    print(
        f"[{index:02d}/"
        f"{len(inventory):02d}] "
        f"{product_name}"
    )


    r, detail_soup = get_page(
        source_url
    )


    if (
        r is None
        or
        r.status_code != 200
    ):

        failed.append({
            "urun_adi":
                product_name,

            "kaynak_url":
                source_url,

            "status":
                (
                    r.status_code
                    if r is not None
                    else None
                ),
        })


        print(
            "  RESULT: ❌"
        )

        continue


    final_url = (
        r.url
        .split("#")[0]
    )


    if (
        final_url.rstrip("/")
        !=
        source_url.rstrip("/")
    ):

        redirected.append({
            "urun_adi":
                product_name,

            "from":
                source_url,

            "to":
                final_url,
        })


    h1 = detail_soup.find(
        "h1"
    )


    h1_text = (
        clean_text(
            h1.get_text(
                " ",
                strip=True
            )
        )
        if h1
        else ""
    )


    raw_text = extract_main_text(
        detail_soup
    )


    # --------------------------------------------------------
    # Detail validity
    # --------------------------------------------------------

    detail_valid = (
        len(raw_text) >= 150
        and
        "finansman"
        in normalize(
            raw_text
        )
    )


    # Category landing redirect kontrolü.
    bad_redirect = False


    if (
        product_name
        not in {
            "Bayide Finansman",
            "BES Teminatlı Finansman",
        }
        and
        normalize(
            product_name
        )
        not in normalize(
            h1_text
        )
    ):

        # Bazı H1'ler küçük isim farkı taşıyabilir,
        # dolayısıyla hemen reject etmiyoruz.
        if (
            normalize(h1_text)
            in {
                "finansmanlar",
                "ihtiyaç finansmanı",
                "taşıt finansmanı",
                "gayrimenkul finansmanı",
            }
            and
            normalize(
                product_name
            )
            != normalize(
                h1_text
            )
        ):

            bad_redirect = True


    if (
        not detail_valid
        or
        bad_redirect
    ):

        failed.append({
            "urun_adi":
                product_name,

            "kaynak_url":
                source_url,

            "final_url":
                final_url,

            "h1":
                h1_text,

            "status":
                r.status_code,

            "error":
                (
                    "detail_invalid"
                    if not detail_valid
                    else
                    "category_redirect"
                ),
        })


        print(
            "  HTTP  :",
            r.status_code
        )

        print(
            "  H1    :",
            h1_text
        )

        print(
            "  Final :",
            final_url
        )

        print(
            "  RESULT: ❌ DETAIL DEĞİL"
        )

        continue


    record = {
        "urun_adi":
            product_name,

        "urun_kategorisi":
            category,

        "kaynak_url":
            final_url,

        "h1":
            h1_text,

        "ham_metin":
            raw_text,
    }


    records.append(
        record
    )


    print(
        "  HTTP  :",
        r.status_code
    )

    print(
        "  H1    :",
        h1_text
    )

    print(
        "  Text  :",
        len(raw_text)
    )

    print(
        "  Final :",
        final_url
    )

    print(
        "  RESULT: ✅"
    )


    time.sleep(
        0.15
    )


# ============================================================
# DUPLICATES
# ============================================================

urls = [
    item[
        "kaynak_url"
    ]
    for item in records
]


duplicate_url = (
    len(urls)
    -
    len(
        set(urls)
    )
)


names = [
    item[
        "urun_adi"
    ]
    for item in records
]


duplicate_name = (
    len(names)
    -
    len(
        set(names)
    )
)


# ============================================================
# CATEGORY COUNT
# ============================================================

category_counts = {}


for item in records:

    category = item[
        "urun_kategorisi"
    ]

    category_counts[
        category
    ] = (
        category_counts.get(
            category,
            0
        )
        + 1
    )


# ============================================================
# EXPECTED CATEGORY DISTRIBUTION
# ============================================================

EXPECTED_CATEGORY_COUNTS = {
    "Konut Finansmanı":
        1,

    "Taşıt Finansmanı":
        5,

    "Gayrimenkul Finansmanı":
        3,

    "Bayide Finansman":
        1,

    "İhtiyaç Finansmanı":
        6,

    "BES Teminatlı Finansman":
        1,
}


errors = []


if missing:

    errors.append(
        (
            "Ana sayfada ürün URL'si "
            f"bulunamayanlar: {missing}"
        )
    )


if len(records) != EXPECTED_COUNT:

    errors.append(
        (
            f"RAW kayıt {len(records)} "
            f"!= {EXPECTED_COUNT}"
        )
    )


if failed:

    errors.append(
        (
            "Fetch/detail error: "
            f"{len(failed)}"
        )
    )


if duplicate_url:

    errors.append(
        (
            "Duplicate URL: "
            f"{duplicate_url}"
        )
    )


if duplicate_name:

    errors.append(
        (
            "Duplicate ürün: "
            f"{duplicate_name}"
        )
    )


if (
    category_counts
    != EXPECTED_CATEGORY_COUNTS
):

    errors.append(
        (
            "Kategori dağılımı yanlış: "
            f"{category_counts}"
        )
    )


# ============================================================
# SAVE
# ============================================================

output = {
    "banka":
        "Albaraka Türk Katılım Bankası A.Ş.",

    "ana_sayfa":
        MAIN_URL,

    "expected_count":
        EXPECTED_COUNT,

    "raw_count":
        len(records),

    "duplicate_url":
        duplicate_url,

    "duplicate_name":
        duplicate_name,

    "validation_error":
        len(errors),

    "kategori_dagilimi":
        category_counts,

    "finansmanlar":
        records,

    "failed":
        failed,

    "redirected":
        redirected,

    "errors":
        errors,
}


with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        output,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# FINAL AUDIT
# ============================================================

print()
print("=" * 118)
print(
    "ALBARAKA TÜRK - "
    "FINANSMAN RAW V1 SONUCU"
)
print("=" * 118)

print(
    "Beklenen ürün      :",
    EXPECTED_COUNT
)

print(
    "Inventory bulunan :",
    len(inventory)
)

print(
    "RAW başarılı      :",
    len(records)
)

print(
    "Fetch/detail error:",
    len(failed)
)

print(
    "Duplicate URL     :",
    duplicate_url
)

print(
    "Duplicate ürün    :",
    duplicate_name
)

print(
    "Validation error  :",
    len(errors)
)

print(
    "RAW JSON          :",
    OUTPUT_FILE
)


print()
print("=" * 118)
print("KATEGORİ DAĞILIMI")
print("=" * 118)


for category, expected in (
    EXPECTED_CATEGORY_COUNTS.items()
):

    print(
        f"{category:<30}: "
        f"{category_counts.get(category, 0)}"
        f" / {expected}"
    )


if redirected:

    print()
    print("=" * 118)
    print("REDIRECT OLANLAR")
    print("=" * 118)

    for item in redirected:

        print(
            "-",
            item[
                "urun_adi"
            ]
        )

        print(
            "  FROM:",
            item[
                "from"
            ]
        )

        print(
            "  TO  :",
            item[
                "to"
            ]
        )


if failed:

    print()
    print("=" * 118)
    print("FAILED")
    print("=" * 118)

    for item in failed:

        print(
            "-",
            item
        )


if errors:

    print()
    print("=" * 118)
    print("HATALAR")
    print("=" * 118)

    for error in errors:

        print(
            "-",
            error
        )


print()
print("=" * 118)


if not errors:

    print(
        "SONUÇ: ALBARAKA TÜRK "
        "FİNANSMAN RAW "
        "17/17 BAŞARILI ✅"
    )

else:

    print(
        "SONUÇ: ALBARAKA TÜRK "
        "FİNANSMAN RAW "
        "KONTROL GEREKİYOR ⚠️"
    )


print("=" * 118)


files.download(
    OUTPUT_FILE
)

ALBARAKA TÜRK - FINANSMAN RAW SCRAPER V1
MAIN HTTP: 200
HTML len : 196410

ÜRÜN ENVANTERİ
[01] Konut Finansmanı
     Kategori: Konut Finansmanı
     URL     : https://www.albaraka.com.tr/tr/bireysel/finansmanlar/konut-finansmani
[02] Taşıt Finansmanı
     Kategori: Taşıt Finansmanı
     URL     : https://www.albaraka.com.tr/tr/bireysel/finansmanlar/tasit-finansmani/tasit-finansmani
[03] Togg Finansmanı
     Kategori: Taşıt Finansmanı
     URL     : https://www.albaraka.com.tr/tr/bireysel/finansmanlar/tasit-finansmani/togg-finansmani
[04] Dijital Araç Finansmanı
     Kategori: Taşıt Finansmanı
     URL     : https://www.albaraka.com.tr/tr/bireysel/finansmanlar/tasit-finansmani/dijital-arac-finansmani
[05] Deniz Taşıtları Finansmanı
     Kategori: Taşıt Finansmanı
     URL     : https://www.albaraka.com.tr/tr/bireysel/finansmanlar/tasit-finansmani/deniz-tasitlari-finansmani
[06] Taşıt Kiralama Finansmanı
     Kategori: Taşıt Finansmanı
     URL     : https://www.albaraka.com.tr/tr/bireys

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# ALBARAKA TÜRK
# FINANSMAN SEMANTIC INSPECTOR V1
#
# Input:
#   /content/albaraka_turk_finansmanlar_raw.json
#
# Amaç:
# Final extractor öncesinde 17 ürünün
# finansal değerlerini semantik olarak görmek.
# ============================================================

import json
import re
from collections import Counter


INPUT_FILE = (
    "/content/"
    "albaraka_turk_finansmanlar_raw.json"
)


# ============================================================
# HELPERS
# ============================================================

def clean_text(value):

    value = str(value or "")

    value = (
        value
        .replace("\xa0", " ")
        .replace("’", "'")
        .replace("‘", "'")
        .replace("–", "-")
        .replace("—", "-")
    )

    value = re.sub(
        r"[ \t]+",
        " ",
        value
    )

    value = re.sub(
        r"\n[ \t]+",
        "\n",
        value
    )

    value = re.sub(
        r"\n{3,}",
        "\n\n",
        value
    )

    return value.strip()


def normalize(value):

    return (
        clean_text(value)
        .replace("İ", "i")
        .replace("I", "ı")
        .casefold()
    )


def unique(values):

    result = []
    seen = set()

    for value in values:

        value = clean_text(value)

        if not value:
            continue

        key = normalize(value)

        if key in seen:
            continue

        seen.add(key)
        result.append(value)

    return result


# ============================================================
# SENTENCES
# ============================================================

def sentences(text):

    text = clean_text(text)

    pieces = re.split(
        r"\n+|(?<=[.!?])\s+",
        text
    )

    return [
        clean_text(x)
        for x in pieces
        if clean_text(x)
    ]


# ============================================================
# FINANSMAN ORANI
#
# % değerini sadece oran semantiği taşıyan
# cümlelerle birlikte gösterir.
# ============================================================

def finance_ratio_candidates(text):

    result = []

    keywords = [
        "finansman oran",
        "ekspertiz",
        "değerinin %",
        "bedelinin %",
        "fatura bedelinin %",
        "finanse",
    ]

    for sentence in sentences(text):

        n = normalize(sentence)

        if not any(
            key in n
            for key in keywords
        ):
            continue

        percents = re.findall(
            r"%\s*\d+(?:[.,]\d+)?",
            sentence
        )

        if percents:

            result.append({
                "oranlar":
                    unique(
                        x.replace(" ", "")
                        for x in percents
                    ),

                "cumle":
                    sentence,
            })

    return result


# ============================================================
# KÂR PAYI ORANI
#
# Sadece açık numeric kâr payı ifadeleri.
# "uygun kâr payı oranı" numeric olmadığı için alınmaz.
# ============================================================

def profit_rate_candidates(text):

    result = []

    patterns = [
        (
            r"(?:kâr|kar)\s+payı\s+"
            r"oran(?:ı|ları)?"
            r"[^.!?\n]{0,80}?"
            r"(%\s*\d+(?:[.,]\d+)?)"
        ),

        (
            r"(%\s*\d+(?:[.,]\d+)?)"
            r"[^.!?\n]{0,80}?"
            r"(?:kâr|kar)\s+payı"
        ),
    ]

    for pattern in patterns:

        for match in re.finditer(
            pattern,
            text,
            flags=re.I
        ):

            value = next(
                (
                    group
                    for group
                    in match.groups()
                    if group
                    and group.startswith("%")
                ),
                None
            )

            if value:

                result.append(
                    value.replace(
                        " ",
                        ""
                    )
                )

    return unique(result)


# ============================================================
# MONEY
# ============================================================

def money_candidates(text):

    result = []

    patterns = [
        (
            r"\b"
            r"\d{1,3}"
            r"(?:[.\s]\d{3})+"
            r"(?:,\d+)?"
            r"\s*(?:TL|₺|USD|EUR)"
            r"\b"
        ),

        (
            r"\b"
            r"\d+"
            r"(?:,\d+)?"
            r"\s*(?:TL|₺|USD|EUR)"
            r"\b"
        ),
    ]

    for pattern in patterns:

        for match in re.finditer(
            pattern,
            text,
            flags=re.I
        ):

            value = clean_text(
                match.group(0)
            )

            value = value.replace(
                "₺",
                "TL"
            )

            value = re.sub(
                r"\s+",
                " ",
                value
            )

            result.append(value)

    return unique(result)


# ============================================================
# VADE
# ============================================================

def term_candidates(text):

    result = []

    patterns = [
        r"\b\d+\s+aya\s+kadar\b",
        r"\b\d+\s+aya\s+varan\b",
        r"\b\d+\s+ay\s+vade\b",
        r"\b\d+\s+aylık\s+vade\b",
        r"\b\d+\s+yıla\s+kadar\b",
        r"\b\d+\s+yıla\s+varan\b",
        r"\b\d+\s+yıl\s+vade\b",
    ]

    for pattern in patterns:

        for match in re.finditer(
            pattern,
            text,
            flags=re.I
        ):

            result.append(
                clean_text(
                    match.group(0)
                )
            )

    return unique(result)


# ============================================================
# TAKSIT
# ============================================================

def installment_candidates(text):

    result = []

    patterns = [
        r"\b\d+\s+taksit\b",
        r"\b\d+\s+taksitli\b",
        r"\b\d+\s+eşit\s+taksit\b",
    ]

    for pattern in patterns:

        for match in re.finditer(
            pattern,
            text,
            flags=re.I
        ):

            result.append(
                clean_text(
                    match.group(0)
                )
            )

    return unique(result)


# ============================================================
# MASRAF
# ============================================================

def fee_candidates(text):

    result = []

    keywords = [
        "tahsis ücreti",
        "dosya masraf",
        "finansman tahsis",
        "ücret alınmaz",
        "ücret alınmamaktadır",
        "masraf alınmaz",
        "masraf bulunmamaktadır",
        "komisyon",
    ]

    for sentence in sentences(text):

        n = normalize(sentence)

        if any(
            key in n
            for key in keywords
        ):

            result.append(sentence)

    return unique(result)


# ============================================================
# CURRENCY
# ============================================================

def currency_candidates(text):

    result = []

    if re.search(
        r"\bTL\b|₺|Türk Liras",
        text,
        flags=re.I
    ):
        result.append("TL")

    if re.search(
        r"\bUSD\b|Amerikan Dolar",
        text,
        flags=re.I
    ):
        result.append("USD")

    if re.search(
        r"\bEUR\b|Euro",
        text,
        flags=re.I
    ):
        result.append("EUR")

    return result


# ============================================================
# TARGET AUDIENCE CANDIDATE SENTENCES
# ============================================================

def target_candidates(text):

    result = []

    keywords = [
        "kimler yararlan",
        "kimler faydalan",
        "müşteriler",
        "müşterilerimiz",
        "18 yaş",
        "emekli",
        "öğrenci",
        "veli",
        "bes",
        "bireysel emeklilik",
        "hac",
        "umre",
    ]

    for sentence in sentences(text):

        n = normalize(sentence)

        if any(
            key in n
            for key in keywords
        ):

            if (
                20
                <= len(sentence)
                <= 450
            ):

                result.append(sentence)

    return unique(result)[:10]


# ============================================================
# LOAD
# ============================================================

with open(
    INPUT_FILE,
    "r",
    encoding="utf-8"
) as f:

    data = json.load(f)


records = data.get(
    "finansmanlar",
    []
)


print("=" * 125)
print(
    "ALBARAKA TÜRK - "
    "FINANSMAN SEMANTIC INSPECTOR V1"
)
print("=" * 125)

print(
    "Toplam ürün:",
    len(records)
)


# ============================================================
# COUNTERS
# ============================================================

with_ratio = 0
with_profit = 0
with_money = 0
with_term = 0
with_installment = 0
with_fee = 0
with_currency = 0


# ============================================================
# EACH PRODUCT
# ============================================================

for index, record in enumerate(
    records,
    start=1
):

    name = record.get(
        "urun_adi",
        ""
    )

    category = record.get(
        "urun_kategorisi",
        ""
    )

    text = record.get(
        "ham_metin",
        ""
    )


    ratios = finance_ratio_candidates(
        text
    )

    profit = profit_rate_candidates(
        text
    )

    money = money_candidates(
        text
    )

    terms = term_candidates(
        text
    )

    installments = installment_candidates(
        text
    )

    fees = fee_candidates(
        text
    )

    currencies = currency_candidates(
        text
    )

    targets = target_candidates(
        text
    )


    if ratios:
        with_ratio += 1

    if profit:
        with_profit += 1

    if money:
        with_money += 1

    if terms:
        with_term += 1

    if installments:
        with_installment += 1

    if fees:
        with_fee += 1

    if currencies:
        with_currency += 1


    print()
    print(
        f"[{index:02d}/"
        f"{len(records):02d}] "
        f"{name}"
    )

    print(
        "  Kategori :",
        category
    )

    print(
        "  Fin.Oran :",
        ratios
    )

    print(
        "  Kâr Payı :",
        profit
    )

    print(
        "  Tutar    :",
        money
    )

    print(
        "  Vade     :",
        terms
    )

    print(
        "  Taksit   :",
        installments
    )

    print(
        "  Masraf   :",
        fees[:5]
    )

    print(
        "  Para     :",
        currencies
    )

    print(
        "  Hedef    :",
        targets[:5]
    )


# ============================================================
# SUMMARY
# ============================================================

print()
print("=" * 125)
print("SEMANTIC AUDIT ÖZETİ")
print("=" * 125)

print(
    "Toplam ürün          :",
    len(records)
)

print(
    "Finansman oranı aday :",
    with_ratio
)

print(
    "Kâr payı numeric     :",
    with_profit
)

print(
    "Tutar bulunan        :",
    with_money
)

print(
    "Vade bulunan         :",
    with_term
)

print(
    "Taksit bulunan       :",
    with_installment
)

print(
    "Masraf bulunan       :",
    with_fee
)

print(
    "Para birimi bulunan  :",
    with_currency
)


# ============================================================
# CATEGORY DISTRIBUTION
# ============================================================

category_counts = Counter(
    record[
        "urun_kategorisi"
    ]
    for record in records
)


print()
print("KATEGORİ DAĞILIMI:")

for category, count in (
    category_counts.items()
):

    print(
        f"- {category}: {count}"
    )


print()
print("=" * 125)
print(
    "SEMANTIC INSPECTOR TAMAMLANDI ✅"
)
print("=" * 125)

ALBARAKA TÜRK - FINANSMAN SEMANTIC INSPECTOR V1
Toplam ürün: 17

[01/17] Konut Finansmanı
  Kategori : Konut Finansmanı
  Fin.Oran : []
  Kâr Payı : []
  Tutar    : ['9.169,06 TL', '210.888,82 TL', '4.457,06 TL', '4.712,00 TL', '150.000,00 TL', '4.744,55 TL', '4.424,51 TL', '145.542,94 TL', '4.888,79 TL', '4.280,27 TL', '140.798,39 TL', '5.037,41 TL', '4.131,65 TL', '135.909,60 TL', '5.190,55 TL', '3.978,51 TL', '130.872,19 TL', '5.348,34 TL', '3.820,72 TL', '125.681,64 TL', '5.510,93 TL', '3.658,13 TL', '120.333,30 TL', '5.678,46 TL', '3.490,60 TL', '114.822,37 TL', '5.851,09 TL', '3.317,97 TL', '109.143,91 TL', '6.028,96 TL', '3.140,10 TL', '103.292,82 TL', '6.212,24 TL', '2.956,82 TL', '97.263,86 TL', '6.401,09 TL', '2.767,97 TL', '91.051,62 TL', '6.595,68 TL', '2.573,38 TL', '84.650,53 TL', '6.796,19 TL', '2.372,87 TL', '78.054,85 TL', '7.002,80 TL', '2.166,26 TL', '71.258,66 TL', '7.215,68 TL', '1.953,38 TL', '64.255,86 TL', '7.435,04 TL', '1.734,02 TL', '57.040,18 TL', '7.661,06 

In [ ]:
# ============================================================
# ALBARAKA TÜRK
# FINANSMAN EXTRACTOR FINAL V1
#
# Input:
#   /content/albaraka_turk_finansmanlar_raw.json
#
# Output:
#   /content/albaraka_turk_finansman_extracted.json
#
# Hedef:
#   17 / 17
#   ortak 18-key schema
# ============================================================

import json
import re
from collections import Counter

from google.colab import files


# ============================================================
# FILES
# ============================================================

INPUT_FILE = (
    "/content/"
    "albaraka_turk_finansmanlar_raw.json"
)

OUTPUT_FILE = (
    "/content/"
    "albaraka_turk_finansman_extracted.json"
)

BANK_NAME = (
    "Albaraka Türk Katılım Bankası A.Ş."
)


# ============================================================
# SCHEMA
# ============================================================

SCHEMA_KEYS = [
    "banka",
    "kayit_turu",
    "urun_adi",
    "urun_kategorisi",
    "kar_payi_orani",
    "finansman_orani",
    "finansman_tutari",
    "vade",
    "taksit_sayisi",
    "masraf_bilgisi",
    "kampanya_turu",
    "kampanya_avantaji",
    "kampanya_suresi",
    "hedef_kitle",
    "para_birimi",
    "kosullar",
    "kaynak_url",
    "ham_metin",
]


LIST_FIELDS = {
    "kar_payi_orani",
    "finansman_orani",
    "finansman_tutari",
    "vade",
    "taksit_sayisi",
    "masraf_bilgisi",
    "kampanya_avantaji",
    "hedef_kitle",
    "para_birimi",
    "kosullar",
}


SCALAR_FIELDS = (
    set(SCHEMA_KEYS)
    - LIST_FIELDS
)


# ============================================================
# HELPERS
# ============================================================

def clean_text(value):

    value = str(value or "")

    value = (
        value
        .replace("\xa0", " ")
        .replace("’", "'")
        .replace("‘", "'")
        .replace("–", "-")
        .replace("—", "-")
        .replace("\u00ad", "")
    )

    value = re.sub(
        r"[ \t]+",
        " ",
        value
    )

    value = re.sub(
        r"\n[ \t]+",
        "\n",
        value
    )

    value = re.sub(
        r"\n{3,}",
        "\n\n",
        value
    )

    return value.strip()


def normalize(value):

    return (
        clean_text(value)
        .replace("İ", "i")
        .replace("I", "ı")
        .casefold()
    )


def unique(values):

    result = []
    seen = set()

    for value in values:

        value = clean_text(value)

        if not value:
            continue

        key = normalize(value)

        if key in seen:
            continue

        seen.add(key)
        result.append(value)

    return result


def has(text, phrase):

    return (
        normalize(phrase)
        in normalize(text)
    )


# ============================================================
# MONEY NORMALIZATION
# ============================================================

def normalize_money(value):

    value = clean_text(value)

    value = value.replace(
        "₺",
        "TL"
    )

    value = re.sub(
        r"\s+TL\b",
        " TL",
        value,
        flags=re.I
    )

    return value


# ============================================================
# PRODUCT-SPECIFIC FINANCE RULES
#
# Bunlar ürün sayfasındaki açık kurallar.
# Hesaplama simülasyonlarından alınmıyor.
# ============================================================

FINANCE_RATIO_OVERRIDES = {

    "Taşıt Finansmanı": [
        "%70",
        "%50",
        "%30",
        "%20",
        "%0",
    ],

    "Togg Finansmanı": [
        "%70",
    ],

    "Dijital Araç Finansmanı": [
        "%70",
        "%50",
        "%30",
        "%20",
        "%0",
    ],

    "İş Yeri Finansmanı": [
        "%100",
    ],

    "Arsa Finansmanı": [
        "%100",
    ],
}


TERM_OVERRIDES = {

    "Konut Finansmanı": [
        "120 aya kadar",
    ],

    # Taşıt değerine göre üst sınırlar.
    "Taşıt Finansmanı": [
        "48 ay",
        "36 ay",
        "24 ay",
        "12 ay",
    ],

    "Togg Finansmanı": [
        "48 aya kadar",
    ],

    "Dijital Araç Finansmanı": [
        "48 ay",
        "36 ay",
        "24 ay",
        "12 ay",
    ],

    "İş Yeri Finansmanı": [
        "60 aya kadar",
    ],

    "Arsa Finansmanı": [
        "60 aya kadar",
    ],

    "2B Arazi Finansmanı": [
        "60 aya kadar",
    ],

    "Bayide Finansman": [
        "36 aya kadar",
    ],

    "Jet Finansman": [
        "36 aya kadar",
    ],

    "Motosiklet, ATV, Bisiklet": [
        "36 ay",
        "24 ay",
        "12 ay",
    ],

    "Eğitim Finansmanı": [
        "12 aya kadar",
    ],
}


INSTALLMENT_OVERRIDES = {

    "Şubesiz Umre Finansmanı": [
        "4",
    ],

    "Jet Finansman": [
        "12",
        "6",
    ],
}


AMOUNT_OVERRIDES = {

    # Açık ürün limiti.
    "Şubesiz Umre Finansmanı": [
        "50.000 TL",
    ],
}


# ============================================================
# VEHICLE TABLE CONDITIONS
# ============================================================

VEHICLE_TABLE_CONDITIONS = [

    (
        "Kasko/Satış değeri 0-400.000 TL "
        "olan taşıtlarda finansman oranı "
        "%70 ve vade üst sınırı 48 aydır."
    ),

    (
        "Kasko/Satış değeri "
        "400.001-800.000 TL olan taşıtlarda "
        "finansman oranı %50 ve vade üst "
        "sınırı 36 aydır."
    ),

    (
        "Kasko/Satış değeri "
        "800.001-1.200.000 TL olan "
        "taşıtlarda finansman oranı %30 "
        "ve vade üst sınırı 24 aydır."
    ),

    (
        "Kasko/Satış değeri "
        "1.200.001-2.000.000 TL olan "
        "taşıtlarda finansman oranı %20 "
        "ve vade üst sınırı 12 aydır."
    ),

    (
        "Kasko/Satış değeri "
        "2.000.001 TL ve üzerindeki "
        "taşıtlarda finansman oranı %0 "
        "olup kullandırım yapılmaz."
    ),
]


# ============================================================
# SENTENCE EXTRACTION
# ============================================================

NOISE_MARKERS = [
    "aylık taksit tutarı",
    "geri ödenecek toplam",
    "taksitler toplamı",
    "ödeme planı",
    "kalan ana para",
    "yıllık maliyet oranı",
    "hesaplama türü",
    "hesaplama aracı",
    "hesaplama sonucu",
    "proforma fatura tutarı",
    "size özel",
]


def sentences(text):

    text = clean_text(text)

    pieces = re.split(
        r"\n+|(?<=[.!?])\s+",
        text
    )

    result = []

    for piece in pieces:

        piece = clean_text(piece)

        if not piece:
            continue

        n = normalize(piece)

        if any(
            marker in n
            for marker in NOISE_MARKERS
        ):
            continue

        # Menü/navigation satırlarını at.
        if (
            "anasayfa bireysel finansmanlar"
            in n
            and
            len(piece) > 200
        ):
            continue

        result.append(piece)

    return result


# ============================================================
# EXPLICIT FINANCING AMOUNTS
#
# Hesaplama tablosundaki taksit/anapara değerleri alınmaz.
# Yalnızca açık limit / azami finansman cümleleri.
# ============================================================

def extract_explicit_amounts(
    name,
    text
):

    if name in AMOUNT_OVERRIDES:

        return list(
            AMOUNT_OVERRIDES[name]
        )


    result = []

    include_markers = [
        "finansman tutarı",
        "finansman limiti",
        "azami finansman",
        "maksimum finansman",
        "en fazla",
        "finansman desteği",
        "kadar finansman",
    ]


    exclude_markers = [
        "kasko/satış değeri",
        "taşıt tutarı",
        "ekspertiz değeri",
        "aylık taksit",
        "geri ödeme",
        "taksit tutarı",
        "ödeme planı",
    ]


    pattern = (
        r"\b"
        r"\d{1,3}"
        r"(?:\.\d{3})+"
        r"(?:,\d+)?"
        r"\s*(?:TL|₺)"
        r"\b"
    )


    for sentence in sentences(text):

        n = normalize(sentence)

        if not any(
            marker in n
            for marker in include_markers
        ):
            continue

        if any(
            marker in n
            for marker in exclude_markers
        ):
            continue


        for match in re.finditer(
            pattern,
            sentence,
            flags=re.I
        ):

            result.append(
                normalize_money(
                    match.group(0)
                )
            )


    return unique(result)


# ============================================================
# PROFIT SHARE RATE
#
# Numeric oran dışında çıkarım YOK.
# "kâr paysız" -> %0 YAPILMAZ.
# ============================================================

def extract_profit_rates(text):

    result = []


    patterns = [

        (
            r"(?:kâr|kar)\s+payı\s+"
            r"oran(?:ı|ları)?"
            r"[^.!?\n]{0,50}?"
            r"(%\s*\d+(?:[.,]\d+)?)"
        ),

        (
            r"(%\s*\d+(?:[.,]\d+)?)"
            r"[^.!?\n]{0,50}?"
            r"(?:kâr|kar)\s+payı"
        ),
    ]


    for sentence in sentences(text):

        n = normalize(sentence)

        # Dinamik hesaplayıcı değeri istemiyoruz.
        if (
            "size özel"
            in n
            or
            "hesapla"
            in n
            or
            "aylık taksit"
            in n
        ):
            continue


        for pattern in patterns:

            for match in re.finditer(
                pattern,
                sentence,
                flags=re.I
            ):

                values = [
                    group
                    for group
                    in match.groups()
                    if (
                        group
                        and
                        group.strip()
                        .startswith("%")
                    )
                ]

                result.extend(
                    x.replace(
                        " ",
                        ""
                    )
                    for x in values
                )


    return unique(result)


# ============================================================
# FEE
# ============================================================

def extract_fees(text):

    result = []

    markers = [
        "tahsis ücreti",
        "dosya masraf",
        "masraf alınmaz",
        "masraf bulunmamaktadır",
        "ücret alınmaz",
        "ücret alınmamaktadır",
        "komisyon",
    ]


    for sentence in sentences(text):

        n = normalize(sentence)

        if (
            "hesaplama"
            in n
            or
            "size özel"
            in n
        ):
            continue


        if any(
            marker in n
            for marker in markers
        ):

            result.append(sentence)


    return unique(result)


# ============================================================
# TARGET AUDIENCE
# ============================================================

TARGET_OVERRIDES = {

    "Togg Finansmanı": [
        "Bireysel müşteriler",
    ],

    "Taşıt Kiralama Finansmanı": [
        "Gerçek kişi müşteriler",
        "18 yaşını tamamlamış müşteriler",
    ],

    "Deniz Taşıtları Finansmanı": [
        "18 yaşını tamamlamış müşteriler",
    ],

    "İş Yeri Finansmanı": [
        "18 yaşını tamamlamış müşteriler",
    ],

    "Arsa Finansmanı": [
        "18 yaşını tamamlamış müşteriler",
    ],

    "2B Arazi Finansmanı": [
        "18 yaşını tamamlamış müşteriler",
    ],

    "Şubesiz Umre Finansmanı": [
        "Albaraka müşterileri",
    ],

    "Motosiklet, ATV, Bisiklet": [
        "Bireysel müşteriler",
        "18 yaşını doldurmuş kişiler",
    ],

    "Eğitim Finansmanı": [
        "Eğitim giderlerini finanse etmek isteyen bireysel müşteriler",
        "Gelir belgesi bulunmayan öğrenciler için veli veya vasi",
    ],

    "BES Teminatlı Finansman": [
        "BES birikimi bulunan müşteriler",
    ],
}


def extract_target(name, text):

    if name in TARGET_OVERRIDES:

        return list(
            TARGET_OVERRIDES[name]
        )


    result = []


    for sentence in sentences(text):

        n = normalize(sentence)

        if (
            "18 yaşını tamamladıysanız"
            in n
            or
            "18 yaşını doldur"
            in n
        ):

            result.append(
                "18 yaşını tamamlamış müşteriler"
            )


        if (
            "sadece bireysel müşteriler"
            in n
        ):

            result.append(
                "Bireysel müşteriler"
            )


    return unique(result)


# ============================================================
# CURRENCY
# ============================================================

def extract_currency(
    name,
    text,
    amounts
):

    result = []


    if any(
        "TL"
        in value.upper()
        for value in amounts
    ):

        result.append("TL")


    # Ürün kural tablolarında TL açıkça mevcut.
    if name in {
        "Taşıt Finansmanı",
        "Dijital Araç Finansmanı",
        "Motosiklet, ATV, Bisiklet",
    }:

        result.append("TL")


    if re.search(
        r"\bUSD\b",
        text,
        flags=re.I
    ):

        # Yalnızca ürünün gerçek finansman para birimi
        # olduğu açık bir cümle varsa ekle.
        for sentence in sentences(text):

            if (
                re.search(
                    r"\bUSD\b",
                    sentence,
                    flags=re.I
                )
                and
                "finansman"
                in normalize(sentence)
            ):

                result.append("USD")
                break


    if re.search(
        r"\bEUR\b",
        text,
        flags=re.I
    ):

        for sentence in sentences(text):

            if (
                re.search(
                    r"\bEUR\b",
                    sentence,
                    flags=re.I
                )
                and
                "finansman"
                in normalize(sentence)
            ):

                result.append("EUR")
                break


    return unique(result)


# ============================================================
# CONDITIONS
# ============================================================

CONDITION_MARKERS = [
    "finansman",
    "vade",
    "ekspertiz",
    "başvuru",
    "yararlan",
    "faydalan",
    "18 yaş",
    "proforma",
    "teminat",
    "taksit",
    "oran",
    "kasko",
    "satış değeri",
    "bireysel müşteri",
    "gerçek kişi",
    "belge",
    "gelir",
    "diyanet",
    "umre",
    "bes",
]


def extract_conditions(
    name,
    text
):

    result = []


    for sentence in sentences(text):

        n = normalize(sentence)


        if not any(
            marker in n
            for marker in CONDITION_MARKERS
        ):
            continue


        if (
            len(sentence) < 20
            or
            len(sentence) > 700
        ):
            continue


        # Başvuru CTA / navigasyon.
        if (
            "size en yakın şube"
            in n
            and
            len(sentence) < 80
        ):
            continue


        result.append(sentence)


    # Araç oran tablosu, HTML'de tek satıra karıştığı için
    # temiz ve anlamlı biçimde koşullara ekleniyor.
    if name in {
        "Taşıt Finansmanı",
        "Dijital Araç Finansmanı",
    }:

        result.extend(
            VEHICLE_TABLE_CONDITIONS
        )


    return unique(result)[:35]


# ============================================================
# EXTRACT RECORD
# ============================================================

def extract_record(raw):

    name = clean_text(
        raw.get(
            "urun_adi",
            ""
        )
    )

    category = clean_text(
        raw.get(
            "urun_kategorisi",
            ""
        )
    )

    source_url = clean_text(
        raw.get(
            "kaynak_url",
            ""
        )
    )

    raw_text = str(
        raw.get(
            "ham_metin",
            ""
        )
    )


    amounts = extract_explicit_amounts(
        name,
        raw_text
    )


    profit_rates = (
        extract_profit_rates(
            raw_text
        )
    )


    finance_ratios = list(
        FINANCE_RATIO_OVERRIDES.get(
            name,
            []
        )
    )


    terms = list(
        TERM_OVERRIDES.get(
            name,
            []
        )
    )


    installments = list(
        INSTALLMENT_OVERRIDES.get(
            name,
            []
        )
    )


    fees = extract_fees(
        raw_text
    )


    target = extract_target(
        name,
        raw_text
    )


    currencies = extract_currency(
        name,
        raw_text,
        amounts
    )


    conditions = extract_conditions(
        name,
        raw_text
    )


    return {

        "banka":
            BANK_NAME,

        "kayit_turu":
            "finansman",

        "urun_adi":
            name,

        "urun_kategorisi":
            category,

        "kar_payi_orani":
            profit_rates,

        "finansman_orani":
            finance_ratios,

        "finansman_tutari":
            amounts,

        "vade":
            terms,

        "taksit_sayisi":
            installments,

        "masraf_bilgisi":
            fees,

        "kampanya_turu":
            "",

        "kampanya_avantaji":
            [],

        "kampanya_suresi":
            "",

        "hedef_kitle":
            target,

        "para_birimi":
            currencies,

        "kosullar":
            conditions,

        "kaynak_url":
            source_url,

        "ham_metin":
            raw_text,
    }


# ============================================================
# LOAD
# ============================================================

with open(
    INPUT_FILE,
    "r",
    encoding="utf-8"
) as f:

    data = json.load(f)


raw_records = data.get(
    "finansmanlar",
    []
)


print("=" * 120)
print(
    "ALBARAKA TÜRK - "
    "FINANSMAN EXTRACTOR FINAL V1"
)
print("=" * 120)

print(
    "RAW ürün:",
    len(raw_records)
)


# ============================================================
# EXTRACT
# ============================================================

records = []

errors = []


for index, raw in enumerate(
    raw_records,
    start=1
):

    try:

        record = extract_record(
            raw
        )

        records.append(record)

    except Exception as error:

        errors.append(
            (
                f"[{index}] "
                f"{type(error).__name__}: "
                f"{error}"
            )
        )


# ============================================================
# SCHEMA VALIDATION
# ============================================================

for index, record in enumerate(
    records,
    start=1
):

    name = record[
        "urun_adi"
    ]


    if list(
        record.keys()
    ) != SCHEMA_KEYS:

        errors.append(
            (
                f"{name} -> "
                "schema/order hatası"
            )
        )


    for field in LIST_FIELDS:

        if not isinstance(
            record[field],
            list
        ):

            errors.append(
                (
                    f"{name} -> "
                    f"{field} list değil"
                )
            )


    for field in SCALAR_FIELDS:

        if not isinstance(
            record[field],
            str
        ):

            errors.append(
                (
                    f"{name} -> "
                    f"{field} string değil"
                )
            )


    if record[
        "banka"
    ] != BANK_NAME:

        errors.append(
            f"{name} -> banka yanlış"
        )


    if record[
        "kayit_turu"
    ] != "finansman":

        errors.append(
            (
                f"{name} -> "
                "kayit_turu yanlış"
            )
        )


    for field in [
        "urun_adi",
        "urun_kategorisi",
        "kaynak_url",
        "ham_metin",
    ]:

        if not record[field]:

            errors.append(
                (
                    f"{name} -> "
                    f"{field} boş"
                )
            )


    # Finansman kayıtlarında kampanya alanları boş.
    if record["kampanya_turu"]:

        errors.append(
            (
                f"{name} -> "
                "kampanya_turu boş olmalı"
            )
        )


    if record["kampanya_avantaji"]:

        errors.append(
            (
                f"{name} -> "
                "kampanya_avantaji boş olmalı"
            )
        )


    if record["kampanya_suresi"]:

        errors.append(
            (
                f"{name} -> "
                "kampanya_suresi boş olmalı"
            )
        )


    if "TRY" in record[
        "para_birimi"
    ]:

        errors.append(
            (
                f"{name} -> "
                "TRY yerine TL kullanılmalı"
            )
        )


# ============================================================
# COUNT / DUPLICATE
# ============================================================

if len(records) != 17:

    errors.append(
        (
            f"Extracted {len(records)} "
            "!= 17"
        )
    )


urls = [
    record[
        "kaynak_url"
    ]
    for record in records
]


duplicate_url = (
    len(urls)
    -
    len(set(urls))
)


names = [
    record[
        "urun_adi"
    ]
    for record in records
]


duplicate_name = (
    len(names)
    -
    len(set(names))
)


if duplicate_url:

    errors.append(
        (
            "Duplicate URL: "
            f"{duplicate_url}"
        )
    )


if duplicate_name:

    errors.append(
        (
            "Duplicate ürün: "
            f"{duplicate_name}"
        )
    )


# ============================================================
# CRITICAL CHECKS
# ============================================================

by_name = {
    record[
        "urun_adi"
    ]:
    record
    for record in records
}


def require(
    name,
    field,
    expected
):

    record = by_name.get(
        name
    )


    if record is None:

        errors.append(
            (
                f"{name} "
                "bulunamadı"
            )
        )

        return


    actual = record[field]


    if actual != expected:

        errors.append(
            (
                f"{name} -> "
                f"{field}: "
                f"{actual} "
                f"!= {expected}"
            )
        )


# ------------------------------------------------------------
# KONUT
# ------------------------------------------------------------

require(
    "Konut Finansmanı",
    "vade",
    ["120 aya kadar"]
)


# ------------------------------------------------------------
# TAŞIT
# ------------------------------------------------------------

require(
    "Taşıt Finansmanı",
    "finansman_orani",
    [
        "%70",
        "%50",
        "%30",
        "%20",
        "%0",
    ]
)

require(
    "Taşıt Finansmanı",
    "vade",
    [
        "48 ay",
        "36 ay",
        "24 ay",
        "12 ay",
    ]
)

# Taşıt satış değeri finansman_tutari DEĞİLDİR.
require(
    "Taşıt Finansmanı",
    "finansman_tutari",
    []
)


# ------------------------------------------------------------
# TOGG
# ------------------------------------------------------------

require(
    "Togg Finansmanı",
    "finansman_orani",
    ["%70"]
)

require(
    "Togg Finansmanı",
    "vade",
    ["48 aya kadar"]
)


# ------------------------------------------------------------
# DİJİTAL ARAÇ
# ------------------------------------------------------------

require(
    "Dijital Araç Finansmanı",
    "finansman_orani",
    [
        "%70",
        "%50",
        "%30",
        "%20",
        "%0",
    ]
)

require(
    "Dijital Araç Finansmanı",
    "vade",
    [
        "48 ay",
        "36 ay",
        "24 ay",
        "12 ay",
    ]
)

require(
    "Dijital Araç Finansmanı",
    "finansman_tutari",
    []
)


# ------------------------------------------------------------
# İŞ YERİ
# ------------------------------------------------------------

require(
    "İş Yeri Finansmanı",
    "finansman_orani",
    ["%100"]
)

require(
    "İş Yeri Finansmanı",
    "vade",
    ["60 aya kadar"]
)


# ------------------------------------------------------------
# ARSA
# ------------------------------------------------------------

require(
    "Arsa Finansmanı",
    "finansman_orani",
    ["%100"]
)

require(
    "Arsa Finansmanı",
    "vade",
    ["60 aya kadar"]
)


# ------------------------------------------------------------
# 2B
# ------------------------------------------------------------

require(
    "2B Arazi Finansmanı",
    "vade",
    ["60 aya kadar"]
)


# ------------------------------------------------------------
# BAYİDE
# ------------------------------------------------------------

require(
    "Bayide Finansman",
    "vade",
    ["36 aya kadar"]
)


# ------------------------------------------------------------
# ŞUBESİZ UMRE
#
# "kâr paysız" ifadesinden %0 türetilmez.
# ------------------------------------------------------------

require(
    "Şubesiz Umre Finansmanı",
    "finansman_tutari",
    ["50.000 TL"]
)

require(
    "Şubesiz Umre Finansmanı",
    "taksit_sayisi",
    ["4"]
)

require(
    "Şubesiz Umre Finansmanı",
    "kar_payi_orani",
    []
)


# ------------------------------------------------------------
# JET
# ------------------------------------------------------------

require(
    "Jet Finansman",
    "vade",
    ["36 aya kadar"]
)

require(
    "Jet Finansman",
    "taksit_sayisi",
    ["12", "6"]
)


# ------------------------------------------------------------
# MOTOSİKLET
#
# 125.000 / 250.000 TL ürün fiyat eşikleridir,
# finansman tutarı değildir.
# ------------------------------------------------------------

require(
    "Motosiklet, ATV, Bisiklet",
    "vade",
    [
        "36 ay",
        "24 ay",
        "12 ay",
    ]
)

require(
    "Motosiklet, ATV, Bisiklet",
    "finansman_tutari",
    []
)


# ------------------------------------------------------------
# EĞİTİM
# ------------------------------------------------------------

require(
    "Eğitim Finansmanı",
    "vade",
    ["12 aya kadar"]
)


# ============================================================
# CATEGORY DISTRIBUTION
# ============================================================

category_counts = Counter(
    record[
        "urun_kategorisi"
    ]
    for record in records
)


EXPECTED_CATEGORIES = {
    "Konut Finansmanı":
        1,

    "Taşıt Finansmanı":
        5,

    "Gayrimenkul Finansmanı":
        3,

    "Bayide Finansman":
        1,

    "İhtiyaç Finansmanı":
        6,

    "BES Teminatlı Finansman":
        1,
}


if dict(
    category_counts
) != EXPECTED_CATEGORIES:

    errors.append(
        (
            "Kategori dağılımı yanlış: "
            f"{dict(category_counts)}"
        )
    )


# ============================================================
# SAVE
#
# Wrapper yok.
# Direkt 17 kayıtlık JSON listesi.
# ============================================================

with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        records,
        f,
        ensure_ascii=False,
        indent=4
    )


# ============================================================
# AUDIT
# ============================================================

print()
print("=" * 120)
print(
    "ALBARAKA TÜRK - "
    "FINANSMAN FINAL AUDIT"
)
print("=" * 120)

print(
    "RAW ürün          :",
    len(raw_records)
)

print(
    "Extracted         :",
    len(records)
)

print(
    "Duplicate URL     :",
    duplicate_url
)

print(
    "Duplicate ürün    :",
    duplicate_name
)

print(
    "Validation error  :",
    len(errors)
)

print(
    "Final JSON        :",
    OUTPUT_FILE
)


print()
print("=" * 120)
print("KATEGORİ DAĞILIMI")
print("=" * 120)


for key, expected in (
    EXPECTED_CATEGORIES.items()
):

    print(
        f"{key:<30}: "
        f"{category_counts.get(key, 0)}"
        f" / {expected}"
    )


# ============================================================
# VALUES
# ============================================================

print()
print("=" * 120)
print("FİNANSMAN DEĞERLERİ")
print("=" * 120)


for record in records:

    print()
    print(
        record[
            "urun_adi"
        ]
    )

    print(
        "  Kâr Payı :",
        record[
            "kar_payi_orani"
        ]
    )

    print(
        "  Fin.Oran :",
        record[
            "finansman_orani"
        ]
    )

    print(
        "  Tutar    :",
        record[
            "finansman_tutari"
        ]
    )

    print(
        "  Vade     :",
        record[
            "vade"
        ]
    )

    print(
        "  Taksit   :",
        record[
            "taksit_sayisi"
        ]
    )

    print(
        "  Masraf   :",
        record[
            "masraf_bilgisi"
        ]
    )

    print(
        "  Para     :",
        record[
            "para_birimi"
        ]
    )

    print(
        "  Hedef    :",
        record[
            "hedef_kitle"
        ]
    )

    print(
        "  Koşul    :",
        len(
            record[
                "kosullar"
            ]
        )
    )


# ============================================================
# ERRORS
# ============================================================

if errors:

    print()
    print("=" * 120)
    print("HATALAR")
    print("=" * 120)

    for error in errors:

        print(
            "-",
            error
        )


print()
print("=" * 120)


if not errors:

    print(
        "SONUÇ: ALBARAKA TÜRK "
        "FİNANSMAN EXTRACTOR "
        "17/17 TAMAMEN BAŞARILI ✅"
    )

else:

    print(
        "SONUÇ: ALBARAKA TÜRK "
        "FİNANSMAN EXTRACTOR "
        "KONTROL GEREKİYOR ❌"
    )


print("=" * 120)


files.download(
    OUTPUT_FILE
)

ALBARAKA TÜRK - FINANSMAN EXTRACTOR FINAL V1
RAW ürün: 17

ALBARAKA TÜRK - FINANSMAN FINAL AUDIT
RAW ürün          : 17
Extracted         : 17
Duplicate URL     : 0
Duplicate ürün    : 0
Validation error  : 2
Final JSON        : /content/albaraka_turk_finansman_extracted.json

KATEGORİ DAĞILIMI
Konut Finansmanı              : 1 / 1
Taşıt Finansmanı              : 5 / 5
Gayrimenkul Finansmanı        : 3 / 3
Bayide Finansman              : 1 / 1
İhtiyaç Finansmanı            : 6 / 6
BES Teminatlı Finansman       : 1 / 1

FİNANSMAN DEĞERLERİ

Konut Finansmanı
  Kâr Payı : []
  Fin.Oran : []
  Tutar    : []
  Vade     : ['120 aya kadar']
  Taksit   : []
  Masraf   : []
  Para     : []
  Hedef    : ['18 yaşını tamamlamış müşteriler']
  Koşul    : 35

Taşıt Finansmanı
  Kâr Payı : []
  Fin.Oran : ['%70', '%50', '%30', '%20', '%0']
  Tutar    : []
  Vade     : ['48 ay', '36 ay', '24 ay', '12 ay']
  Taksit   : []
  Masraf   : []
  Para     : ['TL']
  Hedef    : ['18 yaşını tamamlamış müşterile

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# ALBARAKA TÜRK
# FINANSMAN FINALIZER V2
#
# Sadece 2 semantik hata düzeltilir:
#
# 1) Dijital Araç Finansmanı
#    800.000 / 1.200.000 / 2.000.000 TL
#    -> araç fiyat/kasko değer eşikleri
#    -> finansman_tutari DEĞİL
#
# 2) Motosiklet, ATV, Bisiklet
#    125.000 / 250.000 TL
#    -> ürün fiyat eşikleri
#    -> finansman_tutari DEĞİL
#
# Input:
#   /content/albaraka_turk_finansman_extracted.json
#
# Output:
#   aynı dosya üzerine final hali
# ============================================================

import json
from collections import Counter
from google.colab import files


INPUT_FILE = (
    "/content/"
    "albaraka_turk_finansman_extracted.json"
)

OUTPUT_FILE = (
    "/content/"
    "albaraka_turk_finansman_extracted.json"
)

BANK_NAME = (
    "Albaraka Türk Katılım Bankası A.Ş."
)


# ============================================================
# SCHEMA
# ============================================================

SCHEMA_KEYS = [
    "banka",
    "kayit_turu",
    "urun_adi",
    "urun_kategorisi",
    "kar_payi_orani",
    "finansman_orani",
    "finansman_tutari",
    "vade",
    "taksit_sayisi",
    "masraf_bilgisi",
    "kampanya_turu",
    "kampanya_avantaji",
    "kampanya_suresi",
    "hedef_kitle",
    "para_birimi",
    "kosullar",
    "kaynak_url",
    "ham_metin",
]


LIST_FIELDS = {
    "kar_payi_orani",
    "finansman_orani",
    "finansman_tutari",
    "vade",
    "taksit_sayisi",
    "masraf_bilgisi",
    "kampanya_avantaji",
    "hedef_kitle",
    "para_birimi",
    "kosullar",
}


SCALAR_FIELDS = (
    set(SCHEMA_KEYS)
    - LIST_FIELDS
)


# ============================================================
# LOAD
# ============================================================

with open(
    INPUT_FILE,
    "r",
    encoding="utf-8"
) as f:

    records = json.load(f)


print("=" * 120)
print("ALBARAKA TÜRK - FINANSMAN FINALIZER V2")
print("=" * 120)

print(
    "Input kayıt:",
    len(records)
)


# ============================================================
# INDEX
# ============================================================

by_name = {
    record["urun_adi"]:
    record
    for record in records
}


# ============================================================
# PATCH 1
# DİJİTAL ARAÇ
# ============================================================

name = "Dijital Araç Finansmanı"

record = by_name.get(name)

if record is None:

    raise RuntimeError(
        f"{name} bulunamadı."
    )


print()
print(name)

print(
    "  ESKİ TUTAR:",
    record["finansman_tutari"]
)

record[
    "finansman_tutari"
] = []

print(
    "  YENİ TUTAR:",
    record["finansman_tutari"]
)


# ============================================================
# PATCH 2
# MOTOSİKLET / ATV / BİSİKLET
# ============================================================

name = "Motosiklet, ATV, Bisiklet"

record = by_name.get(name)

if record is None:

    raise RuntimeError(
        f"{name} bulunamadı."
    )


print()
print(name)

print(
    "  ESKİ TUTAR:",
    record["finansman_tutari"]
)

record[
    "finansman_tutari"
] = []

print(
    "  YENİ TUTAR:",
    record["finansman_tutari"]
)


# ============================================================
# FULL VALIDATION
# ============================================================

errors = []


# ------------------------------------------------------------
# COUNT
# ------------------------------------------------------------

if len(records) != 17:

    errors.append(
        f"Kayıt sayısı {len(records)} != 17"
    )


# ------------------------------------------------------------
# SCHEMA + TYPES
# ------------------------------------------------------------

for index, record in enumerate(
    records,
    start=1
):

    name = record.get(
        "urun_adi",
        "?"
    )


    if list(
        record.keys()
    ) != SCHEMA_KEYS:

        errors.append(
            (
                f"[{index}] {name} -> "
                "schema/order hatası"
            )
        )


    for field in LIST_FIELDS:

        if not isinstance(
            record.get(field),
            list
        ):

            errors.append(
                (
                    f"{name} -> "
                    f"{field} list değil"
                )
            )


    for field in SCALAR_FIELDS:

        if not isinstance(
            record.get(field),
            str
        ):

            errors.append(
                (
                    f"{name} -> "
                    f"{field} string değil"
                )
            )


    if (
        record.get("banka")
        != BANK_NAME
    ):

        errors.append(
            f"{name} -> banka yanlış"
        )


    if (
        record.get("kayit_turu")
        != "finansman"
    ):

        errors.append(
            (
                f"{name} -> "
                "kayit_turu yanlış"
            )
        )


    for field in [
        "urun_adi",
        "urun_kategorisi",
        "kaynak_url",
        "ham_metin",
    ]:

        if not record.get(field):

            errors.append(
                (
                    f"{name} -> "
                    f"{field} boş"
                )
            )


    if record["kampanya_turu"]:

        errors.append(
            (
                f"{name} -> "
                "kampanya_turu boş olmalı"
            )
        )


    if record["kampanya_avantaji"]:

        errors.append(
            (
                f"{name} -> "
                "kampanya_avantaji boş olmalı"
            )
        )


    if record["kampanya_suresi"]:

        errors.append(
            (
                f"{name} -> "
                "kampanya_suresi boş olmalı"
            )
        )


# ============================================================
# DUPLICATES
# ============================================================

urls = [
    r["kaynak_url"]
    for r in records
]

names = [
    r["urun_adi"]
    for r in records
]


duplicate_url = (
    len(urls)
    -
    len(set(urls))
)

duplicate_name = (
    len(names)
    -
    len(set(names))
)


if duplicate_url:

    errors.append(
        (
            "Duplicate URL: "
            f"{duplicate_url}"
        )
    )


if duplicate_name:

    errors.append(
        (
            "Duplicate ürün: "
            f"{duplicate_name}"
        )
    )


# ============================================================
# CRITICAL EXPECTED VALUES
# ============================================================

EXPECTED = {

    "Konut Finansmanı": {
        "vade":
            ["120 aya kadar"],
    },

    "Taşıt Finansmanı": {
        "finansman_orani":
            [
                "%70",
                "%50",
                "%30",
                "%20",
                "%0",
            ],

        "finansman_tutari":
            [],

        "vade":
            [
                "48 ay",
                "36 ay",
                "24 ay",
                "12 ay",
            ],
    },

    "Togg Finansmanı": {
        "finansman_orani":
            ["%70"],

        "vade":
            ["48 aya kadar"],
    },

    "Dijital Araç Finansmanı": {
        "finansman_orani":
            [
                "%70",
                "%50",
                "%30",
                "%20",
                "%0",
            ],

        "finansman_tutari":
            [],

        "vade":
            [
                "48 ay",
                "36 ay",
                "24 ay",
                "12 ay",
            ],
    },

    "İş Yeri Finansmanı": {
        "finansman_orani":
            ["%100"],

        "vade":
            ["60 aya kadar"],
    },

    "Arsa Finansmanı": {
        "finansman_orani":
            ["%100"],

        "vade":
            ["60 aya kadar"],
    },

    "2B Arazi Finansmanı": {
        "vade":
            ["60 aya kadar"],
    },

    "Bayide Finansman": {
        "vade":
            ["36 aya kadar"],
    },

    "Şubesiz Umre Finansmanı": {
        "kar_payi_orani":
            [],

        "finansman_tutari":
            ["50.000 TL"],

        "taksit_sayisi":
            ["4"],
    },

    "Jet Finansman": {
        "vade":
            ["36 aya kadar"],

        "taksit_sayisi":
            ["12", "6"],
    },

    "Motosiklet, ATV, Bisiklet": {
        "finansman_tutari":
            [],

        "vade":
            [
                "36 ay",
                "24 ay",
                "12 ay",
            ],
    },

    "Eğitim Finansmanı": {
        "vade":
            ["12 aya kadar"],
    },
}


for name, fields in EXPECTED.items():

    record = by_name.get(name)

    if record is None:

        errors.append(
            f"{name} bulunamadı"
        )

        continue


    for field, expected in fields.items():

        actual = record[field]

        if actual != expected:

            errors.append(
                (
                    f"{name} -> "
                    f"{field}: "
                    f"{actual} != {expected}"
                )
            )


# ============================================================
# IMPORTANT SEMANTIC RULES
# ============================================================

# Şubesiz Umre:
# "kâr paysız" -> %0 kâr payı olarak türetilmez.

if (
    by_name[
        "Şubesiz Umre Finansmanı"
    ][
        "kar_payi_orani"
    ]
):

    errors.append(
        (
            "Şubesiz Umre -> "
            "kâr payı oranı boş olmalı"
        )
    )


# Taşıt değer eşikleri finansman tutarı değildir.

for name in [
    "Taşıt Finansmanı",
    "Dijital Araç Finansmanı",
    "Motosiklet, ATV, Bisiklet",
]:

    if by_name[
        name
    ][
        "finansman_tutari"
    ]:

        errors.append(
            (
                f"{name} -> "
                "finansman_tutari "
                "boş olmalı"
            )
        )


# ============================================================
# CATEGORY DISTRIBUTION
# ============================================================

category_counts = Counter(
    r["urun_kategorisi"]
    for r in records
)


EXPECTED_CATEGORIES = {
    "Konut Finansmanı": 1,
    "Taşıt Finansmanı": 5,
    "Gayrimenkul Finansmanı": 3,
    "Bayide Finansman": 1,
    "İhtiyaç Finansmanı": 6,
    "BES Teminatlı Finansman": 1,
}


if (
    dict(category_counts)
    != EXPECTED_CATEGORIES
):

    errors.append(
        (
            "Kategori dağılımı yanlış: "
            f"{dict(category_counts)}"
        )
    )


# ============================================================
# SAVE
# ============================================================

with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        records,
        f,
        ensure_ascii=False,
        indent=4
    )


# ============================================================
# FINAL AUDIT
# ============================================================

print()
print("=" * 120)
print(
    "ALBARAKA TÜRK - "
    "FINANSMAN FINAL AUDIT V2"
)
print("=" * 120)

print(
    "Kayıt             :",
    len(records)
)

print(
    "Duplicate URL     :",
    duplicate_url
)

print(
    "Duplicate ürün    :",
    duplicate_name
)

print(
    "Validation error  :",
    len(errors)
)

print(
    "Final JSON        :",
    OUTPUT_FILE
)


print()
print("=" * 120)
print("KRİTİK DEĞERLER")
print("=" * 120)


for name in [
    "Taşıt Finansmanı",
    "Togg Finansmanı",
    "Dijital Araç Finansmanı",
    "İş Yeri Finansmanı",
    "Arsa Finansmanı",
    "Şubesiz Umre Finansmanı",
    "Jet Finansman",
    "Motosiklet, ATV, Bisiklet",
]:

    r = by_name[name]

    print()
    print(name)

    print(
        "  Kâr Payı :",
        r["kar_payi_orani"]
    )

    print(
        "  Fin.Oran :",
        r["finansman_orani"]
    )

    print(
        "  Tutar    :",
        r["finansman_tutari"]
    )

    print(
        "  Vade     :",
        r["vade"]
    )

    print(
        "  Taksit   :",
        r["taksit_sayisi"]
    )


# ============================================================
# ERRORS
# ============================================================

if errors:

    print()
    print("=" * 120)
    print("HATALAR")
    print("=" * 120)

    for error in errors:

        print(
            "-",
            error
        )


print()
print("=" * 120)


if not errors:

    print(
        "SONUÇ: ALBARAKA TÜRK "
        "FİNANSMAN 17/17 "
        "TAMAMEN BAŞARILI ✅"
    )

else:

    print(
        "SONUÇ: ALBARAKA TÜRK "
        "FİNANSMAN "
        "KONTROL GEREKİYOR ❌"
    )


print("=" * 120)


files.download(
    OUTPUT_FILE
)

ALBARAKA TÜRK - FINANSMAN FINALIZER V2
Input kayıt: 17

Dijital Araç Finansmanı
  ESKİ TUTAR: ['800.000 TL', '1.200.000 TL', '2.000.000 TL']
  YENİ TUTAR: []

Motosiklet, ATV, Bisiklet
  ESKİ TUTAR: ['125.000 TL', '250.000 TL']
  YENİ TUTAR: []

ALBARAKA TÜRK - FINANSMAN FINAL AUDIT V2
Kayıt             : 17
Duplicate URL     : 0
Duplicate ürün    : 0
Validation error  : 0
Final JSON        : /content/albaraka_turk_finansman_extracted.json

KRİTİK DEĞERLER

Taşıt Finansmanı
  Kâr Payı : []
  Fin.Oran : ['%70', '%50', '%30', '%20', '%0']
  Tutar    : []
  Vade     : ['48 ay', '36 ay', '24 ay', '12 ay']
  Taksit   : []

Togg Finansmanı
  Kâr Payı : []
  Fin.Oran : ['%70']
  Tutar    : []
  Vade     : ['48 aya kadar']
  Taksit   : []

Dijital Araç Finansmanı
  Kâr Payı : []
  Fin.Oran : ['%70', '%50', '%30', '%20', '%0']
  Tutar    : []
  Vade     : ['48 ay', '36 ay', '24 ay', '12 ay']
  Taksit   : []

İş Yeri Finansmanı
  Kâr Payı : []
  Fin.Oran : ['%100']
  Tutar    : []
  Vade     : [

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# ALBARAKA TÜRK
# CAMPAIGN DISCOVERY V1
#
# Amaç:
#   1) İlk sayfadaki kampanya detail linklerini bul
#   2) Kampanya kategori URL'lerini bul
#   3) "Daha Fazla Kampanya Göster" butonunu incele
#   4) HTML / script içinde API-AJAX-loadMore endpoint ara
#
# Output:
#   /content/albaraka_turk_kampanya_discovery_v1.json
# ============================================================

!pip -q install requests==2.32.4 beautifulsoup4

import json
import re
import html
from urllib.parse import urljoin, urlparse

import requests
from bs4 import BeautifulSoup
from google.colab import files


# ============================================================
# URL
# ============================================================

BASE = "https://www.albaraka.com.tr"

MAIN_URL = (
    BASE
    + "/tr/kampanyalar"
)

OUTPUT_FILE = (
    "/content/"
    "albaraka_turk_kampanya_discovery_v1.json"
)


# ============================================================
# SESSION
# ============================================================

session = requests.Session()

session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 "
        "(Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/151.0.0.0 "
        "Safari/537.36"
    ),
    "Accept-Language":
        "tr-TR,tr;q=0.9,en;q=0.8",
})


# ============================================================
# HELPERS
# ============================================================

def clean_text(value):

    value = html.unescape(
        str(value or "")
    )

    value = (
        value
        .replace("\xa0", " ")
        .replace("’", "'")
        .replace("‘", "'")
        .replace("–", "-")
        .replace("—", "-")
    )

    value = re.sub(
        r"\s+",
        " ",
        value
    )

    return value.strip()


def normalize(value):

    return (
        clean_text(value)
        .replace("İ", "i")
        .replace("I", "ı")
        .casefold()
    )


def unique(values):

    result = []
    seen = set()

    for value in values:

        value = clean_text(value)

        if not value:
            continue

        if value in seen:
            continue

        seen.add(value)
        result.append(value)

    return result


# ============================================================
# FETCH
# ============================================================

response = session.get(
    MAIN_URL,
    timeout=40,
    allow_redirects=True
)

response.raise_for_status()

soup = BeautifulSoup(
    response.text,
    "html.parser"
)


print("=" * 120)
print(
    "ALBARAKA TÜRK - "
    "CAMPAIGN DISCOVERY V1"
)
print("=" * 120)

print(
    "HTTP     :",
    response.status_code
)

print(
    "HTML len :",
    len(response.text)
)

print(
    "Final URL:",
    response.url
)


h1 = soup.find("h1")

print(
    "H1       :",
    (
        clean_text(
            h1.get_text(
                " ",
                strip=True
            )
        )
        if h1
        else ""
    )
)


# ============================================================
# 1) DETAIL LINKS
# ============================================================

detail_links = []


for a in soup.find_all(
    "a",
    href=True
):

    href = urljoin(
        MAIN_URL,
        a.get("href", "")
    )

    parsed = urlparse(href)

    path = parsed.path.rstrip("/")

    if not (
        path.startswith(
            "/tr/kampanyalar/detay/"
        )
        or
        path.startswith(
            "/tr/Kampanyalar/"
        )
    ):
        continue


    # geçmiş kampanya landing vb olmasın
    if (
        "/gecmis-kampanyalar"
        in path.casefold()
    ):
        continue


    title = clean_text(
        a.get_text(
            " ",
            strip=True
        )
    )


    detail_links.append({
        "title":
            title,

        "url":
            href.split("#")[0],
    })


# URL bazında dedupe
detail_by_url = {}

for item in detail_links:

    url = item["url"]

    if url not in detail_by_url:

        detail_by_url[url] = item

    else:

        # Daha uzun anchor text varsa onu koru.
        if (
            len(item["title"])
            >
            len(
                detail_by_url[url]["title"]
            )
        ):

            detail_by_url[url] = item


detail_links = list(
    detail_by_url.values()
)


print()
print("=" * 120)
print("İLK HTML DETAIL LINKLERİ")
print("=" * 120)

print(
    "Unique detail link:",
    len(detail_links)
)


for index, item in enumerate(
    detail_links,
    start=1
):

    print(
        f"[{index:02d}] "
        f"{item['title']}"
    )

    print(
        "     ",
        item["url"]
    )


# ============================================================
# 2) CATEGORY LINKS
# ============================================================

category_links = []


for a in soup.find_all(
    "a",
    href=True
):

    text = clean_text(
        a.get_text(
            " ",
            strip=True
        )
    )

    href = urljoin(
        MAIN_URL,
        a.get(
            "href",
            ""
        )
    )


    path = urlparse(
        href
    ).path.rstrip("/")


    if not path.startswith(
        "/tr/kampanyalar"
    ):
        continue


    # detail değil
    if (
        "/detay/"
        in path.casefold()
    ):
        continue


    if not text:
        continue


    n = normalize(text)

    if any(
        keyword in n
        for keyword in [
            "dijital",
            "eflatun",
            "trend",
            "özel bankacılık",
            "bireysel",
            "business",
            "world",
            "geçmiş",
            "tümü",
        ]
    ):

        category_links.append({
            "text":
                text,

            "url":
                href,
        })


# dedupe
temp = {}

for item in category_links:

    key = (
        item["text"],
        item["url"],
    )

    temp[key] = item


category_links = list(
    temp.values()
)


print()
print("=" * 120)
print("KATEGORİ LİNKLERİ")
print("=" * 120)


for item in category_links:

    print(
        f"- {item['text']}"
    )

    print(
        " ",
        item["url"]
    )


# ============================================================
# 3) LOAD MORE BUTTON / ELEMENT
# ============================================================

load_more_elements = []


for tag in soup.find_all(
    [
        "button",
        "a",
        "div",
        "span",
        "input",
    ]
):

    text = clean_text(
        tag.get_text(
            " ",
            strip=True
        )
    )


    if (
        "daha fazla kampanya"
        not in normalize(text)
        and
        "daha fazla göster"
        not in normalize(text)
    ):

        continue


    attrs = {}

    for key, value in (
        tag.attrs.items()
    ):

        if isinstance(
            value,
            list
        ):

            value = " ".join(
                str(x)
                for x in value
            )

        attrs[
            str(key)
        ] = str(value)


    parent = tag.parent

    parent_attrs = {}

    if parent is not None:

        for key, value in (
            parent.attrs.items()
        ):

            if isinstance(
                value,
                list
            ):

                value = " ".join(
                    str(x)
                    for x in value
                )

            parent_attrs[
                str(key)
            ] = str(value)


    load_more_elements.append({
        "tag":
            tag.name,

        "text":
            text,

        "attrs":
            attrs,

        "parent_tag":
            (
                parent.name
                if parent
                else None
            ),

        "parent_attrs":
            parent_attrs,

        "html":
            str(tag)[:3000],
    })


print()
print("=" * 120)
print("LOAD MORE ELEMENT")
print("=" * 120)

print(
    "Bulunan:",
    len(load_more_elements)
)


for index, item in enumerate(
    load_more_elements,
    start=1
):

    print()
    print(
        f"[{index}] TAG:",
        item["tag"]
    )

    print(
        "TEXT:",
        item["text"]
    )

    print(
        "ATTR:",
        item["attrs"]
    )

    print(
        "PARENT ATTR:",
        item["parent_attrs"]
    )

    print(
        "HTML:",
        item["html"]
    )


# ============================================================
# 4) SCRIPT ANALYSIS
# ============================================================

script_sources = []

inline_script_hits = []


KEYWORDS = [
    "campaign",
    "kampanya",
    "loadmore",
    "load-more",
    "load more",
    "showmore",
    "show-more",
    "ajax",
    "api/",
    "/api",
    "page=",
    "pageindex",
    "pagesize",
    "skip",
    "take",
    "offset",
    "limit",
]


for script in soup.find_all(
    "script"
):

    src = script.get(
        "src"
    )

    if src:

        script_sources.append(
            urljoin(
                MAIN_URL,
                src
            )
        )


    content = (
        script.string
        or
        script.get_text(
            "\n",
            strip=False
        )
        or
        ""
    )


    if not content:
        continue


    n = normalize(content)


    matched = [
        keyword
        for keyword in KEYWORDS
        if keyword in n
    ]


    if not matched:
        continue


    # Keyword çevresinden kısa parçalar.
    excerpts = []


    for keyword in matched:

        start = 0

        while True:

            pos = n.find(
                keyword,
                start
            )

            if pos < 0:
                break


            left = max(
                0,
                pos - 250
            )

            right = min(
                len(content),
                pos + 500
            )


            excerpt = clean_text(
                content[
                    left:right
                ]
            )


            excerpts.append(
                excerpt
            )


            start = (
                pos
                +
                len(keyword)
            )


            if len(excerpts) >= 12:
                break


        if len(excerpts) >= 12:
            break


    inline_script_hits.append({
        "keywords":
            matched,

        "excerpts":
            unique(
                excerpts
            )[:12],
    })


script_sources = unique(
    script_sources
)


print()
print("=" * 120)
print("SCRIPT SOURCES")
print("=" * 120)

print(
    "Script src:",
    len(script_sources)
)


for url in script_sources:

    print(
        "-",
        url
    )


print()
print("=" * 120)
print("INLINE SCRIPT HITS")
print("=" * 120)

print(
    "Hit script:",
    len(inline_script_hits)
)


for index, item in enumerate(
    inline_script_hits,
    start=1
):

    print()
    print(
        f"[{index}] Keywords:",
        item["keywords"]
    )

    for excerpt in item[
        "excerpts"
    ]:

        print(
            "  >",
            excerpt[:1000]
        )


# ============================================================
# 5) RAW HTML ENDPOINT / API PATTERN SEARCH
# ============================================================

html_text = html.unescape(
    response.text
)


patterns = {

    "api_paths":
        (
            r"""["'](
            /[^"' ]*
            (?:api|ajax)
            [^"' ]*
            )["']"""
        ),

    "campaign_paths":
        (
            r"""["'](
            /[^"' ]*
            (?:kampanya|campaign)
            [^"' ]*
            )["']"""
        ),

    "url_like":
        (
            r"""https?://
            [^"'<> ]+"""
        ),

    "page_params":
        (
            r"""
            (?:page|pageIndex|pageNumber|
            pageSize|skip|take|offset|limit)
            \s*[:=]\s*
            ["']?\d+
            """
        ),
}


raw_hits = {}


for name, pattern in (
    patterns.items()
):

    try:

        matches = re.findall(
            pattern,
            html_text,
            flags=(
                re.I
                |
                re.X
            )
        )

    except Exception as error:

        matches = [
            f"REGEX ERROR: {error}"
        ]


    if matches:

        if isinstance(
            matches[0],
            tuple
        ):

            matches = [
                next(
                    (
                        x
                        for x in m
                        if x
                    ),
                    ""
                )
                for m in matches
            ]


    raw_hits[
        name
    ] = unique(
        matches
    )[:100]


print()
print("=" * 120)
print("RAW HTML ENDPOINT HITS")
print("=" * 120)


for name, values in (
    raw_hits.items()
):

    print()
    print(
        name,
        ":",
        len(values)
    )

    for value in values[:30]:

        print(
            "-",
            value[:1000]
        )


# ============================================================
# SAVE
# ============================================================

output = {
    "main_url":
        MAIN_URL,

    "http_status":
        response.status_code,

    "html_length":
        len(
            response.text
        ),

    "detail_link_count":
        len(
            detail_links
        ),

    "detail_links":
        detail_links,

    "category_links":
        category_links,

    "load_more_elements":
        load_more_elements,

    "script_sources":
        script_sources,

    "inline_script_hits":
        inline_script_hits,

    "raw_hits":
        raw_hits,
}


with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        output,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# FINAL SUMMARY
# ============================================================

print()
print("=" * 120)
print(
    "ALBARAKA TÜRK - "
    "CAMPAIGN DISCOVERY V1 SONUCU"
)
print("=" * 120)

print(
    "HTML detail link    :",
    len(detail_links)
)

print(
    "Kategori link       :",
    len(category_links)
)

print(
    "Load more element   :",
    len(load_more_elements)
)

print(
    "Script src          :",
    len(script_sources)
)

print(
    "Inline script hit   :",
    len(inline_script_hits)
)

print(
    "API path hit        :",
    len(
        raw_hits[
            "api_paths"
        ]
    )
)

print(
    "Campaign path hit   :",
    len(
        raw_hits[
            "campaign_paths"
        ]
    )
)

print(
    "Page param hit      :",
    len(
        raw_hits[
            "page_params"
        ]
    )
)

print(
    "Discovery JSON      :",
    OUTPUT_FILE
)

print()
print("=" * 120)
print(
    "SONUÇ: DISCOVERY TAMAMLANDI ✅"
)
print("=" * 120)


files.download(
    OUTPUT_FILE
)

ALBARAKA TÜRK - CAMPAIGN DISCOVERY V1
HTTP     : 200
HTML len : 218705
Final URL: https://www.albaraka.com.tr/tr/kampanyalar
H1       : Kampanyalar

İLK HTML DETAIL LINKLERİ
Unique detail link: 11
[01] Dijital Müşterilere Özel Pratik Finansman Kart
      https://www.albaraka.com.tr/tr/kampanyalar/detay/dijital-musterilere-ozel-pratik-finansman-kart
[02] Aylık 2 Defa Ücretsiz İSPARK Otopark İndirimi Albaraka'da!
      https://www.albaraka.com.tr/tr/kampanyalar/detay/albarakalilara-ozel-ucretsiz-ispark-kampanyasi-1
[03] 
      https://www.albaraka.com.tr/tr/kampanyalar/detay/albarakada-masraflara-son
[04] Kâr payı yok. Beklemek yok. 140.000 TL'ye kadar vade farksız destek Albaraka'da!
      https://www.albaraka.com.tr/tr/kampanyalar/detay/vade-farksiz-kampanyasi
[05] "OFT2026” davet kodu ile müşterimiz olun her fatura için 500 TL, toplamda 2.000 TL Worldpuan kazanın!
      https://www.albaraka.com.tr/tr/kampanyalar/detay/agustos-ayina-ozel-fatura-kampanyasi
[06] Yurt Dışına Çıkış Harcı k

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# ALBARAKA TÜRK
# CAMPAIGN JS / AJAX DISCOVERY V2
#
# Amaç:
# "Daha Fazla Kampanya Göster" butonunun gerçek
# AJAX endpoint + parametrelerini bulmak.
# ============================================================

!pip -q install requests==2.32.4 beautifulsoup4

import re
import html
import requests
from bs4 import BeautifulSoup


BASE = "https://www.albaraka.com.tr"

MAIN_URL = (
    BASE
    + "/tr/kampanyalar"
)


# ============================================================
# SESSION
# ============================================================

session = requests.Session()

session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 "
        "(Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/151.0.0.0 "
        "Safari/537.36"
    ),
    "Accept-Language":
        "tr-TR,tr;q=0.9,en;q=0.8",
})


# ============================================================
# HELPERS
# ============================================================

def clean_text(value):

    value = html.unescape(
        str(value or "")
    )

    value = (
        value
        .replace("\xa0", " ")
        .replace("’", "'")
        .replace("‘", "'")
        .replace("–", "-")
        .replace("—", "-")
    )

    return value


def compact(value):

    value = clean_text(value)

    value = re.sub(
        r"[ \t]+",
        " ",
        value
    )

    value = re.sub(
        r"\n{3,}",
        "\n\n",
        value
    )

    return value.strip()


def unique(values):

    result = []
    seen = set()

    for value in values:

        value = str(
            value or ""
        ).strip()

        if not value:
            continue

        if value in seen:
            continue

        seen.add(value)
        result.append(value)

    return result


# ============================================================
# FETCH MAIN PAGE
# ============================================================

main_response = session.get(
    MAIN_URL,
    timeout=40
)

main_response.raise_for_status()


soup = BeautifulSoup(
    main_response.text,
    "html.parser"
)


# ============================================================
# GET SAME-ORIGIN JS FILES
# ============================================================

js_urls = []


for script in soup.find_all(
    "script",
    src=True
):

    src = script.get(
        "src",
        ""
    )


    if src.startswith("//"):

        src = "https:" + src


    elif src.startswith("/"):

        src = BASE + src


    elif not src.startswith(
        "http"
    ):

        src = (
            BASE
            + "/"
            + src.lstrip("/")
        )


    # Sadece Albaraka'nın kendi JS'leri.
    if (
        "www.albaraka.com.tr"
        not in src
    ):

        continue


    js_urls.append(src)


js_urls = unique(
    js_urls
)


print("=" * 125)
print(
    "ALBARAKA TÜRK - "
    "CAMPAIGN JS/AJAX DISCOVERY V2"
)
print("=" * 125)

print(
    "Same-origin JS:",
    len(js_urls)
)


# ============================================================
# KEYWORDS
# ============================================================

KEYWORDS = [
    "Kampanyalar",
    "kampanyalar-page",
    "kampanyalar-card",
    "btn-outline-kampanyalar-primary",
    "Daha Fazla Kampanya",
    "Mustache.render",
    "CampaignImage",
    "PageIndex",
    "PageSize",
    "pageIndex",
    "pageSize",
    "loadMore",
    "loadmore",
]


# ============================================================
# FETCH ALL JS
# ============================================================

hits = []


for index, js_url in enumerate(
    js_urls,
    start=1
):

    try:

        response = session.get(
            js_url,
            timeout=40
        )


        if response.status_code != 200:

            print(
                f"[{index:02d}] "
                f"HTTP {response.status_code} "
                f"{js_url}"
            )

            continue


        text = response.text


        matched = [
            keyword
            for keyword in KEYWORDS
            if keyword.casefold()
            in text.casefold()
        ]


        print(
            f"[{index:02d}] "
            f"HTTP 200 "
            f"| len={len(text):7d} "
            f"| hit={len(matched):02d} "
            f"| {js_url}"
        )


        if not matched:

            continue


        hits.append({
            "url":
                js_url,

            "text":
                text,

            "matched":
                matched,
        })


    except Exception as error:

        print(
            f"[{index:02d}] ERROR "
            f"{type(error).__name__}: "
            f"{js_url}"
        )


# ============================================================
# PRINT RELEVANT CONTEXTS
# ============================================================

print()
print("=" * 125)
print("KAMPANYA JS HITS")
print("=" * 125)

print(
    "Hit JS file:",
    len(hits)
)


for file_index, item in enumerate(
    hits,
    start=1
):

    text = item[
        "text"
    ]

    lower = text.casefold()


    print()
    print("#" * 125)

    print(
        f"JS FILE [{file_index}]"
    )

    print(
        item["url"]
    )

    print(
        "MATCHED:",
        item["matched"]
    )

    print("#" * 125)


    shown_ranges = []


    for keyword in KEYWORDS:

        keyword_lower = (
            keyword.casefold()
        )

        start_pos = 0


        while True:

            pos = lower.find(
                keyword_lower,
                start_pos
            )


            if pos < 0:
                break


            left = max(
                0,
                pos - 2500
            )

            right = min(
                len(text),
                pos + 6000
            )


            # Aynı bölgeyi tekrar yazdırma.
            overlap = any(
                (
                    left
                    <= old_right
                    and
                    right
                    >= old_left
                )
                for old_left, old_right
                in shown_ranges
            )


            if not overlap:

                shown_ranges.append(
                    (
                        left,
                        right
                    )
                )


                snippet = text[
                    left:right
                ]


                print()
                print(
                    "-" * 125
                )

                print(
                    "KEYWORD:",
                    keyword
                )

                print(
                    "-" * 125
                )

                print(
                    snippet
                )


            start_pos = (
                pos
                +
                len(keyword_lower)
            )


            # Her keyword için max 4 occurrence
            if sum(
                1
                for x in shown_ranges
            ) >= 12:

                break


# ============================================================
# AJAX CALL EXTRACTION
# ============================================================

print()
print("=" * 125)
print("AJAX / FETCH BLOKLARI")
print("=" * 125)


ajax_blocks = []


for item in hits:

    text = item[
        "text"
    ]


    patterns = [

        # $.ajax({...})
        r"\$\s*\.\s*ajax\s*\(\s*\{",

        # $.get(...)
        r"\$\s*\.\s*get\s*\(",

        # $.post(...)
        r"\$\s*\.\s*post\s*\(",

        # fetch(...)
        r"\bfetch\s*\(",
    ]


    for pattern in patterns:

        for match in re.finditer(
            pattern,
            text,
            flags=re.I
        ):

            left = max(
                0,
                match.start() - 1200
            )

            right = min(
                len(text),
                match.start() + 5000
            )


            block = text[
                left:right
            ]


            lower_block = (
                block.casefold()
            )


            # Kampanya ile ilgisi olmalı.
            if not any(
                marker in lower_block
                for marker in [
                    "kampanya",
                    "campaign",
                    "campaignimage",
                    "pagesize",
                    "pageindex",
                    "btn-outline-kampanyalar",
                ]
            ):

                continue


            ajax_blocks.append({
                "url":
                    item["url"],

                "block":
                    block,
            })


# dedupe
unique_ajax = []

seen_ajax = set()


for item in ajax_blocks:

    key = item[
        "block"
    ]

    if key in seen_ajax:
        continue

    seen_ajax.add(key)
    unique_ajax.append(item)


print(
    "Campaign AJAX blocks:",
    len(unique_ajax)
)


for index, item in enumerate(
    unique_ajax,
    start=1
):

    print()
    print(
        "#" * 125
    )

    print(
        f"AJAX BLOCK [{index}]"
    )

    print(
        "SOURCE:",
        item["url"]
    )

    print(
        "#" * 125
    )

    print(
        item["block"]
    )


# ============================================================
# URL / ENDPOINT LITERALS NEAR CAMPAIGN CODE
# ============================================================

print()
print("=" * 125)
print("ENDPOINT ADAYLARI")
print("=" * 125)


endpoint_candidates = []


for item in hits:

    text = item[
        "text"
    ]


    for match in re.finditer(
        r"""["']([^"']+)["']""",
        text
    ):

        value = match.group(1)


        if not any(
            marker in value.casefold()
            for marker in [
                "kampanya",
                "campaign",
                "plugin",
                "load",
            ]
        ):

            continue


        if len(value) > 300:
            continue


        endpoint_candidates.append(
            value
        )


endpoint_candidates = unique(
    endpoint_candidates
)


for value in endpoint_candidates:

    print(
        "-",
        value
    )


# ============================================================
# SEARCH IMPORTANT VARIABLE DECLARATIONS
# ============================================================

print()
print("=" * 125)
print("PAGE / SIZE / SKIP VARIABLE HITS")
print("=" * 125)


VARIABLE_PATTERNS = [
    r"\bpage(?:Index|Number)?\b",
    r"\bpageSize\b",
    r"\bskip\b",
    r"\btake\b",
    r"\boffset\b",
    r"\blimit\b",
    r"\bcurrentPage\b",
]


variable_hits = []


for item in hits:

    text = item[
        "text"
    ]


    for pattern in VARIABLE_PATTERNS:

        for match in re.finditer(
            pattern,
            text,
            flags=re.I
        ):

            left = max(
                0,
                match.start() - 500
            )

            right = min(
                len(text),
                match.start() + 1200
            )


            block = text[
                left:right
            ]


            if not any(
                marker
                in block.casefold()
                for marker in [
                    "kampanya",
                    "campaign",
                ]
            ):

                continue


            variable_hits.append(
                compact(block)
            )


variable_hits = unique(
    variable_hits
)


for index, value in enumerate(
    variable_hits,
    start=1
):

    print()
    print(
        f"[{index}]",
        value
    )


# ============================================================
# MOST IMPORTANT SUMMARY
# ============================================================

print()
print("=" * 125)
print(
    "ALBARAKA TÜRK - "
    "CAMPAIGN JS DISCOVERY V2 SONUCU"
)
print("=" * 125)

print(
    "Same-origin JS       :",
    len(js_urls)
)

print(
    "Campaign JS hit      :",
    len(hits)
)

print(
    "Campaign AJAX block  :",
    len(unique_ajax)
)

print(
    "Endpoint candidate   :",
    len(endpoint_candidates)
)

print(
    "Page variable hit    :",
    len(variable_hits)
)

print()
print("=" * 125)
print(
    "JS DISCOVERY TAMAMLANDI ✅"
)
print("=" * 125)

ALBARAKA TÜRK - CAMPAIGN JS/AJAX DISCOVERY V2
Same-origin JS: 26
[01] HTTP 200 | len= 104878 | hit=00 | https://www.albaraka.com.tr/adrum/adrum.js
[02] HTTP 200 | len=  89476 | hit=00 | https://www.albaraka.com.tr/_assets/js/core/jquery-3.5.1.min.js
[03] HTTP 200 | len=   1872 | hit=00 | https://www.albaraka.com.tr/_assets/js/core/browser.js
[04] HTTP 200 | len=  20302 | hit=00 | https://www.albaraka.com.tr/_assets/js/plugins/popper.min.js
[05] HTTP 200 | len=  63244 | hit=00 | https://www.albaraka.com.tr/_assets/js/plugins/bootstrap.min.js
[06] HTTP 200 | len=  47533 | hit=00 | https://www.albaraka.com.tr/_assets/js/plugins/jquery-ui.js
[07] HTTP 200 | len=   5253 | hit=00 | https://www.albaraka.com.tr/_assets/js/plugins/jquery.ui.touch-punch.js
[08] HTTP 200 | len=  37974 | hit=00 | https://www.albaraka.com.tr/_assets/js/plugins/jquery.fullPage.min.js
[09] HTTP 200 | len=  42862 | hit=00 | https://www.albaraka.com.tr/_assets/js/plugins/slick.min.js
[10] HTTP 200 | len=  80312 | hit=0

In [ ]:
# ============================================================
# ALBARAKA TÜRK
# CAMPAIGN BROWSER RAW SCRAPER V6
#
# Selenium / chromedriver YOK.
# Playwright kendi Chromium'unu kullanır.
#
# Output:
#   /content/albaraka_turk_kampanyalar_raw.json
# ============================================================

!pip -q install playwright requests beautifulsoup4
!playwright install --with-deps chromium

import json
import re
import time
import requests

from bs4 import BeautifulSoup

from playwright.async_api import (
    async_playwright,
    TimeoutError as PlaywrightTimeoutError,
)

from google.colab import files


# ============================================================
# CONFIG
# ============================================================

BASE = "https://www.albaraka.com.tr"

MAIN_URL = (
    BASE
    + "/tr/kampanyalar"
)

OUTPUT_FILE = (
    "/content/"
    "albaraka_turk_kampanyalar_raw.json"
)


# Geçmiş kampanyalar dahil DEĞİL.
CATEGORY_PAGES = {
    "tumu":
        MAIN_URL,

    "dijital":
        MAIN_URL
        + "?slug=dijital",

    "eflatun":
        MAIN_URL
        + "?slug=eflatun",

    "trend":
        MAIN_URL
        + "?slug=trend",

    "ozel-bankacilik":
        MAIN_URL
        + "?slug=ozel-bankacilik",

    "bireysel":
        MAIN_URL
        + "?slug=bireysel",

    "business":
        MAIN_URL
        + "?slug=business",

    "world-kampanyalari":
        MAIN_URL
        + "?slug=world-kampanyalari",
}


CARD_SELECTOR = (
    ".campaign-list-wrap "
    ".kampanyalar-card"
)

BUTTON_SELECTOR = (
    ".kampanyalar-page "
    ".btn-outline-kampanyalar-primary"
)


# ============================================================
# HELPERS
# ============================================================

def clean_text(value):

    value = str(
        value or ""
    )

    value = (
        value
        .replace("\xa0", " ")
        .replace("’", "'")
        .replace("‘", "'")
        .replace("–", "-")
        .replace("—", "-")
        .replace("\u00ad", "")
    )

    value = re.sub(
        r"[ \t]+",
        " ",
        value
    )

    value = re.sub(
        r"\n[ \t]+",
        "\n",
        value
    )

    value = re.sub(
        r"\n{3,}",
        "\n\n",
        value
    )

    return value.strip()


def normalize(value):

    return (
        clean_text(value)
        .replace("İ", "i")
        .replace("I", "ı")
        .casefold()
    )


def unique(values):

    result = []
    seen = set()

    for value in values:

        value = clean_text(
            value
        )

        if not value:
            continue

        key = normalize(
            value
        )

        if key in seen:
            continue

        seen.add(
            key
        )

        result.append(
            value
        )

    return result


# ============================================================
# PLAYWRIGHT CARD EXTRACTOR
# ============================================================

async def extract_cards(
    page,
    discovery_category,
):

    result = []

    cards = page.locator(
        CARD_SELECTOR
    )

    count = await cards.count()


    for index in range(
        count
    ):

        card = cards.nth(
            index
        )


        # ----------------------------------------------------
        # URL
        # ----------------------------------------------------

        detail_links = (
            card.locator(
                (
                    'a[href*='
                    '"/tr/kampanyalar/detay/"]'
                )
            )
        )


        if (
            await detail_links.count()
            == 0
        ):

            continue


        href = await (
            detail_links
            .first
            .get_attribute(
                "href"
            )
        )


        href = clean_text(
            href
        )


        if not href:
            continue


        if href.startswith(
            "/"
        ):

            href = (
                BASE
                + href
            )


        href = (
            href
            .split("#")[0]
        )


        # ----------------------------------------------------
        # TITLE
        # ----------------------------------------------------

        title = ""

        title_locator = (
            card.locator(
                ".card-title"
            )
        )


        if (
            await title_locator.count()
            > 0
        ):

            title = clean_text(
                await
                title_locator
                .first
                .inner_text()
            )


        # ----------------------------------------------------
        # DESCRIPTION
        # ----------------------------------------------------

        description = ""

        description_locator = (
            card.locator(
                ".card-text"
            )
        )


        if (
            await
            description_locator
            .count()
            > 0
        ):

            description = clean_text(
                await
                description_locator
                .first
                .inner_text()
            )


        # ----------------------------------------------------
        # CATEGORY LABEL
        # ----------------------------------------------------

        card_category = ""

        category_locator = (
            card.locator(
                ".card-image-over"
            )
        )


        if (
            await
            category_locator
            .count()
            > 0
        ):

            card_category = clean_text(
                await
                category_locator
                .first
                .inner_text()
            )


        result.append({
            "kampanya_adi":
                title,

            "kart_aciklama":
                description,

            "kategori":
                card_category,

            "kaynak_url":
                href,

            "discovery_category":
                discovery_category,
        })


    return result


# ============================================================
# CATEGORY EXHAUST
# ============================================================

async def exhaust_category(
    page,
    category_name,
    url,
):

    print()
    print(
        "=" * 120
    )

    print(
        "CATEGORY:",
        category_name
    )

    print(
        "=" * 120
    )

    print(
        "URL:",
        url
    )


    await page.goto(
        url,
        wait_until="domcontentloaded",
        timeout=60000,
    )


    await page.wait_for_timeout(
        1800
    )


    initial_count = (
        await
        page.locator(
            CARD_SELECTOR
        )
        .count()
    )


    print(
        "Initial card:",
        initial_count
    )


    click_count = 0
    no_growth = False
    ajax_results = []


    # --------------------------------------------------------
    # LOAD MORE
    # --------------------------------------------------------

    for _ in range(
        50
    ):

        buttons = (
            page.locator(
                BUTTON_SELECTOR
            )
        )


        if (
            await buttons.count()
            == 0
        ):

            print(
                "Load-more button yok."
            )

            break


        button = buttons.first


        button_class = (
            await
            button.get_attribute(
                "class"
            )
            or ""
        )


        if (
            "hide"
            in
            button_class.split()
        ):

            print(
                "Load-more HIDE."
            )

            break


        try:

            visible = (
                await
                button.is_visible()
            )

        except Exception:

            visible = False


        if not visible:

            print(
                "Load-more görünür değil."
            )

            break


        before_count = (
            await
            page.locator(
                CARD_SELECTOR
            )
            .count()
        )


        await (
            button
            .scroll_into_view_if_needed()
        )


        ajax_info = {
            "before_count":
                before_count,

            "http_status":
                None,

            "url":
                "",

            "body":
                "",

            "json":
                None,
        }


        # ----------------------------------------------------
        # AJAX RESPONSE'U YAKALA
        # ----------------------------------------------------

        try:

            async with (
                page.expect_response(
                    lambda response:
                        (
                            "/plugins/GetCampaigns"
                            in response.url
                        ),
                    timeout=10000,
                )
            ) as response_info:

                await button.click(
                    force=True
                )


            ajax_response = (
                await
                response_info.value
            )


            ajax_info[
                "http_status"
            ] = (
                ajax_response.status
            )


            ajax_info[
                "url"
            ] = (
                ajax_response.url
            )


            try:

                body = (
                    await
                    ajax_response.text()
                )

            except Exception:

                body = ""


            ajax_info[
                "body"
            ] = body[:5000]


            try:

                ajax_info[
                    "json"
                ] = json.loads(
                    body
                )

            except Exception:

                pass


        except PlaywrightTimeoutError:

            print(
                "  AJAX response "
                "yakalanamadı."
            )


            try:

                await button.click(
                    force=True
                )

            except Exception:

                pass


        click_count += 1


        await page.wait_for_timeout(
            1800
        )


        after_count = (
            await
            page.locator(
                CARD_SELECTOR
            )
            .count()
        )


        ajax_info[
            "after_count"
        ] = after_count


        ajax_results.append(
            ajax_info
        )


        print(
            f"Click {click_count:02d}: "
            f"{before_count} -> "
            f"{after_count}"
        )


        if ajax_info[
            "http_status"
        ] is not None:

            print(
                "  AJAX HTTP:",
                ajax_info[
                    "http_status"
                ]
            )


        if ajax_info[
            "json"
        ] is not None:

            ajax_json = (
                ajax_info[
                    "json"
                ]
            )


            print(
                "  AJAX JSON:",
                {
                    "Result":
                        ajax_json.get(
                            "Result"
                        ),

                    "Error":
                        ajax_json.get(
                            "Error"
                        ),

                    "TotalCount":
                        (
                            (
                                ajax_json
                                .get(
                                    "Data"
                                )
                                or {}
                            )
                            .get(
                                "TotalCount"
                            )
                        ),
                }
            )


        elif ajax_info[
            "body"
        ]:

            print(
                "  AJAX BODY:",
                ajax_info[
                    "body"
                ][:500]
            )


        # ----------------------------------------------------
        # GROWTH CONTROL
        # ----------------------------------------------------

        if (
            after_count
            <=
            before_count
        ):

            no_growth = True

            print(
                "  Yeni kart eklenmedi."
            )

            break


    final_cards = (
        await
        extract_cards(
            page,
            category_name,
        )
    )


    print()
    print(
        "Final card :",
        len(final_cards)
    )

    print(
        "Click count:",
        click_count
    )

    print(
        "No growth  :",
        no_growth
    )


    return {
        "category":
            category_name,

        "url":
            url,

        "initial_count":
            initial_count,

        "final_count":
            len(
                final_cards
            ),

        "click_count":
            click_count,

        "no_growth":
            no_growth,

        "ajax_results":
            ajax_results,

        "campaigns":
            final_cards,
    }


# ============================================================
# RUN PLAYWRIGHT
# ============================================================

async def run_browser():

    category_results = []


    async with async_playwright() as p:

        browser = await (
            p.chromium.launch(
                headless=True,
                args=[
                    "--no-sandbox",
                    "--disable-dev-shm-usage",
                ],
            )
        )


        context = await (
            browser.new_context(
                locale="tr-TR",

                user_agent=(
                    "Mozilla/5.0 "
                    "(Windows NT 10.0; "
                    "Win64; x64) "
                    "AppleWebKit/537.36 "
                    "(KHTML, like Gecko) "
                    "Chrome/151.0.0.0 "
                    "Safari/537.36"
                ),
            )
        )


        page = await (
            context.new_page()
        )


        for (
            category_name,
            url
        ) in CATEGORY_PAGES.items():

            result = (
                await
                exhaust_category(
                    page,
                    category_name,
                    url,
                )
            )


            category_results.append(
                result
            )


        await browser.close()


    return category_results


print()
print("=" * 120)
print(
    "ALBARAKA TÜRK - "
    "CAMPAIGN PLAYWRIGHT RAW V6"
)
print("=" * 120)


category_results = (
    await run_browser()
)


# ============================================================
# CATEGORY SUMMARY
# ============================================================

print()
print("=" * 120)
print(
    "CATEGORY DISCOVERY ÖZETİ"
)
print("=" * 120)


for item in category_results:

    print(
        f"{item['category']:<20}"
        f" initial={item['initial_count']:<3}"
        f" final={item['final_count']:<3}"
        f" clicks={item['click_count']:<3}"
        f" no_growth={item['no_growth']}"
    )


# ============================================================
# UNION
# ============================================================

all_candidates = []


for result in category_results:

    all_candidates.extend(
        result[
            "campaigns"
        ]
    )


by_url = {}


for item in all_candidates:

    url_key = (
        item[
            "kaynak_url"
        ]
        .split("?")[0]
        .rstrip("/")
    )


    if url_key not in by_url:

        by_url[
            url_key
        ] = {
            "kampanya_adi":
                item[
                    "kampanya_adi"
                ],

            "kart_aciklama":
                item[
                    "kart_aciklama"
                ],

            "kategori":
                item[
                    "kategori"
                ],

            "kaynak_url":
                item[
                    "kaynak_url"
                ],

            "discovery_categories":
                [
                    item[
                        "discovery_category"
                    ]
                ],
        }


    else:

        old = by_url[
            url_key
        ]


        if (
            len(
                item[
                    "kampanya_adi"
                ]
            )
            >
            len(
                old[
                    "kampanya_adi"
                ]
            )
        ):

            old[
                "kampanya_adi"
            ] = item[
                "kampanya_adi"
            ]


        if (
            len(
                item[
                    "kart_aciklama"
                ]
            )
            >
            len(
                old[
                    "kart_aciklama"
                ]
            )
        ):

            old[
                "kart_aciklama"
            ] = item[
                "kart_aciklama"
            ]


        if (
            not old[
                "kategori"
            ]
            and
            item[
                "kategori"
            ]
        ):

            old[
                "kategori"
            ] = item[
                "kategori"
            ]


        discovery_category = (
            item[
                "discovery_category"
            ]
        )


        if (
            discovery_category
            not in
            old[
                "discovery_categories"
            ]
        ):

            old[
                "discovery_categories"
            ].append(
                discovery_category
            )


discovered = list(
    by_url.values()
)


print()
print("=" * 120)
print("ACTIVE CAMPAIGN UNION")
print("=" * 120)

print(
    "Category refs   :",
    len(all_candidates)
)

print(
    "Unique campaign :",
    len(discovered)
)


for index, item in enumerate(
    discovered,
    start=1,
):

    print()
    print(
        f"[{index:02d}/"
        f"{len(discovered):02d}] "
        f"{item['kampanya_adi']}"
    )

    print(
        "  Cats:",
        item[
            "discovery_categories"
        ]
    )

    print(
        "  URL :",
        item[
            "kaynak_url"
        ]
    )


# ============================================================
# DETAIL PAGE
# ============================================================

request_session = (
    requests.Session()
)

request_session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 "
        "(Windows NT 10.0; "
        "Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/151.0.0.0 "
        "Safari/537.36"
    ),

    "Accept-Language":
        "tr-TR,tr;q=0.9,en;q=0.8",
})


def extract_detail_text(soup):

    clone = BeautifulSoup(
        str(soup),
        "html.parser"
    )


    for selector in [
        "script",
        "style",
        "noscript",
        "svg",
        "header",
        "footer",
        "nav",
        "form",
    ]:

        for tag in clone.select(
            selector
        ):

            tag.decompose()


    main = clone.find(
        "main"
    )


    if main:

        text = clean_text(
            main.get_text(
                "\n",
                strip=True
            )
        )


        if len(text) >= 100:

            return text


    h1 = clone.find(
        "h1"
    )


    if h1:

        current = h1


        for _ in range(
            10
        ):

            if current is None:
                break


            text = clean_text(
                current.get_text(
                    "\n",
                    strip=True
                )
            )


            if (
                200
                <= len(text)
                <= 30000
            ):

                return text


            current = (
                current.parent
            )


    if clone.body:

        return clean_text(
            clone.body.get_text(
                "\n",
                strip=True
            )
        )


    return ""


# ============================================================
# DATE
# ============================================================

TR_MONTHS = (
    "Ocak|Şubat|Mart|Nisan|Mayıs|"
    "Haziran|Temmuz|Ağustos|Eylül|"
    "Ekim|Kasım|Aralık"
)


def extract_dates(text):

    result = []


    patterns = [
        (
            r"\b"
            r"\d{1,2}\s+"
            rf"(?:{TR_MONTHS})\s+"
            r"\d{4}\b"
        ),

        (
            r"\b"
            r"\d{1,2}"
            r"[./-]"
            r"\d{1,2}"
            r"[./-]"
            r"\d{4}\b"
        ),
    ]


    for pattern in patterns:

        for match in re.finditer(
            pattern,
            text,
            flags=re.I
        ):

            result.append(
                clean_text(
                    match.group(0)
                )
            )


    return unique(
        result
    )


# ============================================================
# DETAIL SCRAPE
# ============================================================

print()
print("=" * 120)
print("DETAIL SCRAPE")
print("=" * 120)


records = []
failed = []


for index, item in enumerate(
    discovered,
    start=1,
):

    name = item[
        "kampanya_adi"
    ]

    url = item[
        "kaynak_url"
    ]


    print()
    print(
        f"[{index:02d}/"
        f"{len(discovered):02d}] "
        f"{name}"
    )


    try:

        response = (
            request_session.get(
                url,
                timeout=40,
                allow_redirects=True,
            )
        )


        if (
            response.status_code
            != 200
        ):

            failed.append({
                "kampanya_adi":
                    name,

                "kaynak_url":
                    url,

                "status":
                    response.status_code,
            })

            print(
                "  RESULT: ❌ "
                f"HTTP "
                f"{response.status_code}"
            )

            continue


        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )


        h1 = soup.find(
            "h1"
        )


        h1_text = (
            clean_text(
                h1.get_text(
                    " ",
                    strip=True
                )
            )
            if h1
            else ""
        )


        if not name:

            name = h1_text


        raw_text = (
            extract_detail_text(
                soup
            )
        )


        if (
            len(raw_text)
            < 100
        ):

            failed.append({
                "kampanya_adi":
                    name,

                "kaynak_url":
                    url,

                "error":
                    "raw_text_too_short",
            })

            print(
                "  RESULT: ❌ "
                "TEXT TOO SHORT"
            )

            continue


        dates = extract_dates(
            raw_text
        )


        records.append({
            "kampanya_adi":
                name,

            "kategori":
                item[
                    "kategori"
                ],

            "discovery_categories":
                item[
                    "discovery_categories"
                ],

            "kart_aciklama":
                item[
                    "kart_aciklama"
                ],

            "kaynak_url":
                response.url
                .split("#")[0],

            "http_status":
                response.status_code,

            "h1":
                h1_text,

            "tarih_adaylari":
                dates,

            "ham_metin":
                raw_text,
        })


        print(
            "  HTTP :",
            response.status_code
        )

        print(
            "  H1   :",
            h1_text
        )

        print(
            "  Text :",
            len(raw_text)
        )

        print(
            "  Tarih:",
            dates
        )

        print(
            "  RESULT: ✅"
        )


        time.sleep(
            0.1
        )


    except Exception as error:

        failed.append({
            "kampanya_adi":
                name,

            "kaynak_url":
                url,

            "error":
                (
                    f"{type(error).__name__}: "
                    f"{error}"
                ),
        })


        print(
            "  RESULT: ❌",
            type(error).__name__,
            error
        )


# ============================================================
# AUDIT
# ============================================================

urls = [
    x[
        "kaynak_url"
    ]
    for x in records
]

names = [
    x[
        "kampanya_adi"
    ]
    for x in records
]


duplicate_url = (
    len(urls)
    -
    len(set(urls))
)

duplicate_name = (
    len(names)
    -
    len(set(names))
)


with_dates = sum(
    bool(
        x[
            "tarih_adaylari"
        ]
    )
    for x in records
)


load_more_problems = []


for item in category_results:

    if (
        item[
            "click_count"
        ] > 0
        and
        item[
            "no_growth"
        ]
    ):

        load_more_problems.append(
            item[
                "category"
            ]
        )


errors = []


if not discovered:

    errors.append(
        "Hiç kampanya bulunamadı."
    )


if (
    len(records)
    !=
    len(discovered)
):

    errors.append(
        (
            "Detail count "
            f"{len(records)} != "
            f"{len(discovered)}"
        )
    )


if failed:

    errors.append(
        (
            "Detail fetch error: "
            f"{len(failed)}"
        )
    )


if duplicate_url:

    errors.append(
        (
            "Duplicate URL: "
            f"{duplicate_url}"
        )
    )


if duplicate_name:

    errors.append(
        (
            "Duplicate title: "
            f"{duplicate_name}"
        )
    )


# ============================================================
# SAVE
# ============================================================

output = {
    "banka":
        (
            "Albaraka Türk "
            "Katılım Bankası A.Ş."
        ),

    "ana_sayfa":
        MAIN_URL,

    "discovery_method":
        "playwright_category_union",

    "category_results": [
        {
            "category":
                x[
                    "category"
                ],

            "url":
                x[
                    "url"
                ],

            "initial_count":
                x[
                    "initial_count"
                ],

            "final_count":
                x[
                    "final_count"
                ],

            "click_count":
                x[
                    "click_count"
                ],

            "no_growth":
                x[
                    "no_growth"
                ],

            "ajax_results":
                x[
                    "ajax_results"
                ],
        }
        for x
        in category_results
    ],

    "category_reference_count":
        len(
            all_candidates
        ),

    "unique_campaign_count":
        len(
            discovered
        ),

    "raw_count":
        len(
            records
        ),

    "tarih_bulunan":
        with_dates,

    "duplicate_url":
        duplicate_url,

    "duplicate_title":
        duplicate_name,

    "load_more_problem_categories":
        load_more_problems,

    "validation_error":
        len(
            errors
        ),

    "kampanyalar":
        records,

    "failed":
        failed,

    "errors":
        errors,
}


with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        output,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# FINAL SUMMARY
# ============================================================

print()
print("=" * 120)
print(
    "ALBARAKA TÜRK - "
    "CAMPAIGN PLAYWRIGHT RAW V6 SONUCU"
)
print("=" * 120)

print(
    "Kategori sayısı     :",
    len(
        category_results
    )
)

print(
    "Category refs       :",
    len(
        all_candidates
    )
)

print(
    "Unique campaign     :",
    len(
        discovered
    )
)

print(
    "RAW detail success  :",
    len(
        records
    )
)

print(
    "Detail fetch error  :",
    len(
        failed
    )
)

print(
    "Tarih bulunan       :",
    (
        f"{with_dates} / "
        f"{len(records)}"
    )
)

print(
    "Duplicate URL       :",
    duplicate_url
)

print(
    "Duplicate title     :",
    duplicate_name
)

print(
    "Load-more problem   :",
    load_more_problems
)

print(
    "Validation error    :",
    len(
        errors
    )
)

print(
    "RAW JSON            :",
    OUTPUT_FILE
)


if errors:

    print()
    print("=" * 120)
    print("HATALAR")
    print("=" * 120)

    for error in errors:

        print(
            "-",
            error
        )


print()
print("=" * 120)

if not errors:

    print(
        "SONUÇ: ALBARAKA TÜRK "
        "KAMPANYA RAW "
        f"{len(records)}/"
        f"{len(discovered)} "
        "TEKNİK OLARAK BAŞARILI ✅"
    )

else:

    print(
        "SONUÇ: ALBARAKA TÜRK "
        "KAMPANYA RAW "
        "KONTROL GEREKİYOR ⚠️"
    )

print("=" * 120)


files.download(
    OUTPUT_FILE
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 6.5 MB/s eta 0:00:00
Installing dependencies...
Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:5 https://cli.github.com/packages stable InRelease
Hit:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state informati

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# ALBARAKA TÜRK
# CAMPAIGN SEMANTIC INSPECTOR V1
#
# Input:
#   /content/albaraka_turk_kampanyalar_raw.json
#
# Amaç:
#   48 kampanyanın final extractor öncesi
#   semantik adaylarını görmek.
# ============================================================

import json
import re


INPUT_FILE = (
    "/content/"
    "albaraka_turk_kampanyalar_raw.json"
)


# ============================================================
# HELPERS
# ============================================================

def clean_text(value):
    value = str(value or "")

    value = (
        value
        .replace("\xa0", " ")
        .replace("’", "'")
        .replace("‘", "'")
        .replace("–", "-")
        .replace("—", "-")
        .replace("\u00ad", "")
    )

    value = re.sub(
        r"[ \t]+",
        " ",
        value
    )

    value = re.sub(
        r"\n[ \t]+",
        "\n",
        value
    )

    value = re.sub(
        r"\n{3,}",
        "\n\n",
        value
    )

    return value.strip()


def normalize(value):
    return (
        clean_text(value)
        .replace("İ", "i")
        .replace("I", "ı")
        .casefold()
    )


def unique(values):
    result = []
    seen = set()

    for value in values:
        value = clean_text(value)

        if not value:
            continue

        key = normalize(value)

        if key in seen:
            continue

        seen.add(key)
        result.append(value)

    return result


def sentences(text):
    text = clean_text(text)

    parts = re.split(
        r"\n+|(?<=[.!?])\s+",
        text
    )

    return [
        clean_text(x)
        for x in parts
        if clean_text(x)
    ]


# ============================================================
# PERCENT
# ============================================================

def percent_candidates(text):
    values = re.findall(
        r"%\s*\d+(?:[.,]\d+)?",
        text
    )

    return unique(
        x.replace(" ", "")
        for x in values
    )


# ============================================================
# MONEY
# ============================================================

def money_candidates(text):
    result = []

    patterns = [
        (
            r"\b"
            r"\d{1,3}"
            r"(?:[.\s]\d{3})+"
            r"(?:,\d+)?"
            r"\s*(?:TL|₺)"
            r"\b"
        ),
        (
            r"\b"
            r"\d+(?:,\d+)?"
            r"\s*(?:TL|₺)"
            r"\b"
        ),
    ]

    for pattern in patterns:
        for match in re.finditer(
            pattern,
            text,
            flags=re.I
        ):
            value = clean_text(
                match.group(0)
            ).replace("₺", "TL")

            result.append(value)

    return unique(result)


# ============================================================
# WORLDPUAN
# ============================================================

def worldpuan_candidates(text):
    result = []

    pattern = (
        r"\b"
        r"\d{1,3}"
        r"(?:\.\d{3})*"
        r"(?:,\d+)?"
        r"\s*TL"
        r"\s*Worldpuan"
        r"\b"
    )

    for match in re.finditer(
        pattern,
        text,
        flags=re.I
    ):
        result.append(
            clean_text(
                match.group(0)
            )
        )

    return unique(result)


# ============================================================
# TAKSIT
# ============================================================

def installment_candidates(text):
    result = []

    patterns = [
        r"\b\d+\s+taksit\b",
        r"\b\d+\s+taksitli\b",
        r"\b\d+\s+eşit\s+taksit\b",
        r"\b\d+\s+aya\s+varan\s+taksit\b",
    ]

    for pattern in patterns:
        for match in re.finditer(
            pattern,
            text,
            flags=re.I
        ):
            result.append(
                clean_text(
                    match.group(0)
                )
            )

    return unique(result)


# ============================================================
# DATE RANGE SENTENCES
# ============================================================

def campaign_period_candidates(text):
    result = []

    keywords = [
        "kampanya",
        "geçerlidir",
        "geçerli",
        "tarihleri arasında",
        "tarihleri arasinda",
        "son katılım",
        "son katilim",
    ]

    for sentence in sentences(text):
        n = normalize(sentence)

        has_date = bool(
            re.search(
                (
                    r"\d{1,2}[./-]\d{1,2}[./-]\d{4}"
                    r"|"
                    r"\d{1,2}\s+"
                    r"(?:ocak|şubat|mart|nisan|mayıs|haziran|"
                    r"temmuz|ağustos|eylül|ekim|kasım|aralık)"
                    r"\s+\d{4}"
                ),
                n,
                flags=re.I
            )
        )

        if (
            has_date
            and
            any(
                keyword in n
                for keyword in keywords
            )
        ):
            result.append(sentence)

    return unique(result)[:8]


# ============================================================
# ADVANTAGE SENTENCES
# ============================================================

def advantage_candidates(text):
    result = []

    keywords = [
        "worldpuan",
        "indirim",
        "iade",
        "ücretsiz",
        "hediye",
        "taksit",
        "vade farksız",
        "vade farksiz",
        "ödül",
        "kazanın",
        "kazanabilirsiniz",
        "avantaj",
        "destek",
        "paylaşım oran",
        "paylasim oran",
        "masraf",
        "ücret",
        "internet",
        "gb",
    ]

    for sentence in sentences(text):
        n = normalize(sentence)

        if not any(
            key in n
            for key in keywords
        ):
            continue

        if not (
            10
            <= len(sentence)
            <= 550
        ):
            continue

        result.append(sentence)

    return unique(result)[:12]


# ============================================================
# TARGET
# ============================================================

def target_candidates(text):
    result = []

    keywords = [
        "müşteri",
        "müşteriler",
        "müşterilerimiz",
        "kart sahip",
        "kart sahibi",
        "albarakalı",
        "emekli",
        "yeni müşteri",
        "dijital müşteri",
        "özel bankacılık",
        "trend",
        "eflatun",
        "bireysel",
        "18 yaş",
    ]

    for sentence in sentences(text):
        n = normalize(sentence)

        if not any(
            key in n
            for key in keywords
        ):
            continue

        if not (
            15
            <= len(sentence)
            <= 500
        ):
            continue

        result.append(sentence)

    return unique(result)[:8]


# ============================================================
# CONDITION CANDIDATES
# ============================================================

def condition_candidates(text):
    result = []

    keywords = [
        "yararlan",
        "geçerli",
        "geçerlidir",
        "katılım",
        "katilim",
        "talimat",
        "harcama",
        "işlem",
        "islem",
        "tek seferde",
        "aynı gün",
        "ayni gün",
        "kampanyadan",
        "kart",
        "mobil",
        "başvuru",
        "basvuru",
        "sadece",
        "en az",
        "en fazla",
        "dahil",
        "hariç",
        "haric",
    ]

    for sentence in sentences(text):
        n = normalize(sentence)

        if not any(
            key in n
            for key in keywords
        ):
            continue

        if not (
            20
            <= len(sentence)
            <= 650
        ):
            continue

        result.append(sentence)

    return unique(result)[:12]


# ============================================================
# CAMPAIGN TYPE HEURISTIC
# Sadece audit için.
# ============================================================

def type_candidates(title, text):
    combined = normalize(
        title + "\n" + text
    )

    result = []

    if "worldpuan" in combined:
        result.append("Worldpuan")

    if "indirim" in combined:
        result.append("İndirim")

    if "iade" in combined:
        result.append("İade")

    if "ücretsiz" in combined:
        result.append("Ücretsiz Hizmet")

    if "taksit" in combined:
        result.append("Taksit")

    if (
        "finansman"
        in combined
        or
        "destek"
        in combined
    ):
        result.append(
            "Finansman Avantajı"
        )

    if (
        "paylaşım oran"
        in combined
        or
        "paylasim oran"
        in combined
    ):
        result.append(
            "Paylaşım Oranı"
        )

    if "promosyon" in combined:
        result.append("Promosyon")

    return unique(result)


# ============================================================
# LOAD
# ============================================================

with open(
    INPUT_FILE,
    "r",
    encoding="utf-8"
) as f:
    data = json.load(f)


records = data.get(
    "kampanyalar",
    []
)


print("=" * 125)
print(
    "ALBARAKA TÜRK - "
    "CAMPAIGN SEMANTIC INSPECTOR V1"
)
print("=" * 125)

print(
    "Toplam RAW kampanya:",
    len(records)
)


# ============================================================
# COUNTERS
# ============================================================

with_dates = 0
with_percent = 0
with_money = 0
with_worldpuan = 0
with_installment = 0
redirect_like = []


# ============================================================
# EACH CAMPAIGN
# ============================================================

for index, record in enumerate(
    records,
    start=1
):

    title = clean_text(
        record.get(
            "kampanya_adi",
            ""
        )
    )

    h1 = clean_text(
        record.get(
            "h1",
            ""
        )
    )

    text = clean_text(
        record.get(
            "ham_metin",
            ""
        )
    )

    dates = record.get(
        "tarih_adaylari",
        []
    )

    percents = (
        percent_candidates(text)
    )

    money = money_candidates(
        text
    )

    worldpuan = (
        worldpuan_candidates(
            text
        )
    )

    installments = (
        installment_candidates(
            text
        )
    )

    periods = (
        campaign_period_candidates(
            text
        )
    )

    advantages = (
        advantage_candidates(
            text
        )
    )

    targets = target_candidates(
        text
    )

    conditions = (
        condition_candidates(
            text
        )
    )

    types = type_candidates(
        title,
        text
    )


    if dates:
        with_dates += 1

    if percents:
        with_percent += 1

    if money:
        with_money += 1

    if worldpuan:
        with_worldpuan += 1

    if installments:
        with_installment += 1


    mismatch = (
        bool(title)
        and
        bool(h1)
        and
        normalize(title)
        !=
        normalize(h1)
    )


    if mismatch:
        redirect_like.append(
            title
        )


    print()
    print(
        f"[{index:02d}/"
        f"{len(records):02d}] "
        f"{title}"
    )

    print(
        "  H1       :",
        h1
    )

    print(
        "  Redirect?:",
        mismatch
    )

    print(
        "  Cats     :",
        record.get(
            "discovery_categories",
            []
        )
    )

    print(
        "  Tür      :",
        types
    )

    print(
        "  Tarih    :",
        dates
    )

    print(
        "  %        :",
        percents
    )

    print(
        "  TL       :",
        money
    )

    print(
        "  Worldpuan:",
        worldpuan
    )

    print(
        "  Taksit   :",
        installments
    )

    print(
        "  Süre Cüm :",
        periods[:4]
    )

    print(
        "  Avantaj  :",
        advantages[:6]
    )

    print(
        "  Hedef    :",
        targets[:4]
    )

    print(
        "  Koşul    :",
        conditions[:6]
    )


# ============================================================
# SUMMARY
# ============================================================

print()
print("=" * 125)
print("SEMANTIC AUDIT ÖZETİ")
print("=" * 125)

print(
    "Toplam kampanya     :",
    len(records)
)

print(
    "Tarih bulunan       :",
    with_dates
)

print(
    "Yüzde bulunan       :",
    with_percent
)

print(
    "TL bulunan          :",
    with_money
)

print(
    "Worldpuan bulunan   :",
    with_worldpuan
)

print(
    "Taksit bulunan      :",
    with_installment
)

print(
    "H1/title mismatch   :",
    len(redirect_like)
)

print(
    "Redirect-like       :",
    redirect_like
)


print()
print("=" * 125)
print(
    "SEMANTIC INSPECTOR TAMAMLANDI ✅"
)
print("=" * 125)

ALBARAKA TÜRK - CAMPAIGN SEMANTIC INSPECTOR V1
Toplam RAW kampanya: 48

[01/48] Vade Farksız 140.000 TL'ye Varan Destek!
  H1       : Vade Farksız 140.000 TL'ye Varan Destek!
  Redirect?: False
  Cats     : ['tumu', 'dijital']
  Tür      : ['Ücretsiz Hizmet', 'Taksit', 'Finansman Avantajı']
  Tarih    : ['1 Ocak 2026', '31 Aralık 2026', '01.01.2026', '31.12.2026']
  %        : ['%0']
  TL       : ['140.000 TL', '40.000 TL', '100.000 TL', '1.000 TL', '30.000 TL', '500.000 TL', '000 TL']
  Worldpuan: []
  Taksit   : ['6 taksit', '4 taksitli']
  Süre Cüm : []
  Avantaj  : ["Vade Farksız 140.000 TL'ye Varan Destek!", 'vade farksız destek', 'seçili sektörlerde vade farksız taksitli alışveriş fırsatından yararlanabiliyor.', '3 aya varan ödemesiz dönem ve 4 taksitli', "'ye kadar vade farksız", 'taksitli alışveriş']
  Hedef    : ["Şimdi Albaraka Mobil'den müşteri olanlar,", 'Albaraka Türk müşterisi olun;', '*Kampanya kapsamında Bankamıza iletilen Pratik Finansman Kart ve Kredi Kartı talepleri 

In [ ]:
# ============================================================
# ALBARAKA TÜRK
# CAMPAIGN EXTRACTOR FINAL V1
#
# Input:
#   /content/albaraka_turk_kampanyalar_raw.json
#
# Output:
#   /content/albaraka_turk_kampanya_extracted.json
#
# Hedef:
#   48 / 48
#   ortak 18-key schema
# ============================================================

import json
import re
from datetime import datetime, date
from collections import Counter

from google.colab import files


# ============================================================
# FILES
# ============================================================

INPUT_FILE = (
    "/content/"
    "albaraka_turk_kampanyalar_raw.json"
)

OUTPUT_FILE = (
    "/content/"
    "albaraka_turk_kampanya_extracted.json"
)

BANK_NAME = (
    "Albaraka Türk Katılım Bankası A.Ş."
)

TODAY = date(
    2026,
    8,
    23,
)


# ============================================================
# SCHEMA
# ============================================================

SCHEMA_KEYS = [
    "banka",
    "kayit_turu",
    "urun_adi",
    "urun_kategorisi",
    "kar_payi_orani",
    "finansman_orani",
    "finansman_tutari",
    "vade",
    "taksit_sayisi",
    "masraf_bilgisi",
    "kampanya_turu",
    "kampanya_avantaji",
    "kampanya_suresi",
    "hedef_kitle",
    "para_birimi",
    "kosullar",
    "kaynak_url",
    "ham_metin",
]


LIST_FIELDS = {
    "kar_payi_orani",
    "finansman_orani",
    "finansman_tutari",
    "vade",
    "taksit_sayisi",
    "masraf_bilgisi",
    "kampanya_avantaji",
    "hedef_kitle",
    "para_birimi",
    "kosullar",
}


SCALAR_FIELDS = (
    set(SCHEMA_KEYS)
    - LIST_FIELDS
)


# ============================================================
# HELPERS
# ============================================================

def clean_text(value):

    value = str(value or "")

    value = (
        value
        .replace("\xa0", " ")
        .replace("’", "'")
        .replace("‘", "'")
        .replace("–", "-")
        .replace("—", "-")
        .replace("\u00ad", "")
    )

    value = re.sub(
        r"[ \t]+",
        " ",
        value
    )

    value = re.sub(
        r"\n[ \t]+",
        "\n",
        value
    )

    value = re.sub(
        r"\n{3,}",
        "\n\n",
        value
    )

    return value.strip()


def normalize(value):

    return (
        clean_text(value)
        .replace("İ", "i")
        .replace("I", "ı")
        .casefold()
    )


def unique(values):

    result = []
    seen = set()

    for value in values:

        value = clean_text(
            value
        )

        if not value:
            continue

        key = normalize(
            value
        )

        if key in seen:
            continue

        seen.add(key)
        result.append(value)

    return result


def units(text):

    text = clean_text(text)

    result = []

    for line in text.splitlines():

        line = clean_text(line)

        if not line:
            continue

        parts = re.split(
            r"(?<=[.!?])\s+",
            line
        )

        for part in parts:

            part = clean_text(part)

            if part:
                result.append(part)

    return result


# ============================================================
# CATEGORY
# ============================================================

CATEGORY_LABELS = {
    "dijital":
        "Dijital",

    "eflatun":
        "Eflatun",

    "trend":
        "Trend",

    "ozel-bankacilik":
        "Özel Bankacılık",

    "bireysel":
        "Bireysel",

    "business":
        "Business",

    "world-kampanyalari":
        "World Kampanyaları",
}


def extract_category(raw):

    categories = raw.get(
        "discovery_categories",
        []
    )

    for category in categories:

        if category == "tumu":
            continue

        if category in CATEGORY_LABELS:

            return CATEGORY_LABELS[
                category
            ]

    return "Genel"


# ============================================================
# CAMPAIGN TYPE
#
# Inspector heuristic'i finalde doğrudan kullanılmıyor.
# Semantik olarak sabitlenmiştir.
# ============================================================

CAMPAIGN_TYPES = {

    "Vade Farksız 140.000 TL'ye Varan Destek!":
        "Finansman Avantajı + Taksit",

    "Ağustos Ayına Özel Fatura Kampanyası":
        "Worldpuan",

    "Yurt Dışı Çıkış Harcı Kampanyası | 1.250 TL Worldpuan!":
        "Worldpuan",

    "Yakınını Davet Et Kampanyası":
        "Worldpuan",

    "Dijital Katılma Hesabı'na Özel Paylaşım Oranları!":
        "Paylaşım Oranı",

    "Ücretsiz İSPARK Otopark Kampanyası | Albarakalılara Özel":
        "Ücretsiz Hizmet",

    "Sağlık Harcamalarına Vade Farksız 6 Taksit Kampanyası":
        "Taksit",

    "Eğitim Harcamalarınıza Vade Farksız 6 Taksit Kampanyası":
        "Taksit",

    "Kırtasiye Kampanyası":
        "Taksit",

    "Otomatik Fatura Ödeme Talimatlarınıza Toplamda 2.000 TL Worldpuan!":
        "Worldpuan",

    "HGS Talimatınıza 500 TL Worldpuan!":
        "Worldpuan",

    "QR Kod İle Ödemelerinize Toplamda 300 TL Worldpuan!":
        "Worldpuan",

    "Akaryakıt Kampanyası":
        "Worldpuan",

    "Restoran Kampanyası":
        "Worldpuan",

    "Otopark ve Vale Harcamalarınıza %50 İade Albaraka'da!":
        "İade",

    "Albaraka ile Geleceğim Güvende!":
        "Worldpuan",

    "Albaraka ile Kazandıran Sigorta!":
        "Worldpuan",

    "DASK Sigortasında Worldpuan Fırsatı!":
        "Worldpuan + Taksit",

    "8 Taksit Fırsatıyla KASKO Zamanı!":
        "Taksit",

    "Limitsiz İMM Sigortasında Vade Farksız 3 Taksit!":
        "Taksit",

    'Albaraka Mobil\'de Şimdi "Seçkin Fırsatlar" Zamanı!':
        "İndirim",

    "Kahve Keyfiniz Albaraka'dan!":
        "Hediye",

    "Hızlı Çiçek %20 İndirim Kampanyası":
        "İndirim",

    "Albaraka Sadakat Programı":
        "Sadakat Programı",

    "Taksitlio.com Alışveriş Finansmanı":
        "Finansman Avantajı",

    "albaFX'te Karma Düzey 1 Ücretsiz!":
        "Ücretsiz Hizmet",

    "Dijitale Özel Konut ve Taşıt Finansmanı Kampanyası":
        "Finansman Avantajı",

    "Satırdan Sanata, Ek %10 İndirim ve Vade Farksız 4 Taksit Albaraka'da!":
        "İndirim + Taksit",

    "Trend Kredi Kart Kırtasiye Harcamaları İndirim Kampanyası!":
        "İndirim",

    "Trend Kredi Kart ile Sinema Harcamalarına %25 İndirim!":
        "İndirim",

    "Payını Sen Seç Finansmanı":
        "Finansman Avantajı",

    "Hızlı Döviz İşlemleri Yanınızda!":
        "Kur Avantajı",

    "Enterprise Araç Kiralama Kampanyası":
        "İndirim",

    "Enterprise Araç Kiralama Kampanyası | %35 İndirim!":
        "İndirim",

    "Dijital Müşterilere Özel Pratik Finansman Kart":
        "Finansman Avantajı + Taksit",

    "Bilet.com ile Otel Rezervasyonlarında İndirim!":
        "İndirim",

    "Bilet.com ile Yunan Adaları Feribot Bileti Alımlarında İndirim!":
        "İndirim",

    "Albaraka Müşterilerine Özel Marina Aquapark'ta %20 indirim!":
        "İndirim",

    "Togg Taşıt Finansmanı Kampanyası":
        "Finansman Avantajı",

    "Umre Finansmanı Kampanyası":
        "Finansman Avantajı",

    "Eflatun Kredi Kart ile Kozmetik Harcamalarına 200 TL'ye varan Worldpuan!":
        "Worldpuan",

    "Trend 4 GB İnternet Kampanyası":
        "Hediye",

    "Restoran Harcamalarınıza Özel %10 İndirim Fırsatı!":
        "İade",

    "Otopark Harcamalarınıza Özel %10 İndirim Fırsatı!":
        "İade",

    "Konaklamanın Ayrıcalıklı Hali!":
        "İade",

    "Emekli Promosyon 2026 | 30.000 TL'ye Varan Ödül!":
        "Promosyon",

    "Albaraka'da Masraflara Son!":
        "Ücretsiz Hizmet",

    "Ücretsiz Ortak ATM Kampanyası":
        "Ücretsiz Hizmet",
}


# ============================================================
# CORE ADVANTAGE
#
# Sadece kampanyanın esas faydası.
# Harcama eşikleri / ödül yükleme tarihleri / hukuki
# açıklamalar burada tutulmaz.
# ============================================================

ADVANTAGES = {

    "Vade Farksız 140.000 TL'ye Varan Destek!": [
        "140.000 TL'ye varan vade farksız destek",
        "Seçili sektörlerde vade farksız taksitli alışveriş imkânı",
    ],

    "Ağustos Ayına Özel Fatura Kampanyası": [
        "Her yeni otomatik fatura ödeme talimatına 500 TL, toplamda 2.000 TL Worldpuan",
    ],

    "Yurt Dışı Çıkış Harcı Kampanyası | 1.250 TL Worldpuan!": [
        "1.250 TL Worldpuan",
    ],

    "Yakınını Davet Et Kampanyası": [
        "Davet edilen her yeni müşteri için 500 TL, toplamda 5.000 TL'ye varan Worldpuan",
        "Davet edilen müşteri için 500 TL Worldpuan",
    ],

    "Dijital Katılma Hesabı'na Özel Paylaşım Oranları!": [
        "98/2 kâr paylaşım oranlı Dijital Katılma Hesabı",
    ],

    "Ücretsiz İSPARK Otopark Kampanyası | Albarakalılara Özel": [
        "Ayda 2 defa ücretsiz İSPARK kullanımı",
    ],

    "Sağlık Harcamalarına Vade Farksız 6 Taksit Kampanyası": [
        "1.000 TL-100.000 TL arası sağlık harcamalarına vade farksız 6 taksit",
    ],

    "Eğitim Harcamalarınıza Vade Farksız 6 Taksit Kampanyası": [
        "30.000 TL-500.000 TL arası okul ödemelerine vade farksız 6 taksit",
    ],

    "Kırtasiye Kampanyası": [
        "1.000 TL-20.000 TL arası kırtasiye harcamalarına vade farksız 4 taksit",
    ],

    "Otomatik Fatura Ödeme Talimatlarınıza Toplamda 2.000 TL Worldpuan!": [
        "Her yeni otomatik fatura ödeme talimatına 200 TL, toplamda 2.000 TL'ye varan Worldpuan",
    ],

    "HGS Talimatınıza 500 TL Worldpuan!": [
        "İlk HGS etiketi için 500 TL Worldpuan",
    ],

    "QR Kod İle Ödemelerinize Toplamda 300 TL Worldpuan!": [
        "QR kod ile ödemelerde toplamda 300 TL'ye varan Worldpuan",
    ],

    "Akaryakıt Kampanyası": [
        "Her 1.500 TL ve üzeri akaryakıt harcamasına 50 TL, toplamda 200 TL'ye varan Worldpuan",
    ],

    "Restoran Kampanyası": [
        "Her 2.000 TL ve üzeri restoran harcamasına 75 TL, toplamda 300 TL'ye varan Worldpuan",
    ],

    "Otopark ve Vale Harcamalarınıza %50 İade Albaraka'da!": [
        "Otopark ve vale harcamalarında %50 iade",
        "Toplamda 300 TL'ye varan iade",
    ],

    "Albaraka ile Geleceğim Güvende!": [
        "BES ve Erken BES alımlarında toplamda 2.000 TL'ye varan Worldpuan",
    ],

    "Albaraka ile Kazandıran Sigorta!": [
        "Sigorta alımlarında toplamda 2.000 TL'ye varan Worldpuan",
    ],

    "DASK Sigortasında Worldpuan Fırsatı!": [
        "DASK alımında 300 TL Worldpuan",
        "3 taksit fırsatı",
    ],

    "8 Taksit Fırsatıyla KASKO Zamanı!": [
        "Kasko işlemlerinde 8 taksite varan ödeme fırsatı",
    ],

    "Limitsiz İMM Sigortasında Vade Farksız 3 Taksit!": [
        "Limitsiz İMM Sigortasında vade farksız 3 taksit",
    ],

    'Albaraka Mobil\'de Şimdi "Seçkin Fırsatlar" Zamanı!': [
        "Seçkin Fırsatlar kapsamında farklı sektörlerde indirim ve ayrıcalıklar",
    ],

    "Kahve Keyfiniz Albaraka'dan!": [
        "Elmas ve Platin müşterilere haftada 1 kahve hediyesi",
    ],

    "Hızlı Çiçek %20 İndirim Kampanyası": [
        "Hızlı Çiçek alışverişlerinde %20 indirim",
    ],

    "Albaraka Sadakat Programı": [
        "Sadakat segmenti yükseldikçe artan avantaj, indirim ve ayrıcalıklar",
    ],

    "Taksitlio.com Alışveriş Finansmanı": [
        "Taksitlio.com anlaşmalı mağazalarda alışveriş finansmanı imkânı",
    ],

    "albaFX'te Karma Düzey 1 Ücretsiz!": [
        "albaFX Karma Düzey 1 yapay zekâ destekli araştırma raporlarına ücretsiz erişim",
    ],

    "Dijitale Özel Konut ve Taşıt Finansmanı Kampanyası": [
        "Konut ve taşıt finansmanında dijitale özel avantajlı oranlar",
    ],

    "Satırdan Sanata, Ek %10 İndirim ve Vade Farksız 4 Taksit Albaraka'da!": [
        "Kültür Sanat Yayıncılık'ta mevcut indirimlere ek %10 indirim",
        "Vade farksız 4 taksit",
    ],

    "Trend Kredi Kart Kırtasiye Harcamaları İndirim Kampanyası!": [
        "Kırtasiye harcamalarında %10 indirim",
        "Ayda en fazla 250 TL indirim",
    ],

    "Trend Kredi Kart ile Sinema Harcamalarına %25 İndirim!": [
        "Sinema harcamalarında %25 indirim",
        "Ayda en fazla 250 TL indirim",
    ],

    "Payını Sen Seç Finansmanı": [
        "Payını Sen Seç modeliyle daha avantajlı toplam geri ödeme imkânı",
    ],

    "Hızlı Döviz İşlemleri Yanınızda!": [
        "Altın, gümüş ve döviz işlemlerinde hafta içi 24 saat dar makas ve avantajlı kur",
    ],

    "Enterprise Araç Kiralama Kampanyası": [
        "Enterprise online araç kiralamalarında liste fiyatı üzerinden %30 indirim",
    ],

    "Enterprise Araç Kiralama Kampanyası | %35 İndirim!": [
        "Enterprise online araç kiralamalarında liste fiyatı üzerinden %35 indirim",
    ],

    "Dijital Müşterilere Özel Pratik Finansman Kart": [
        "40.000 TL'ye kadar vade farksız Pratik Finansman Kart fırsatı",
        "Uygun oranlı hoş geldin finansmanı",
    ],

    "Bilet.com ile Otel Rezervasyonlarında İndirim!": [
        "10.000 TL ve üzeri otel rezervasyonlarında 2.500 TL indirim",
    ],

    "Bilet.com ile Yunan Adaları Feribot Bileti Alımlarında İndirim!": [
        "Yunan Adaları gidiş-dönüş feribot bileti alımlarında 500 TL indirim",
    ],

    "Albaraka Müşterilerine Özel Marina Aquapark'ta %20 indirim!": [
        "Marina Aquapark gişe ödemelerinde %20 indirim",
    ],

    # Kampanya kartı artık ürün sayfasına redirect oluyor.
    # Eski kampanya şartı uydurulmuyor.
    "Togg Taşıt Finansmanı Kampanyası": [],

    "Umre Finansmanı Kampanyası": [],

    "Eflatun Kredi Kart ile Kozmetik Harcamalarına 200 TL'ye varan Worldpuan!": [
        "500 TL ve üzeri kozmetik harcamasına 50 TL, ayda toplam 200 TL'ye varan Worldpuan",
    ],

    "Trend 4 GB İnternet Kampanyası": [
        "Ayda 750 TL ve üzeri Trend Kredi Kart harcamasına 1 hafta geçerli 4 GB internet",
    ],

    "Restoran Harcamalarınıza Özel %10 İndirim Fırsatı!": [
        "Özel Bankacılık kredi kartı ile restoran harcamalarında %10 iade",
    ],

    "Otopark Harcamalarınıza Özel %10 İndirim Fırsatı!": [
        "5.000 TL'ye kadar otopark harcamalarında %10 iade",
    ],

    "Konaklamanın Ayrıcalıklı Hali!": [
        "10.000-25.000 TL arası konaklama harcamasına 1.000 TL iade",
        "25.001 TL ve üzeri konaklama harcamasına 2.000 TL iade",
    ],

    "Emekli Promosyon 2026 | 30.000 TL'ye Varan Ödül!": [
        "Emekli maaşını Albaraka'ya taşıyanlara 30.000 TL'ye varan ödül",
    ],

    "Albaraka'da Masraflara Son!": [
        "Hesap işletim ücreti alınmayan hesaplar",
        "Mobil, internet ve ATM kanallarında ücretsiz EFT, Havale ve FAST",
        "Aidatsız banka ve kredi kartı",
    ],

    "Ücretsiz Ortak ATM Kampanyası": [
        "8.500'den fazla ortak ATM'de ücretsiz işlem imkânı",
    ],
}


# ============================================================
# FINANCING CAMPAIGNS
# ============================================================

FINANCE_CAMPAIGNS = {
    "Vade Farksız 140.000 TL'ye Varan Destek!",
    "Taksitlio.com Alışveriş Finansmanı",
    "Dijitale Özel Konut ve Taşıt Finansmanı Kampanyası",
    "Payını Sen Seç Finansmanı",
    "Dijital Müşterilere Özel Pratik Finansman Kart",
}


REDIRECT_CAMPAIGNS = {
    "Togg Taşıt Finansmanı Kampanyası",
    "Umre Finansmanı Kampanyası",
    "Ücretsiz Ortak ATM Kampanyası",
}


# ============================================================
# EXPLICIT FINANCE AMOUNTS
#
# Kampanya ödülü / indirim eşiği ile karışmaması için
# yalnızca açık finansman kampanyalarında sabitleniyor.
# ============================================================

FINANCE_AMOUNT_OVERRIDES = {

    "Vade Farksız 140.000 TL'ye Varan Destek!": [
        "140.000 TL",
    ],

    "Taksitlio.com Alışveriş Finansmanı": [
        "150.000 TL",
    ],

    "Dijital Müşterilere Özel Pratik Finansman Kart": [
        "40.000 TL",
    ],
}


# ============================================================
# KÂR PAYI / FINANSMAN ORANI
# ============================================================

PERCENT_PATTERN = (
    r"%\s*\d+(?:[.,]\d+)?"
)


def extract_finance_rates(
    name,
    text,
):

    if name in REDIRECT_CAMPAIGNS:
        return [], []


    if name not in FINANCE_CAMPAIGNS:
        return [], []


    kar_payi = []
    finansman = []


    for unit in units(text):

        n = normalize(unit)


        percents = re.findall(
            PERCENT_PATTERN,
            unit
        )

        percents = [
            re.sub(
                r"\s+",
                "",
                x
            )
            for x in percents
        ]


        if not percents:
            continue


        # -----------------------------------------------
        # KÂR PAYI
        # -----------------------------------------------

        if any(
            marker in n
            for marker in [
                "kâr payı",
                "kar payı",
                "kâr oranı",
                "kar oranı",
                "kâr payi",
                "kar payi",
            ]
        ):

            kar_payi.extend(
                percents
            )

            continue


        # -----------------------------------------------
        # FİNANSMAN ORANI
        # -----------------------------------------------

        if any(
            marker in n
            for marker in [
                "finansman oran",
                "finansmanında %",
                "finansmaninda %",
                "aylık oran",
                "aylik oran",
            ]
        ):

            finansman.extend(
                percents
            )

            continue


        # Açık %0 vade farkı
        if (
            "%0"
            in [
                x.replace(
                    ",0",
                    ""
                )
                for x in percents
            ]
            and
            (
                "vade fark"
                in n
                or
                "vade farksız"
                in n
            )
        ):

            finansman.append(
                "%0"
            )


    # Kampanya 1'de audit açık %0 tespit etti.
    if (
        name
        ==
        "Vade Farksız 140.000 TL'ye Varan Destek!"
    ):

        finansman.append(
            "%0"
        )


    # 98/2, yüzde işaretiyle yazılmayan paylaşım oranıdır.
    if (
        name
        ==
        "Dijital Katılma Hesabı'na Özel Paylaşım Oranları!"
    ):

        kar_payi.append(
            "98/2"
        )


    return (
        unique(kar_payi),
        unique(finansman),
    )


# ============================================================
# INSTALLMENTS
# ============================================================

def extract_installments(
    text
):

    result = []


    patterns = [
        r"\b(\d{1,2})\s+taksit\b",
        r"\b(\d{1,2})\s+taksitli\b",
        r"\b(\d{1,2})\s+eşit\s+taksit\b",
        r"\b(\d{1,2})\s+taksite\b",
    ]


    for pattern in patterns:

        for match in re.finditer(
            pattern,
            text,
            flags=re.I
        ):

            result.append(
                match.group(1)
            )


    return unique(
        result
    )


# ============================================================
# VADE
#
# "6 taksit" vade değildir.
# Yalnızca açık "vade / ay vade / aya kadar vade" ifadeleri.
# ============================================================

def extract_terms(
    name,
    text
):

    if (
        name
        not in FINANCE_CAMPAIGNS
    ):

        return []


    result = []


    patterns = [
        r"\b\d+\s+ay\s+vade\b",
        r"\b\d+\s+aya\s+kadar\s+vade\b",
        r"\b\d+\s+aya\s+varan\s+vade\b",
        r"\bvade\s+süresi\s+\d+\s+ay\b",
    ]


    for pattern in patterns:

        for match in re.finditer(
            pattern,
            text,
            flags=re.I
        ):

            value = clean_text(
                match.group(0)
            )

            # ödemesiz dönem = vade değil
            context_start = max(
                0,
                match.start() - 50
            )

            context_end = min(
                len(text),
                match.end() + 50
            )

            context = normalize(
                text[
                    context_start:
                    context_end
                ]
            )


            if "ödemesiz" in context:
                continue


            result.append(value)


    return unique(
        result
    )


# ============================================================
# FEE INFORMATION
# ============================================================

def extract_fees(
    name,
    text
):

    result = []


    markers = [
        "ücret alın",
        "ücret öde",
        "ücret tahsil",
        "aidat",
        "masraf",
        "komisyon",
        "hesap işletim",
        "hesap isletim",
    ]


    for unit in units(text):

        n = normalize(unit)


        if not any(
            marker in n
            for marker in markers
        ):
            continue


        if (
            len(unit)
            < 20
            or
            len(unit)
            > 700
        ):
            continue


        result.append(unit)


    return unique(
        result
    )[:12]


# ============================================================
# CAMPAIGN PERIOD
#
# RAW scraper date candidates:
# - sözel tarihler
# - numeric tarihler
#
# Numeric çift, gerçek campaign start/end olarak
# kullanılıyor. Reward yükleme tarihleri sözel tarafta
# kalıyor.
# ============================================================

NUMERIC_DATE_RE = (
    r"^\d{1,2}[./-]\d{1,2}[./-]\d{4}$"
)


def normalize_date_string(value):

    value = clean_text(value)

    for fmt in [
        "%d.%m.%Y",
        "%d/%m/%Y",
        "%d-%m-%Y",
    ]:

        try:

            parsed = datetime.strptime(
                value,
                fmt
            )

            return parsed.strftime(
                "%d.%m.%Y"
            )

        except Exception:
            pass

    return ""


def extract_period(
    name,
    raw
):

    if name in REDIRECT_CAMPAIGNS:
        return ""


    candidates = raw.get(
        "tarih_adaylari",
        []
    )


    numeric = []


    for value in candidates:

        value = clean_text(value)


        if not re.match(
            NUMERIC_DATE_RE,
            value
        ):

            continue


        normalized = normalize_date_string(
            value
        )


        if normalized:

            numeric.append(
                normalized
            )


    numeric = unique(
        numeric
    )


    if len(numeric) >= 2:

        # Son numeric çift scraper çıktısında
        # kampanya start/end değerleridir.
        start = numeric[-2]
        end = numeric[-1]

        return (
            f"{start} - {end}"
        )


    return ""


# ============================================================
# TARGET AUDIENCE
# ============================================================

TARGET_OVERRIDES = {

    "Kahve Keyfiniz Albaraka'dan!": [
        "Elmas ve Platin müşteriler",
    ],

    "Trend Kredi Kart Kırtasiye Harcamaları İndirim Kampanyası!": [
        "Trend kredi kartı sahipleri",
    ],

    "Trend Kredi Kart ile Sinema Harcamalarına %25 İndirim!": [
        "Trend kredi kartı sahipleri",
    ],

    "Enterprise Araç Kiralama Kampanyası | %35 İndirim!": [
        "Albaraka Özel Bankacılık kredi kartı sahipleri",
    ],

    "Dijital Müşterilere Özel Pratik Finansman Kart": [
        "Albaraka Mobil üzerinden görüntülü görüşmeyle yeni müşteri olanlar",
    ],

    "Eflatun Kredi Kart ile Kozmetik Harcamalarına 200 TL'ye varan Worldpuan!": [
        "Eflatun kredi kartı sahipleri",
    ],

    "Trend 4 GB İnternet Kampanyası": [
        "Trend kredi kartı sahipleri",
    ],

    "Restoran Harcamalarınıza Özel %10 İndirim Fırsatı!": [
        "Albaraka Özel Bankacılık kredi kartı sahipleri",
    ],

    "Otopark Harcamalarınıza Özel %10 İndirim Fırsatı!": [
        "Albaraka Özel Bankacılık kredi kartı sahibi bireysel müşteriler",
    ],

    "Konaklamanın Ayrıcalıklı Hali!": [
        "Albaraka Özel Bankacılık kredi kartı sahibi bireysel müşteriler",
    ],

    "Emekli Promosyon 2026 | 30.000 TL'ye Varan Ödül!": [
        "Emekli maaşını Albaraka'ya taşıyan müşteriler",
    ],

    "Ücretsiz Ortak ATM Kampanyası": [
        "Albaraka Türk müşterileri",
    ],
}


def extract_targets(
    name,
    text
):

    if name in TARGET_OVERRIDES:

        return TARGET_OVERRIDES[
            name
        ]


    if (
        name
        in {
            "Togg Taşıt Finansmanı Kampanyası",
            "Umre Finansmanı Kampanyası",
        }
    ):

        return []


    result = []


    markers = [
        "müşteriler faydalanabilir",
        "müşterilerimiz faydalanabilir",
        "müşteri olanlar",
        "ilk kez müşteri",
        "kart sahibi müşter",
        "kredi kartı sahibi",
        "bireysel müşter",
        "gerçek kişiler",
        "emekli maaş",
    ]


    for unit in units(text):

        n = normalize(unit)


        if not any(
            marker in n
            for marker in markers
        ):

            continue


        if (
            len(unit)
            < 20
            or
            len(unit)
            > 550
        ):

            continue


        result.append(unit)


    return unique(
        result
    )[:8]


# ============================================================
# CONDITIONS
# ============================================================

def extract_conditions(
    name,
    text
):

    if (
        name
        in {
            "Togg Taşıt Finansmanı Kampanyası",
            "Umre Finansmanı Kampanyası",
        }
    ):

        # Redirect product page conditions are intentionally
        # not copied into the campaign record.
        return []


    result = []


    markers = [
        "kampanya",
        "yararlan",
        "faydalan",
        "geçerli",
        "katılım",
        "harcama",
        "talimat",
        "kart",
        "işlem",
        "başvuru",
        "en fazla",
        "en az",
        "maksimum",
        "minimum",
        "dahil",
        "hariç",
        "kod",
        "ödül",
        "müşteri",
        "taksit",
    ]


    for unit in units(text):

        n = normalize(unit)


        if not any(
            marker in n
            for marker in markers
        ):

            continue


        if (
            len(unit)
            < 20
            or
            len(unit)
            > 700
        ):

            continue


        # Sadece menü/başlık satırı olmasın.
        if n in {
            "kampanyaya katılım adımları",
            "kampanyadan kimler faydalanabilir?",
            "kampanyadan kimler faydalanabilir",
        }:

            continue


        result.append(unit)


    return unique(
        result
    )[:35]


# ============================================================
# CURRENCY
# ============================================================

def extract_currency(
    name,
    raw_text,
    advantages,
    finance_amounts,
):

    result = []


    semantic_text = "\n".join(
        advantages
        +
        finance_amounts
    )


    # Avantaj / finansman tutarı TL ise kesin TL.
    if re.search(
        r"\bTL\b|₺",
        semantic_text,
        flags=re.I
    ):

        result.append(
            "TL"
        )


    # Metinde TL ile gerçek kampanya eşikleri varsa.
    if (
        not result
        and
        re.search(
            r"\b\d[\d.,]*\s*TL\b",
            raw_text,
            flags=re.I
        )
    ):

        result.append(
            "TL"
        )


    # Döviz kampanyasında açık USD/EUR varsa eklenebilir.
    if (
        name
        ==
        "Hızlı Döviz İşlemleri Yanınızda!"
    ):

        if re.search(
            r"\bUSD\b",
            raw_text,
            flags=re.I
        ):

            result.append(
                "USD"
            )


        if re.search(
            r"\bEUR\b",
            raw_text,
            flags=re.I
        ):

            result.append(
                "EUR"
            )


    return unique(
        result
    )


# ============================================================
# LOAD
# ============================================================

with open(
    INPUT_FILE,
    "r",
    encoding="utf-8"
) as f:

    raw_data = json.load(f)


raw_records = raw_data.get(
    "kampanyalar",
    []
)


print("=" * 120)
print(
    "ALBARAKA TÜRK - "
    "CAMPAIGN EXTRACTOR FINAL V1"
)
print("=" * 120)

print(
    "RAW kampanya:",
    len(raw_records)
)


# ============================================================
# EXTRACT
# ============================================================

records = []
extract_errors = []


for index, raw in enumerate(
    raw_records,
    start=1
):

    try:

        name = clean_text(
            raw.get(
                "kampanya_adi",
                ""
            )
        )

        raw_text = str(
            raw.get(
                "ham_metin",
                ""
            )
        )

        source_url = clean_text(
            raw.get(
                "kaynak_url",
                ""
            )
        )


        if name not in CAMPAIGN_TYPES:

            raise RuntimeError(
                (
                    "CAMPAIGN_TYPES eksik: "
                    f"{name}"
                )
            )


        if name not in ADVANTAGES:

            raise RuntimeError(
                (
                    "ADVANTAGES eksik: "
                    f"{name}"
                )
            )


        # ----------------------------------------------------
        # RATES
        # ----------------------------------------------------

        kar_payi, finance_rates = (
            extract_finance_rates(
                name,
                raw_text
            )
        )


        # Dijital Katılma hesabı finansman campaign
        # allowlist'inde değil; explicit 98/2 burada eklenir.
        if (
            name
            ==
            "Dijital Katılma Hesabı'na Özel Paylaşım Oranları!"
        ):

            kar_payi = [
                "98/2"
            ]


        # ----------------------------------------------------
        # AMOUNT
        # ----------------------------------------------------

        finance_amounts = list(
            FINANCE_AMOUNT_OVERRIDES.get(
                name,
                []
            )
        )


        # ----------------------------------------------------
        # TERM / INSTALLMENT
        # ----------------------------------------------------

        terms = extract_terms(
            name,
            raw_text
        )

        installments = (
            extract_installments(
                raw_text
            )
        )


        # ----------------------------------------------------
        # ADVANTAGE
        # ----------------------------------------------------

        advantages = list(
            ADVANTAGES[
                name
            ]
        )


        # ----------------------------------------------------
        # PERIOD
        # ----------------------------------------------------

        campaign_period = (
            extract_period(
                name,
                raw
            )
        )


        # ----------------------------------------------------
        # TARGET / CONDITION / FEE
        # ----------------------------------------------------

        targets = extract_targets(
            name,
            raw_text
        )

        conditions = (
            extract_conditions(
                name,
                raw_text
            )
        )

        fees = extract_fees(
            name,
            raw_text
        )


        # ----------------------------------------------------
        # CURRENCY
        # ----------------------------------------------------

        currencies = (
            extract_currency(
                name,
                raw_text,
                advantages,
                finance_amounts,
            )
        )


        record = {

            "banka":
                BANK_NAME,

            "kayit_turu":
                "kampanya",

            "urun_adi":
                name,

            "urun_kategorisi":
                extract_category(
                    raw
                ),

            "kar_payi_orani":
                kar_payi,

            "finansman_orani":
                finance_rates,

            "finansman_tutari":
                finance_amounts,

            "vade":
                terms,

            "taksit_sayisi":
                installments,

            "masraf_bilgisi":
                fees,

            "kampanya_turu":
                CAMPAIGN_TYPES[
                    name
                ],

            "kampanya_avantaji":
                advantages,

            "kampanya_suresi":
                campaign_period,

            "hedef_kitle":
                targets,

            "para_birimi":
                currencies,

            "kosullar":
                conditions,

            "kaynak_url":
                source_url,

            "ham_metin":
                raw_text,
        }


        records.append(
            record
        )


    except Exception as error:

        extract_errors.append(
            (
                f"[{index}] "
                f"{type(error).__name__}: "
                f"{error}"
            )
        )


# ============================================================
# VALIDATION
# ============================================================

errors = list(
    extract_errors
)


# ------------------------------------------------------------
# COUNT
# ------------------------------------------------------------

if len(records) != 48:

    errors.append(
        (
            f"Extracted {len(records)} "
            "!= 48"
        )
    )


# ------------------------------------------------------------
# SCHEMA / TYPES
# ------------------------------------------------------------

for index, record in enumerate(
    records,
    start=1
):

    name = record[
        "urun_adi"
    ]


    if list(
        record.keys()
    ) != SCHEMA_KEYS:

        errors.append(
            (
                f"{name} -> "
                "schema/order hatası"
            )
        )


    for field in LIST_FIELDS:

        if not isinstance(
            record[field],
            list
        ):

            errors.append(
                (
                    f"{name} -> "
                    f"{field} list değil"
                )
            )


    for field in SCALAR_FIELDS:

        if not isinstance(
            record[field],
            str
        ):

            errors.append(
                (
                    f"{name} -> "
                    f"{field} string değil"
                )
            )


    if (
        record[
            "banka"
        ]
        != BANK_NAME
    ):

        errors.append(
            (
                f"{name} -> "
                "banka yanlış"
            )
        )


    if (
        record[
            "kayit_turu"
        ]
        != "kampanya"
    ):

        errors.append(
            (
                f"{name} -> "
                "kayit_turu yanlış"
            )
        )


    for field in [
        "urun_adi",
        "urun_kategorisi",
        "kampanya_turu",
        "kaynak_url",
        "ham_metin",
    ]:

        if not record[
            field
        ]:

            errors.append(
                (
                    f"{name} -> "
                    f"{field} boş"
                )
            )


    if (
        "TRY"
        in record[
            "para_birimi"
        ]
    ):

        errors.append(
            (
                f"{name} -> "
                "TRY yerine TL kullanılmalı"
            )
        )


# ============================================================
# DUPLICATES
# ============================================================

names = [
    r[
        "urun_adi"
    ]
    for r in records
]

urls = [
    r[
        "kaynak_url"
    ]
    for r in records
]


duplicate_name = (
    len(names)
    -
    len(set(names))
)

duplicate_url = (
    len(urls)
    -
    len(set(urls))
)


if duplicate_name:

    errors.append(
        (
            "Duplicate title: "
            f"{duplicate_name}"
        )
    )


if duplicate_url:

    errors.append(
        (
            "Duplicate URL: "
            f"{duplicate_url}"
        )
    )


# ============================================================
# RATE LEAKAGE
#
# İndirim yüzdeleri financing ratio olamaz.
# ============================================================

RATE_ALLOWED = (
    FINANCE_CAMPAIGNS
    |
    {
        "Dijital Katılma Hesabı'na Özel Paylaşım Oranları!",
    }
)


for record in records:

    name = record[
        "urun_adi"
    ]


    if (
        name
        not in RATE_ALLOWED
        and
        (
            record[
                "kar_payi_orani"
            ]
            or
            record[
                "finansman_orani"
            ]
        )
    ):

        errors.append(
            (
                f"{name} -> "
                "indirim/başka yüzde "
                "finansman oranına sızmış"
            )
        )


# ============================================================
# REDIRECT VALIDATION
# ============================================================

for name in REDIRECT_CAMPAIGNS:

    found = next(
        (
            r
            for r in records
            if (
                r[
                    "urun_adi"
                ]
                == name
            )
        ),
        None
    )


    if found is None:

        errors.append(
            (
                f"Redirect kayıt "
                f"eksik: {name}"
            )
        )

        continue


    if found[
        "kampanya_suresi"
    ]:

        errors.append(
            (
                f"{name} -> "
                "kampanya_suresi boş olmalı"
            )
        )


# Togg redirect ürün sayfasındaki
# %100 elektrik / %70 ürün finansmanı
# kampanya oranı olarak alınmamalı.
togg = next(
    r
    for r in records
    if (
        r["urun_adi"]
        ==
        "Togg Taşıt Finansmanı Kampanyası"
    )
)

if (
    togg["kar_payi_orani"]
    or
    togg["finansman_orani"]
):

    errors.append(
        (
            "Togg redirect -> "
            "ürün oranları kampanyaya sızmış"
        )
    )


# ============================================================
# PERIOD VALIDATION
# ============================================================

empty_periods = [
    r[
        "urun_adi"
    ]
    for r in records
    if not r[
        "kampanya_suresi"
    ]
]


EXPECTED_EMPTY_PERIODS = {
    "Togg Taşıt Finansmanı Kampanyası",
    "Umre Finansmanı Kampanyası",
    "Ücretsiz Ortak ATM Kampanyası",
}


if (
    set(
        empty_periods
    )
    !=
    EXPECTED_EMPTY_PERIODS
):

    errors.append(
        (
            "Boş kampanya süresi seti "
            "beklenenden farklı: "
            f"{empty_periods}"
        )
    )


# ============================================================
# ACTIVE DATE CHECK
# ============================================================

expired = []


for record in records:

    period = record[
        "kampanya_suresi"
    ]


    if not period:
        continue


    match = re.match(
        (
            r"(\d{2}\.\d{2}\.\d{4})"
            r"\s*-\s*"
            r"(\d{2}\.\d{2}\.\d{4})"
        ),
        period
    )


    if not match:
        errors.append(
            (
                f"{record['urun_adi']} -> "
                f"süre formatı yanlış: {period}"
            )
        )

        continue


    end_date = datetime.strptime(
        match.group(2),
        "%d.%m.%Y"
    ).date()


    if end_date < TODAY:

        expired.append(
            record[
                "urun_adi"
            ]
        )


if expired:

    errors.append(
        (
            "Aktif listede süresi "
            "geçmiş kampanya var: "
            f"{expired}"
        )
    )


# ============================================================
# CRITICAL SEMANTIC CHECKS
# ============================================================

by_name = {
    r[
        "urun_adi"
    ]:
    r
    for r in records
}


def require(
    name,
    field,
    expected,
):

    actual = by_name[
        name
    ][
        field
    ]


    if actual != expected:

        errors.append(
            (
                f"{name} -> "
                f"{field}: "
                f"{actual} "
                f"!= {expected}"
            )
        )


# %0 explicit financing benefit
require(
    "Vade Farksız 140.000 TL'ye Varan Destek!",
    "finansman_orani",
    ["%0"]
)


require(
    "Vade Farksız 140.000 TL'ye Varan Destek!",
    "finansman_tutari",
    ["140.000 TL"]
)


require(
    "Dijital Katılma Hesabı'na Özel Paylaşım Oranları!",
    "kar_payi_orani",
    ["98/2"]
)


require(
    "Sağlık Harcamalarına Vade Farksız 6 Taksit Kampanyası",
    "taksit_sayisi",
    ["6"]
)


require(
    "Eğitim Harcamalarınıza Vade Farksız 6 Taksit Kampanyası",
    "taksit_sayisi",
    ["6"]
)


require(
    "Kırtasiye Kampanyası",
    "taksit_sayisi",
    ["4"]
)


require(
    "8 Taksit Fırsatıyla KASKO Zamanı!",
    "taksit_sayisi",
    ["8"]
)


require(
    "Limitsiz İMM Sigortasında Vade Farksız 3 Taksit!",
    "taksit_sayisi",
    ["3"]
)


require(
    "Satırdan Sanata, Ek %10 İndirim ve Vade Farksız 4 Taksit Albaraka'da!",
    "taksit_sayisi",
    ["4"]
)


# Discount % must NOT enter financing ratio
for name in [
    "Otopark ve Vale Harcamalarınıza %50 İade Albaraka'da!",
    "Hızlı Çiçek %20 İndirim Kampanyası",
    "Trend Kredi Kart Kırtasiye Harcamaları İndirim Kampanyası!",
    "Trend Kredi Kart ile Sinema Harcamalarına %25 İndirim!",
    "Enterprise Araç Kiralama Kampanyası",
    "Enterprise Araç Kiralama Kampanyası | %35 İndirim!",
    "Albaraka Müşterilerine Özel Marina Aquapark'ta %20 indirim!",
    "Restoran Harcamalarınıza Özel %10 İndirim Fırsatı!",
    "Otopark Harcamalarınıza Özel %10 İndirim Fırsatı!",
]:

    if (
        by_name[
            name
        ][
            "finansman_orani"
        ]
        or
        by_name[
            name
        ][
            "kar_payi_orani"
        ]
    ):

        errors.append(
            (
                f"{name} -> "
                "indirim yüzdesi finansman "
                "oranına yazılmış"
            )
        )


# ============================================================
# SAVE DIRECT LIST
# ============================================================

with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        records,
        f,
        ensure_ascii=False,
        indent=4
    )


# ============================================================
# AUDIT COUNTS
# ============================================================

type_counts = Counter(
    r[
        "kampanya_turu"
    ]
    for r in records
)


category_counts = Counter(
    r[
        "urun_kategorisi"
    ]
    for r in records
)


with_worldpuan = sum(
    (
        "Worldpuan"
        in r[
            "kampanya_turu"
        ]
    )
    for r in records
)


with_installment = sum(
    bool(
        r[
            "taksit_sayisi"
        ]
    )
    for r in records
)


with_period = sum(
    bool(
        r[
            "kampanya_suresi"
        ]
    )
    for r in records
)


# ============================================================
# FINAL PRINT
# ============================================================

print()
print("=" * 120)
print(
    "ALBARAKA TÜRK - "
    "CAMPAIGN FINAL AUDIT"
)
print("=" * 120)

print(
    "RAW kampanya       :",
    len(raw_records)
)

print(
    "Extracted          :",
    len(records)
)

print(
    "Süre bulunan       :",
    f"{with_period} / {len(records)}"
)

print(
    "Worldpuan türü     :",
    with_worldpuan
)

print(
    "Taksit bulunan     :",
    with_installment
)

print(
    "Duplicate URL      :",
    duplicate_url
)

print(
    "Duplicate title    :",
    duplicate_name
)

print(
    "Expired            :",
    len(expired)
)

print(
    "Validation error   :",
    len(errors)
)

print(
    "Final JSON         :",
    OUTPUT_FILE
)


# ============================================================
# TYPE DISTRIBUTION
# ============================================================

print()
print("=" * 120)
print("KAMPANYA TÜRÜ DAĞILIMI")
print("=" * 120)

for key, value in (
    type_counts
    .most_common()
):

    print(
        f"{key:<32}: "
        f"{value}"
    )


# ============================================================
# CATEGORY DISTRIBUTION
# ============================================================

print()
print("=" * 120)
print("KATEGORİ DAĞILIMI")
print("=" * 120)

for key, value in (
    category_counts
    .most_common()
):

    print(
        f"{key:<25}: "
        f"{value}"
    )


# ============================================================
# CRITICAL VALUES
# ============================================================

print()
print("=" * 120)
print("KRİTİK KAMPANYA DEĞERLERİ")
print("=" * 120)

for name in [
    "Vade Farksız 140.000 TL'ye Varan Destek!",
    "Dijital Katılma Hesabı'na Özel Paylaşım Oranları!",
    "Sağlık Harcamalarına Vade Farksız 6 Taksit Kampanyası",
    "Otopark ve Vale Harcamalarınıza %50 İade Albaraka'da!",
    "Taksitlio.com Alışveriş Finansmanı",
    "Dijitale Özel Konut ve Taşıt Finansmanı Kampanyası",
    "Satırdan Sanata, Ek %10 İndirim ve Vade Farksız 4 Taksit Albaraka'da!",
    "Dijital Müşterilere Özel Pratik Finansman Kart",
    "Togg Taşıt Finansmanı Kampanyası",
    "Umre Finansmanı Kampanyası",
    "Emekli Promosyon 2026 | 30.000 TL'ye Varan Ödül!",
    "Ücretsiz Ortak ATM Kampanyası",
]:

    r = by_name[
        name
    ]

    print()
    print(name)

    print(
        "  Tür       :",
        r[
            "kampanya_turu"
        ]
    )

    print(
        "  Kâr Payı  :",
        r[
            "kar_payi_orani"
        ]
    )

    print(
        "  Fin.Oran  :",
        r[
            "finansman_orani"
        ]
    )

    print(
        "  Fin.Tutar :",
        r[
            "finansman_tutari"
        ]
    )

    print(
        "  Taksit    :",
        r[
            "taksit_sayisi"
        ]
    )

    print(
        "  Süre      :",
        r[
            "kampanya_suresi"
        ]
    )

    print(
        "  Avantaj   :",
        r[
            "kampanya_avantaji"
        ]
    )


# ============================================================
# ERRORS
# ============================================================

if errors:

    print()
    print("=" * 120)
    print("HATALAR")
    print("=" * 120)

    for error in errors:

        print(
            "-",
            error
        )


print()
print("=" * 120)

if not errors:

    print(
        "SONUÇ: ALBARAKA TÜRK "
        "KAMPANYA EXTRACTOR "
        "48/48 TAMAMEN BAŞARILI ✅"
    )

else:

    print(
        "SONUÇ: ALBARAKA TÜRK "
        "KAMPANYA EXTRACTOR "
        "KONTROL GEREKİYOR ❌"
    )

print("=" * 120)


files.download(
    OUTPUT_FILE
)

ALBARAKA TÜRK - CAMPAIGN EXTRACTOR FINAL V1
RAW kampanya: 48

ALBARAKA TÜRK - CAMPAIGN FINAL AUDIT
RAW kampanya       : 48
Extracted          : 48
Süre bulunan       : 45 / 48
Worldpuan türü     : 12
Taksit bulunan     : 9
Duplicate URL      : 0
Duplicate title    : 0
Expired            : 0
Validation error   : 0
Final JSON         : /content/albaraka_turk_kampanya_extracted.json

KAMPANYA TÜRÜ DAĞILIMI
Worldpuan                       : 11
İndirim                         : 9
Taksit                          : 5
Finansman Avantajı              : 5
Ücretsiz Hizmet                 : 4
İade                            : 4
Finansman Avantajı + Taksit     : 2
Hediye                          : 2
Paylaşım Oranı                  : 1
Worldpuan + Taksit              : 1
Sadakat Programı                : 1
İndirim + Taksit                : 1
Kur Avantajı                    : 1
Promosyon                       : 1

KATEGORİ DAĞILIMI
Bireysel                 : 34
Dijital                  : 5
Özel Banka

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# ALBARAKA TÜRK
# FINAL SEMANTIC CONTEXT CHECKER
# ============================================================

import json
import re


INPUT_FILE = (
    "/content/"
    "albaraka_turk_kampanyalar_raw.json"
)


TARGETS = [
    "Vade Farksız 140.000 TL'ye Varan Destek!",
    "Taksitlio.com Alışveriş Finansmanı",
    "Dijitale Özel Konut ve Taşıt Finansmanı Kampanyası",
    "Dijital Müşterilere Özel Pratik Finansman Kart",
]


KEYWORDS = [
    "%",
    "kâr pay",
    "kar pay",
    "kâr oran",
    "kar oran",
    "finansman oran",
    "aylık oran",
    "vade fark",
    "vade farksız",
    "taksit",
    "tahsis",
    "ücret",
    "masraf",
    "komisyon",
    "150.000",
    "140.000",
    "40.000",
    "2,87",
    "3,19",
    "2,99",
    "0,5",
]


def clean_text(value):

    value = str(value or "")

    value = (
        value
        .replace("\xa0", " ")
        .replace("’", "'")
        .replace("‘", "'")
        .replace("–", "-")
        .replace("—", "-")
    )

    value = re.sub(
        r"[ \t]+",
        " ",
        value
    )

    return value.strip()


def normalize(value):

    return (
        clean_text(value)
        .replace("İ", "i")
        .replace("I", "ı")
        .casefold()
    )


def split_units(text):

    text = clean_text(text)

    parts = re.split(
        r"\n+|(?<=[.!?;])\s+",
        text
    )

    return [
        clean_text(x)
        for x in parts
        if clean_text(x)
    ]


with open(
    INPUT_FILE,
    "r",
    encoding="utf-8"
) as f:

    data = json.load(f)


records = data[
    "kampanyalar"
]


by_name = {
    r["kampanya_adi"]:
    r
    for r in records
}


print("=" * 125)
print(
    "ALBARAKA TÜRK - "
    "FINAL SEMANTIC CONTEXT CHECKER"
)
print("=" * 125)


for target in TARGETS:

    record = by_name[
        target
    ]

    text = record[
        "ham_metin"
    ]

    units = split_units(
        text
    )


    print()
    print("#" * 125)
    print(target)
    print("#" * 125)


    hits = []


    for index, unit in enumerate(
        units
    ):

        n = normalize(
            unit
        )


        if not any(
            normalize(keyword)
            in n
            for keyword in KEYWORDS
        ):

            continue


        # Önceki + mevcut + sonraki unit
        start = max(
            0,
            index - 1
        )

        end = min(
            len(units),
            index + 2
        )


        context = " || ".join(
            units[start:end]
        )


        if context not in hits:

            hits.append(
                context
            )


    for i, hit in enumerate(
        hits,
        start=1
    ):

        print()
        print(
            f"[{i:02d}]"
        )

        print(
            hit
        )


    print()
    print(
        "TOPLAM CONTEXT:",
        len(hits)
    )


print()
print("=" * 125)
print(
    "CONTEXT CHECK TAMAMLANDI ✅"
)
print("=" * 125)

ALBARAKA TÜRK - FINAL SEMANTIC CONTEXT CHECKER

#############################################################################################################################
Vade Farksız 140.000 TL'ye Varan Destek!
#############################################################################################################################

[01]
Vade Farksız 140.000 TL'ye Varan Destek! || Anasayfa

[02]
Müşteri Ol || Kâr payı yok. || Beklemek yok.

[03]
Beklemek yok. || 140.000 TL || 'ye kadar

[04]
'ye kadar || vade farksız destek || Albaraka'da!

[05]
Şimdi Albaraka Mobil'den müşteri olanlar, || %0 kâr payı ile 40.000 TL'ye kadar || Pratik Finansman Kart (İhtiyaç Finansmanı) kullanabiliyor ve

[06]
100.000 TL'ye varan || seçili sektörlerde vade farksız taksitli alışveriş fırsatından yararlanabiliyor. || Hemen siz de

[07]
Albaraka Türk müşterisi olun; || %0 || kâr paylı

[08]
%0 || kâr paylı || 40.000 TL

[09]
kâr paylı || 40.000 TL || 'ye kadar,

[10]
'ye kadar, || 3 aya varan ödemes

In [ ]:
# ============================================================
# ALBARAKA TÜRK
# CAMPAIGN SEMANTIC FINALIZER V2
#
# Input / Output:
#   /content/albaraka_turk_kampanya_extracted.json
#
# 4 kayıt semantik olarak düzeltilir.
# ============================================================

import json

from google.colab import files


FILE = (
    "/content/"
    "albaraka_turk_kampanya_extracted.json"
)


with open(
    FILE,
    "r",
    encoding="utf-8"
) as f:
    records = json.load(f)


by_name = {
    x["urun_adi"]: x
    for x in records
}


# ============================================================
# 1) VADE FARKSIZ 140.000 TL
# ============================================================

name = (
    "Vade Farksız 140.000 TL'ye "
    "Varan Destek!"
)

r = by_name[name]


# %0 açıkça KÂR PAYI olarak geçiyor.
r["kar_payi_orani"] = [
    "%0"
]

# %0 finansman oranı olarak ayrıca türetilmez.
r["finansman_orani"] = []


# Toplam kampanya desteği.
r["finansman_tutari"] = [
    "140.000 TL"
]


# Pratik Finansman Kart bileşeni
# 6 aya kadar vadeli.
r["vade"] = [
    "6 aya kadar"
]


# Kampanya iki farklı avantaj içeriyor:
# - PFK: 4 taksit
# - kredi kartı: 6 taksit
r["taksit_sayisi"] = [
    "4",
    "6"
]


r["kampanya_avantaji"] = [
    (
        "40.000 TL'ye kadar %0 kâr paylı "
        "Pratik Finansman Kart"
    ),
    (
        "3 aya varan ödemesiz dönem ve "
        "4 taksitli Pratik Finansman Kart"
    ),
    (
        "100.000 TL'ye kadar seçili "
        "sektörlerde vade farksız "
        "6 taksit"
    ),
    (
        "Toplamda 140.000 TL'ye kadar "
        "vade farksız finansman desteği"
    ),
]


# ============================================================
# 2) TAKSİTLİO
# ============================================================

name = (
    "Taksitlio.com "
    "Alışveriş Finansmanı"
)

r = by_name[name]


r["kar_payi_orani"] = [
    "%2,99"
]

r["finansman_orani"] = []

r["finansman_tutari"] = [
    "150.000 TL"
]


# %0,5 KÂR PAYI DEĞİL.
# Tahsis ücreti.
r["masraf_bilgisi"] = [
    (
        "Finansman tahsis ücreti, "
        "toplam finansman tutarının "
        "%0,5'i oranında tahsil edilir."
    )
]


# ============================================================
# 3) DİJİTALE ÖZEL KONUT + TAŞIT
# ============================================================

name = (
    "Dijitale Özel Konut ve Taşıt "
    "Finansmanı Kampanyası"
)

r = by_name[name]


# Metin açıkça "kâr oranı" diyor.
r["kar_payi_orani"] = [
    "%2,87",
    "%3,19",
]

r["finansman_orani"] = []


r["kampanya_avantaji"] = [
    (
        "Konut finansmanında "
        "%2,87'den başlayan kâr oranı"
    ),
    (
        "Taşıt finansmanında "
        "%3,19'dan başlayan kâr oranı"
    ),
]


# ============================================================
# 4) DİJİTAL PRATİK FİNANSMAN KART
# ============================================================

name = (
    "Dijital Müşterilere Özel "
    "Pratik Finansman Kart"
)

r = by_name[name]


# Açık aylık kâr oranı tablosu.
r["kar_payi_orani"] = [
    "%0",
    "%3,95",
    "%3,90",
    "%3,85",
]

r["finansman_orani"] = []


# 40.000 TL = %0 özel tranche
# 150.000 TL = azami finansman.
r["finansman_tutari"] = [
    "40.000 TL",
    "150.000 TL",
]


# Oran tablosundaki finansman vadeleri.
r["vade"] = [
    "1-6 ay",
    "7-12 ay",
    "13-36 ay",
]


# 12/3/6 değerleri cep telefonu/bilgisayar/tablet
# sektörel alışveriş taksit limitleri.
# Kampanyanın esas özel taksiti 4.
r["taksit_sayisi"] = [
    "4"
]


r["kampanya_avantaji"] = [
    (
        "40.000 TL'ye kadar %0 kâr paylı "
        "Pratik Finansman Kart"
    ),
    (
        "3 aya varan ödemesiz dönem ve "
        "4 taksit imkânı"
    ),
    (
        "150.000 TL'ye kadar özel oranlı "
        "finansman"
    ),
    (
        "36 aya varan vade imkânı"
    ),
]


# Oran-vade eşleşmesini kaybetmemek için
# koşullara deterministik şekilde ekliyoruz.
semantic_conditions = [
    (
        "250 TL-40.000 TL finansmanda "
        "1-6 ay vade için kâr oranı %0'dır."
    ),
    (
        "40.001 TL-150.000 TL finansmanda "
        "1-6 ay vade için kâr oranı %3,95'tir."
    ),
    (
        "250 TL-150.000 TL finansmanda "
        "7-12 ay vade için kâr oranı %3,90'dır."
    ),
    (
        "250 TL-150.000 TL finansmanda "
        "13-36 ay vade için kâr oranı %3,85'tir."
    ),
    (
        "40.000 TL'ye kadar başvurularda "
        "3 aya varan ödemesiz dönem ve "
        "4 taksit imkânı vardır."
    ),
    (
        "Pratik Finansman Kart ile "
        "asgari 250 TL, azami 150.000 TL "
        "finansman kullanılabilir."
    ),
]


old_conditions = r[
    "kosullar"
]


for condition in semantic_conditions:

    if condition not in old_conditions:
        old_conditions.append(
            condition
        )


# ============================================================
# VALIDATION
# ============================================================

errors = []


if len(records) != 48:
    errors.append(
        f"Kayıt {len(records)} != 48"
    )


def require(
    name,
    field,
    expected
):

    actual = (
        by_name[name][field]
    )

    if actual != expected:

        errors.append(
            (
                f"{name} -> "
                f"{field}: "
                f"{actual} "
                f"!= {expected}"
            )
        )


# ------------------------------------------------------------
# 140K
# ------------------------------------------------------------

require(
    (
        "Vade Farksız 140.000 TL'ye "
        "Varan Destek!"
    ),
    "kar_payi_orani",
    ["%0"]
)

require(
    (
        "Vade Farksız 140.000 TL'ye "
        "Varan Destek!"
    ),
    "finansman_orani",
    []
)

require(
    (
        "Vade Farksız 140.000 TL'ye "
        "Varan Destek!"
    ),
    "taksit_sayisi",
    ["4", "6"]
)


# ------------------------------------------------------------
# TAKSİTLİO
# ------------------------------------------------------------

require(
    "Taksitlio.com Alışveriş Finansmanı",
    "kar_payi_orani",
    ["%2,99"]
)

require(
    "Taksitlio.com Alışveriş Finansmanı",
    "finansman_orani",
    []
)

require(
    "Taksitlio.com Alışveriş Finansmanı",
    "finansman_tutari",
    ["150.000 TL"]
)


# ------------------------------------------------------------
# DİJİTAL KONUT / TAŞIT
# ------------------------------------------------------------

require(
    (
        "Dijitale Özel Konut ve Taşıt "
        "Finansmanı Kampanyası"
    ),
    "kar_payi_orani",
    [
        "%2,87",
        "%3,19",
    ]
)

require(
    (
        "Dijitale Özel Konut ve Taşıt "
        "Finansmanı Kampanyası"
    ),
    "finansman_orani",
    []
)


# ------------------------------------------------------------
# PRATİK FİNANSMAN KART
# ------------------------------------------------------------

require(
    (
        "Dijital Müşterilere Özel "
        "Pratik Finansman Kart"
    ),
    "kar_payi_orani",
    [
        "%0",
        "%3,95",
        "%3,90",
        "%3,85",
    ]
)

require(
    (
        "Dijital Müşterilere Özel "
        "Pratik Finansman Kart"
    ),
    "finansman_orani",
    []
)

require(
    (
        "Dijital Müşterilere Özel "
        "Pratik Finansman Kart"
    ),
    "finansman_tutari",
    [
        "40.000 TL",
        "150.000 TL",
    ]
)

require(
    (
        "Dijital Müşterilere Özel "
        "Pratik Finansman Kart"
    ),
    "vade",
    [
        "1-6 ay",
        "7-12 ay",
        "13-36 ay",
    ]
)

require(
    (
        "Dijital Müşterilere Özel "
        "Pratik Finansman Kart"
    ),
    "taksit_sayisi",
    ["4"]
)


# ============================================================
# GENERAL LEAKAGE CHECK
# ============================================================

for record in records:

    name = record[
        "urun_adi"
    ]

    # Finansman oranı alanını kampanya datasetinde
    # yalnızca açık "finansman oranı" ifadesi varsa
    # kullanıyoruz.
    #
    # Bu dört kritik kayıtta hiçbirisi öyle değil.
    if name in {
        (
            "Vade Farksız 140.000 TL'ye "
            "Varan Destek!"
        ),
        "Taksitlio.com Alışveriş Finansmanı",
        (
            "Dijitale Özel Konut ve Taşıt "
            "Finansmanı Kampanyası"
        ),
        (
            "Dijital Müşterilere Özel "
            "Pratik Finansman Kart"
        ),
    }:

        if record[
            "finansman_orani"
        ]:

            errors.append(
                (
                    f"{name} -> "
                    "finansman_orani "
                    "boş olmalı"
                )
            )


# ============================================================
# DUPLICATES
# ============================================================

names = [
    x["urun_adi"]
    for x in records
]

urls = [
    x["kaynak_url"]
    for x in records
]


duplicate_name = (
    len(names)
    -
    len(set(names))
)

duplicate_url = (
    len(urls)
    -
    len(set(urls))
)


if duplicate_name:
    errors.append(
        (
            "Duplicate title: "
            f"{duplicate_name}"
        )
    )


if duplicate_url:
    errors.append(
        (
            "Duplicate URL: "
            f"{duplicate_url}"
        )
    )


# ============================================================
# SAVE
# ============================================================

with open(
    FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        records,
        f,
        ensure_ascii=False,
        indent=4
    )


# ============================================================
# PRINT
# ============================================================

print("=" * 120)
print(
    "ALBARAKA TÜRK - "
    "CAMPAIGN SEMANTIC FINALIZER V2"
)
print("=" * 120)

print(
    "Kayıt             :",
    len(records)
)

print(
    "Duplicate URL     :",
    duplicate_url
)

print(
    "Duplicate title   :",
    duplicate_name
)

print(
    "Validation error  :",
    len(errors)
)

print(
    "Final JSON        :",
    FILE
)


print()
print("=" * 120)
print("DÜZELTİLEN 4 KAYIT")
print("=" * 120)


for name in [
    (
        "Vade Farksız 140.000 TL'ye "
        "Varan Destek!"
    ),
    "Taksitlio.com Alışveriş Finansmanı",
    (
        "Dijitale Özel Konut ve Taşıt "
        "Finansmanı Kampanyası"
    ),
    (
        "Dijital Müşterilere Özel "
        "Pratik Finansman Kart"
    ),
]:

    r = by_name[
        name
    ]

    print()
    print(name)

    print(
        "  Kâr Payı :",
        r[
            "kar_payi_orani"
        ]
    )

    print(
        "  Fin.Oran :",
        r[
            "finansman_orani"
        ]
    )

    print(
        "  Fin.Tutar:",
        r[
            "finansman_tutari"
        ]
    )

    print(
        "  Vade     :",
        r[
            "vade"
        ]
    )

    print(
        "  Taksit   :",
        r[
            "taksit_sayisi"
        ]
    )

    print(
        "  Masraf   :",
        r[
            "masraf_bilgisi"
        ]
    )

    print(
        "  Avantaj  :",
        r[
            "kampanya_avantaji"
        ]
    )


if errors:

    print()
    print("=" * 120)
    print("HATALAR")
    print("=" * 120)

    for error in errors:

        print(
            "-",
            error
        )


print()
print("=" * 120)

if not errors:

    print(
        "SONUÇ: ALBARAKA TÜRK "
        "KAMPANYA SEMANTİK AUDIT "
        "48/48 BAŞARILI ✅"
    )

else:

    print(
        "SONUÇ: KONTROL GEREKİYOR ❌"
    )

print("=" * 120)


files.download(
    FILE
)

ALBARAKA TÜRK - CAMPAIGN SEMANTIC FINALIZER V2
Kayıt             : 48
Duplicate URL     : 0
Duplicate title   : 0
Validation error  : 0
Final JSON        : /content/albaraka_turk_kampanya_extracted.json

DÜZELTİLEN 4 KAYIT

Vade Farksız 140.000 TL'ye Varan Destek!
  Kâr Payı : ['%0']
  Fin.Oran : []
  Fin.Tutar: ['140.000 TL']
  Vade     : ['6 aya kadar']
  Taksit   : ['4', '6']
  Masraf   : ['Masrafsız Banka ve Kredi Kartı']
  Avantaj  : ["40.000 TL'ye kadar %0 kâr paylı Pratik Finansman Kart", '3 aya varan ödemesiz dönem ve 4 taksitli Pratik Finansman Kart', "100.000 TL'ye kadar seçili sektörlerde vade farksız 6 taksit", "Toplamda 140.000 TL'ye kadar vade farksız finansman desteği"]

Taksitlio.com Alışveriş Finansmanı
  Kâr Payı : ['%2,99']
  Fin.Oran : []
  Fin.Tutar: ['150.000 TL']
  Vade     : ['6 ay\nvade']
  Taksit   : []
  Masraf   : ["Finansman tahsis ücreti, toplam finansman tutarının %0,5'i oranında tahsil edilir."]
  Avantaj  : ['Taksitlio.com anlaşmalı mağazalarda alışveri

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# ALBARAKA TÜRK
# FINAL CLEANUP + MERGER V1
#
# Inputs:
#   /content/albaraka_turk_finansman_extracted.json
#   /content/albaraka_turk_kampanya_extracted.json
#
# Output:
#   /content/albaraka_turk_final.json
#
# Expected:
#   17 finansman
#   48 kampanya
#   65 toplam
# ============================================================

import json
from collections import Counter

from google.colab import files


# ============================================================
# FILES
# ============================================================

FINANCE_FILE = (
    "/content/"
    "albaraka_turk_finansman_extracted.json"
)

CAMPAIGN_FILE = (
    "/content/"
    "albaraka_turk_kampanya_extracted.json"
)

OUTPUT_FILE = (
    "/content/"
    "albaraka_turk_final.json"
)


BANK_NAME = (
    "Albaraka Türk Katılım Bankası A.Ş."
)


# ============================================================
# SCHEMA
# ============================================================

SCHEMA_KEYS = [
    "banka",
    "kayit_turu",
    "urun_adi",
    "urun_kategorisi",
    "kar_payi_orani",
    "finansman_orani",
    "finansman_tutari",
    "vade",
    "taksit_sayisi",
    "masraf_bilgisi",
    "kampanya_turu",
    "kampanya_avantaji",
    "kampanya_suresi",
    "hedef_kitle",
    "para_birimi",
    "kosullar",
    "kaynak_url",
    "ham_metin",
]


LIST_FIELDS = {
    "kar_payi_orani",
    "finansman_orani",
    "finansman_tutari",
    "vade",
    "taksit_sayisi",
    "masraf_bilgisi",
    "kampanya_avantaji",
    "hedef_kitle",
    "para_birimi",
    "kosullar",
}


SCALAR_FIELDS = (
    set(SCHEMA_KEYS)
    -
    LIST_FIELDS
)


# ============================================================
# LOAD
# ============================================================

with open(
    FINANCE_FILE,
    "r",
    encoding="utf-8"
) as f:
    finance_records = json.load(f)


with open(
    CAMPAIGN_FILE,
    "r",
    encoding="utf-8"
) as f:
    campaign_records = json.load(f)


print("=" * 120)
print(
    "ALBARAKA TÜRK - "
    "FINAL CLEANUP + MERGER V1"
)
print("=" * 120)

print(
    "Finansman input:",
    len(finance_records)
)

print(
    "Kampanya input :",
    len(campaign_records)
)


# ============================================================
# CAMPAIGN INDEX
# ============================================================

campaign_by_name = {
    item["urun_adi"]: item
    for item in campaign_records
}


# ============================================================
# FINAL CLEANUP 1
#
# Vade Farksız 140.000 TL kampanyasında
# "Masrafsız Banka ve Kredi Kartı"
# generic page-footer / cross-promo bilgisidir.
# ============================================================

name = (
    "Vade Farksız 140.000 TL'ye "
    "Varan Destek!"
)

record = campaign_by_name[
    name
]

print()
print(
    name
)

print(
    "  Eski masraf:",
    record[
        "masraf_bilgisi"
    ]
)

record[
    "masraf_bilgisi"
] = []

print(
    "  Yeni masraf:",
    record[
        "masraf_bilgisi"
    ]
)


# ============================================================
# FINAL CLEANUP 2
#
# Taksitlio:
# Context açık bir sayısal vade değeri
# doğrulamıyor.
# ============================================================

name = (
    "Taksitlio.com "
    "Alışveriş Finansmanı"
)

record = campaign_by_name[
    name
]

print()
print(
    name
)

print(
    "  Eski vade:",
    record[
        "vade"
    ]
)

record[
    "vade"
] = []

print(
    "  Yeni vade:",
    record[
        "vade"
    ]
)


# ============================================================
# SAVE CLEAN CAMPAIGN FILE
# ============================================================

with open(
    CAMPAIGN_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        campaign_records,
        f,
        ensure_ascii=False,
        indent=4
    )


# ============================================================
# MERGE
# ============================================================

final_records = (
    finance_records
    +
    campaign_records
)


# ============================================================
# VALIDATION
# ============================================================

errors = []


# ------------------------------------------------------------
# COUNTS
# ------------------------------------------------------------

if len(
    finance_records
) != 17:

    errors.append(
        (
            "Finansman kayıt sayısı "
            f"{len(finance_records)} != 17"
        )
    )


if len(
    campaign_records
) != 48:

    errors.append(
        (
            "Kampanya kayıt sayısı "
            f"{len(campaign_records)} != 48"
        )
    )


if len(
    final_records
) != 65:

    errors.append(
        (
            "Final kayıt sayısı "
            f"{len(final_records)} != 65"
        )
    )


# ============================================================
# SCHEMA + TYPES
# ============================================================

for index, record in enumerate(
    final_records,
    start=1
):

    name = record.get(
        "urun_adi",
        f"#{index}"
    )


    # Exact schema order
    if list(
        record.keys()
    ) != SCHEMA_KEYS:

        errors.append(
            (
                f"{name} -> "
                "schema/order hatası"
            )
        )


    # List types
    for field in LIST_FIELDS:

        if not isinstance(
            record.get(field),
            list
        ):

            errors.append(
                (
                    f"{name} -> "
                    f"{field} list değil"
                )
            )


    # Scalar types
    for field in SCALAR_FIELDS:

        if not isinstance(
            record.get(field),
            str
        ):

            errors.append(
                (
                    f"{name} -> "
                    f"{field} string değil"
                )
            )


    # Bank
    if (
        record.get(
            "banka"
        )
        != BANK_NAME
    ):

        errors.append(
            (
                f"{name} -> "
                "banka yanlış"
            )
        )


    # Record type
    if record.get(
        "kayit_turu"
    ) not in {
        "finansman",
        "kampanya",
    }:

        errors.append(
            (
                f"{name} -> "
                "kayit_turu geçersiz"
            )
        )


    # Required
    for field in [
        "urun_adi",
        "kaynak_url",
        "ham_metin",
    ]:

        if not record.get(
            field
        ):

            errors.append(
                (
                    f"{name} -> "
                    f"{field} boş"
                )
            )


    # Currency standard
    if (
        "TRY"
        in record.get(
            "para_birimi",
            []
        )
    ):

        errors.append(
            (
                f"{name} -> "
                "TRY bulundu"
            )
        )


# ============================================================
# TYPE COUNTS
# ============================================================

finance_count = sum(
    record[
        "kayit_turu"
    ]
    == "finansman"
    for record in final_records
)


campaign_count = sum(
    record[
        "kayit_turu"
    ]
    == "kampanya"
    for record in final_records
)


if finance_count != 17:

    errors.append(
        (
            "Final finansman count "
            f"{finance_count} != 17"
        )
    )


if campaign_count != 48:

    errors.append(
        (
            "Final kampanya count "
            f"{campaign_count} != 48"
        )
    )


# ============================================================
# DUPLICATES
# ============================================================

urls = [
    record[
        "kaynak_url"
    ]
    for record in final_records
]


duplicate_url = (
    len(urls)
    -
    len(set(urls))
)


# Aynı ürün adı finansman ve kampanya tarafında
# bilinçli olarak geçebilir:
# örn. Togg benzeri.
#
# Bu nedenle title duplicate global error değil.
#
# Aynı kayit_turu + urun_adi duplicate kontrol edilir.

name_type_keys = [
    (
        record[
            "kayit_turu"
        ],
        record[
            "urun_adi"
        ],
    )
    for record in final_records
]


duplicate_name_type = (
    len(name_type_keys)
    -
    len(set(name_type_keys))
)


if duplicate_url:

    errors.append(
        (
            "Duplicate URL: "
            f"{duplicate_url}"
        )
    )


if duplicate_name_type:

    errors.append(
        (
            "Duplicate "
            "(kayit_turu, urun_adi): "
            f"{duplicate_name_type}"
        )
    )


# ============================================================
# FINANCE/CAMPAIGN FIELD LEAKAGE
# ============================================================

for record in final_records:

    name = record[
        "urun_adi"
    ]


    if (
        record[
            "kayit_turu"
        ]
        ==
        "finansman"
    ):

        # Finansman records campaign-only fields empty.
        for field in [
            "kampanya_turu",
            "kampanya_suresi",
        ]:

            if record[
                field
            ]:

                errors.append(
                    (
                        f"{name} -> "
                        f"finansmanda "
                        f"{field} dolu"
                    )
                )


        for field in [
            "kampanya_avantaji",
        ]:

            if record[
                field
            ]:

                errors.append(
                    (
                        f"{name} -> "
                        f"finansmanda "
                        f"{field} dolu"
                    )
                )


# ============================================================
# CRITICAL CAMPAIGN SEMANTIC CHECKS
# ============================================================

campaign_by_name = {
    record[
        "urun_adi"
    ]:
    record
    for record
    in campaign_records
}


def require(
    name,
    field,
    expected
):

    actual = (
        campaign_by_name[
            name
        ][
            field
        ]
    )


    if actual != expected:

        errors.append(
            (
                f"{name} -> "
                f"{field}: "
                f"{actual} "
                f"!= {expected}"
            )
        )


# ------------------------------------------------------------
# 140K campaign
# ------------------------------------------------------------

require(
    (
        "Vade Farksız 140.000 TL'ye "
        "Varan Destek!"
    ),
    "kar_payi_orani",
    ["%0"]
)


require(
    (
        "Vade Farksız 140.000 TL'ye "
        "Varan Destek!"
    ),
    "finansman_orani",
    []
)


require(
    (
        "Vade Farksız 140.000 TL'ye "
        "Varan Destek!"
    ),
    "finansman_tutari",
    ["140.000 TL"]
)


require(
    (
        "Vade Farksız 140.000 TL'ye "
        "Varan Destek!"
    ),
    "masraf_bilgisi",
    []
)


# ------------------------------------------------------------
# Taksitlio
# ------------------------------------------------------------

require(
    "Taksitlio.com Alışveriş Finansmanı",
    "kar_payi_orani",
    ["%2,99"]
)


require(
    "Taksitlio.com Alışveriş Finansmanı",
    "finansman_orani",
    []
)


require(
    "Taksitlio.com Alışveriş Finansmanı",
    "finansman_tutari",
    ["150.000 TL"]
)


require(
    "Taksitlio.com Alışveriş Finansmanı",
    "vade",
    []
)


# %0.5 tahsis ücreti var
taksitlio_fee = (
    campaign_by_name[
        "Taksitlio.com Alışveriş Finansmanı"
    ][
        "masraf_bilgisi"
    ]
)


if not any(
    "%0,5"
    in value
    for value
    in taksitlio_fee
):

    errors.append(
        (
            "Taksitlio -> "
            "%0,5 tahsis ücreti eksik"
        )
    )


# ------------------------------------------------------------
# Digital housing / vehicle
# ------------------------------------------------------------

require(
    (
        "Dijitale Özel Konut ve Taşıt "
        "Finansmanı Kampanyası"
    ),
    "kar_payi_orani",
    [
        "%2,87",
        "%3,19",
    ]
)


require(
    (
        "Dijitale Özel Konut ve Taşıt "
        "Finansmanı Kampanyası"
    ),
    "finansman_orani",
    []
)


# ------------------------------------------------------------
# Digital PFK
# ------------------------------------------------------------

require(
    (
        "Dijital Müşterilere Özel "
        "Pratik Finansman Kart"
    ),
    "kar_payi_orani",
    [
        "%0",
        "%3,95",
        "%3,90",
        "%3,85",
    ]
)


require(
    (
        "Dijital Müşterilere Özel "
        "Pratik Finansman Kart"
    ),
    "finansman_tutari",
    [
        "40.000 TL",
        "150.000 TL",
    ]
)


require(
    (
        "Dijital Müşterilere Özel "
        "Pratik Finansman Kart"
    ),
    "taksit_sayisi",
    ["4"]
)


# ============================================================
# REDIRECT CAMPAIGNS
# ============================================================

for name in [
    "Togg Taşıt Finansmanı Kampanyası",
    "Umre Finansmanı Kampanyası",
    "Ücretsiz Ortak ATM Kampanyası",
]:

    if (
        campaign_by_name[
            name
        ][
            "kampanya_suresi"
        ]
    ):

        errors.append(
            (
                f"{name} -> "
                "redirect kampanya süresi "
                "boş olmalı"
            )
        )


# ============================================================
# SAVE FINAL DIRECT LIST
# ============================================================

with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        final_records,
        f,
        ensure_ascii=False,
        indent=4
    )


# ============================================================
# DISTRIBUTIONS
# ============================================================

record_type_counts = Counter(
    record[
        "kayit_turu"
    ]
    for record
    in final_records
)


finance_category_counts = Counter(
    record[
        "urun_kategorisi"
    ]
    for record
    in finance_records
)


campaign_category_counts = Counter(
    record[
        "urun_kategorisi"
    ]
    for record
    in campaign_records
)


# ============================================================
# PRINT
# ============================================================

print()
print("=" * 120)
print(
    "ALBARAKA TÜRK - "
    "FINAL AUDIT"
)
print("=" * 120)

print(
    "Toplam kayıt       :",
    len(final_records)
)

print(
    "Finansman          :",
    finance_count
)

print(
    "Kampanya           :",
    campaign_count
)

print(
    "Duplicate URL      :",
    duplicate_url
)

print(
    "Duplicate type/name:",
    duplicate_name_type
)

print(
    "Validation error   :",
    len(errors)
)

print(
    "Final JSON         :",
    OUTPUT_FILE
)


print()
print("=" * 120)
print(
    "KAYIT TÜRÜ DAĞILIMI"
)
print("=" * 120)

for key, value in (
    record_type_counts.items()
):

    print(
        f"{key:<15}: "
        f"{value}"
    )


print()
print("=" * 120)
print(
    "FİNANSMAN KATEGORİLERİ"
)
print("=" * 120)

for key, value in (
    finance_category_counts.items()
):

    print(
        f"{key:<35}: "
        f"{value}"
    )


print()
print("=" * 120)
print(
    "KAMPANYA KATEGORİLERİ"
)
print("=" * 120)

for key, value in (
    campaign_category_counts.items()
):

    print(
        f"{key:<25}: "
        f"{value}"
    )


if errors:

    print()
    print("=" * 120)
    print("HATALAR")
    print("=" * 120)

    for error in errors:

        print(
            "-",
            error
        )


print()
print("=" * 120)

if not errors:

    print(
        "SONUÇ: ALBARAKA TÜRK "
        "65/65 TAMAMEN BAŞARILI ✅"
    )

else:

    print(
        "SONUÇ: ALBARAKA TÜRK "
        "FINAL KONTROL GEREKİYOR ❌"
    )

print("=" * 120)


files.download(
    OUTPUT_FILE
)

ALBARAKA TÜRK - FINAL CLEANUP + MERGER V1
Finansman input: 17
Kampanya input : 48

Vade Farksız 140.000 TL'ye Varan Destek!
  Eski masraf: ['Masrafsız Banka ve Kredi Kartı']
  Yeni masraf: []

Taksitlio.com Alışveriş Finansmanı
  Eski vade: ['6 ay\nvade']
  Yeni vade: []

ALBARAKA TÜRK - FINAL AUDIT
Toplam kayıt       : 65
Finansman          : 17
Kampanya           : 48
Duplicate URL      : 2
Duplicate type/name: 0
Validation error   : 1
Final JSON         : /content/albaraka_turk_final.json

KAYIT TÜRÜ DAĞILIMI
finansman      : 17
kampanya       : 48

FİNANSMAN KATEGORİLERİ
Konut Finansmanı                   : 1
Taşıt Finansmanı                   : 5
Gayrimenkul Finansmanı             : 3
Bayide Finansman                   : 1
İhtiyaç Finansmanı                 : 6
BES Teminatlı Finansman            : 1

KAMPANYA KATEGORİLERİ
Dijital                  : 5
Bireysel                 : 34
Trend                    : 3
Özel Bankacılık          : 4
Eflatun                  : 1
Genel          

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# ALBARAKA TÜRK
# REDIRECT URL FIX + FINAL MERGER V2
# ============================================================

import json
from collections import Counter
from google.colab import files


FINANCE_FILE = (
    "/content/"
    "albaraka_turk_finansman_extracted.json"
)

CAMPAIGN_FILE = (
    "/content/"
    "albaraka_turk_kampanya_extracted.json"
)

OUTPUT_FILE = (
    "/content/"
    "albaraka_turk_final.json"
)


BANK_NAME = (
    "Albaraka Türk Katılım Bankası A.Ş."
)


SCHEMA_KEYS = [
    "banka",
    "kayit_turu",
    "urun_adi",
    "urun_kategorisi",
    "kar_payi_orani",
    "finansman_orani",
    "finansman_tutari",
    "vade",
    "taksit_sayisi",
    "masraf_bilgisi",
    "kampanya_turu",
    "kampanya_avantaji",
    "kampanya_suresi",
    "hedef_kitle",
    "para_birimi",
    "kosullar",
    "kaynak_url",
    "ham_metin",
]


LIST_FIELDS = {
    "kar_payi_orani",
    "finansman_orani",
    "finansman_tutari",
    "vade",
    "taksit_sayisi",
    "masraf_bilgisi",
    "kampanya_avantaji",
    "hedef_kitle",
    "para_birimi",
    "kosullar",
}


SCALAR_FIELDS = (
    set(SCHEMA_KEYS)
    -
    LIST_FIELDS
)


# ============================================================
# LOAD
# ============================================================

with open(
    FINANCE_FILE,
    "r",
    encoding="utf-8"
) as f:
    finance_records = json.load(f)


with open(
    CAMPAIGN_FILE,
    "r",
    encoding="utf-8"
) as f:
    campaign_records = json.load(f)


campaign_by_name = {
    record["urun_adi"]: record
    for record in campaign_records
}


print("=" * 120)
print(
    "ALBARAKA TÜRK - "
    "REDIRECT URL FIX + FINAL MERGER V2"
)
print("=" * 120)


# ============================================================
# ORIGINAL CAMPAIGN URLS
# ============================================================

URL_FIXES = {

    "Togg Taşıt Finansmanı Kampanyası":
        (
            "https://www.albaraka.com.tr/"
            "tr/kampanyalar/detay/"
            "togg-tasit-finansmani-kampanyasi"
        ),

    "Umre Finansmanı Kampanyası":
        (
            "https://www.albaraka.com.tr/"
            "tr/kampanyalar/detay/"
            "umre-finansmani-kampanyasi"
        ),

    "Ücretsiz Ortak ATM Kampanyası":
        (
            "https://www.albaraka.com.tr/"
            "tr/kampanyalar/detay/"
            "ortak-atm-is-birlikleri"
        ),
}


# ============================================================
# PATCH
# ============================================================

print()
print("=" * 120)
print("REDIRECT URL DÜZELTMELERİ")
print("=" * 120)


for name, original_url in URL_FIXES.items():

    record = campaign_by_name[
        name
    ]

    old_url = record[
        "kaynak_url"
    ]

    record[
        "kaynak_url"
    ] = original_url


    print()
    print(name)

    print(
        "  ESKİ:",
        old_url
    )

    print(
        "  YENİ:",
        record[
            "kaynak_url"
        ]
    )


# ============================================================
# SAVE UPDATED CAMPAIGNS
# ============================================================

with open(
    CAMPAIGN_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        campaign_records,
        f,
        ensure_ascii=False,
        indent=4
    )


# ============================================================
# MERGE
# ============================================================

final_records = (
    finance_records
    +
    campaign_records
)


# ============================================================
# VALIDATION
# ============================================================

errors = []


# ------------------------------------------------------------
# COUNTS
# ------------------------------------------------------------

if len(finance_records) != 17:

    errors.append(
        (
            "Finansman kayıt sayısı "
            f"{len(finance_records)} != 17"
        )
    )


if len(campaign_records) != 48:

    errors.append(
        (
            "Kampanya kayıt sayısı "
            f"{len(campaign_records)} != 48"
        )
    )


if len(final_records) != 65:

    errors.append(
        (
            "Final kayıt sayısı "
            f"{len(final_records)} != 65"
        )
    )


# ============================================================
# SCHEMA + TYPES
# ============================================================

for index, record in enumerate(
    final_records,
    start=1
):

    name = record.get(
        "urun_adi",
        f"#{index}"
    )


    if list(
        record.keys()
    ) != SCHEMA_KEYS:

        errors.append(
            (
                f"{name} -> "
                "schema/order hatası"
            )
        )


    for field in LIST_FIELDS:

        if not isinstance(
            record.get(field),
            list
        ):

            errors.append(
                (
                    f"{name} -> "
                    f"{field} list değil"
                )
            )


    for field in SCALAR_FIELDS:

        if not isinstance(
            record.get(field),
            str
        ):

            errors.append(
                (
                    f"{name} -> "
                    f"{field} string değil"
                )
            )


    if (
        record.get("banka")
        != BANK_NAME
    ):

        errors.append(
            (
                f"{name} -> "
                "banka yanlış"
            )
        )


    if record.get(
        "kayit_turu"
    ) not in {
        "finansman",
        "kampanya",
    }:

        errors.append(
            (
                f"{name} -> "
                "kayit_turu yanlış"
            )
        )


    for field in [
        "urun_adi",
        "kaynak_url",
        "ham_metin",
    ]:

        if not record.get(
            field
        ):

            errors.append(
                (
                    f"{name} -> "
                    f"{field} boş"
                )
            )


    if (
        "TRY"
        in record.get(
            "para_birimi",
            []
        )
    ):

        errors.append(
            (
                f"{name} -> "
                "TRY bulundu"
            )
        )


# ============================================================
# TYPE COUNTS
# ============================================================

finance_count = sum(
    record[
        "kayit_turu"
    ]
    == "finansman"
    for record
    in final_records
)


campaign_count = sum(
    record[
        "kayit_turu"
    ]
    == "kampanya"
    for record
    in final_records
)


if finance_count != 17:

    errors.append(
        (
            "Finansman count "
            f"{finance_count} != 17"
        )
    )


if campaign_count != 48:

    errors.append(
        (
            "Kampanya count "
            f"{campaign_count} != 48"
        )
    )


# ============================================================
# DUPLICATE URL
# ============================================================

urls = [
    record[
        "kaynak_url"
    ].rstrip("/")
    for record
    in final_records
]


url_counter = Counter(
    urls
)


duplicate_urls = {
    url: count
    for url, count
    in url_counter.items()
    if count > 1
}


duplicate_url_count = sum(
    count - 1
    for count in
    duplicate_urls.values()
)


if duplicate_url_count:

    errors.append(
        (
            "Duplicate URL: "
            f"{duplicate_url_count}"
        )
    )


# ============================================================
# DUPLICATE TYPE + NAME
# ============================================================

name_type_keys = [
    (
        record[
            "kayit_turu"
        ],
        record[
            "urun_adi"
        ],
    )
    for record
    in final_records
]


duplicate_name_type = (
    len(name_type_keys)
    -
    len(set(name_type_keys))
)


if duplicate_name_type:

    errors.append(
        (
            "Duplicate "
            "(kayit_turu, urun_adi): "
            f"{duplicate_name_type}"
        )
    )


# ============================================================
# REDIRECT URL CHECKS
# ============================================================

for name, expected_url in (
    URL_FIXES.items()
):

    actual = (
        campaign_by_name[
            name
        ][
            "kaynak_url"
        ]
    )


    if actual != expected_url:

        errors.append(
            (
                f"{name} -> "
                "campaign URL yanlış: "
                f"{actual}"
            )
        )


# ============================================================
# CRITICAL SEMANTIC CHECKS
# ============================================================

def require(
    name,
    field,
    expected
):

    actual = (
        campaign_by_name[
            name
        ][
            field
        ]
    )


    if actual != expected:

        errors.append(
            (
                f"{name} -> "
                f"{field}: "
                f"{actual} "
                f"!= {expected}"
            )
        )


# 140K
require(
    "Vade Farksız 140.000 TL'ye Varan Destek!",
    "kar_payi_orani",
    ["%0"]
)

require(
    "Vade Farksız 140.000 TL'ye Varan Destek!",
    "finansman_orani",
    []
)

require(
    "Vade Farksız 140.000 TL'ye Varan Destek!",
    "finansman_tutari",
    ["140.000 TL"]
)

require(
    "Vade Farksız 140.000 TL'ye Varan Destek!",
    "masraf_bilgisi",
    []
)


# Taksitlio
require(
    "Taksitlio.com Alışveriş Finansmanı",
    "kar_payi_orani",
    ["%2,99"]
)

require(
    "Taksitlio.com Alışveriş Finansmanı",
    "finansman_orani",
    []
)

require(
    "Taksitlio.com Alışveriş Finansmanı",
    "finansman_tutari",
    ["150.000 TL"]
)

require(
    "Taksitlio.com Alışveriş Finansmanı",
    "vade",
    []
)


# Dijital konut / taşıt
require(
    (
        "Dijitale Özel Konut ve Taşıt "
        "Finansmanı Kampanyası"
    ),
    "kar_payi_orani",
    [
        "%2,87",
        "%3,19",
    ]
)


# Pratik Finansman Kart
require(
    (
        "Dijital Müşterilere Özel "
        "Pratik Finansman Kart"
    ),
    "kar_payi_orani",
    [
        "%0",
        "%3,95",
        "%3,90",
        "%3,85",
    ]
)

require(
    (
        "Dijital Müşterilere Özel "
        "Pratik Finansman Kart"
    ),
    "finansman_tutari",
    [
        "40.000 TL",
        "150.000 TL",
    ]
)

require(
    (
        "Dijital Müşterilere Özel "
        "Pratik Finansman Kart"
    ),
    "taksit_sayisi",
    ["4"]
)


# ============================================================
# SAVE FINAL
# ============================================================

with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        final_records,
        f,
        ensure_ascii=False,
        indent=4
    )


# ============================================================
# PRINT FINAL
# ============================================================

print()
print("=" * 120)
print(
    "ALBARAKA TÜRK - "
    "FINAL AUDIT V2"
)
print("=" * 120)

print(
    "Toplam kayıt        :",
    len(final_records)
)

print(
    "Finansman           :",
    finance_count
)

print(
    "Kampanya            :",
    campaign_count
)

print(
    "Duplicate URL       :",
    duplicate_url_count
)

print(
    "Duplicate type/name :",
    duplicate_name_type
)

print(
    "Validation error    :",
    len(errors)
)

print(
    "Final JSON          :",
    OUTPUT_FILE
)


if duplicate_urls:

    print()
    print("=" * 120)
    print("DUPLICATE URL DETAY")
    print("=" * 120)

    for url, count in (
        duplicate_urls.items()
    ):

        print(
            count,
            "x",
            url
        )


if errors:

    print()
    print("=" * 120)
    print("HATALAR")
    print("=" * 120)

    for error in errors:

        print(
            "-",
            error
        )


print()
print("=" * 120)

if not errors:

    print(
        "SONUÇ: ALBARAKA TÜRK "
        "65/65 TAMAMEN BAŞARILI ✅"
    )

else:

    print(
        "SONUÇ: ALBARAKA TÜRK "
        "FINAL KONTROL GEREKİYOR ❌"
    )

print("=" * 120)


# Güncellenmiş standalone kampanya dosyası
files.download(
    CAMPAIGN_FILE
)

# Birleşik final
files.download(
    OUTPUT_FILE
)

ALBARAKA TÜRK - REDIRECT URL FIX + FINAL MERGER V2

REDIRECT URL DÜZELTMELERİ

Togg Taşıt Finansmanı Kampanyası
  ESKİ: https://www.albaraka.com.tr/tr/bireysel/finansmanlar/tasit-finansmani/togg-finansmani
  YENİ: https://www.albaraka.com.tr/tr/kampanyalar/detay/togg-tasit-finansmani-kampanyasi

Umre Finansmanı Kampanyası
  ESKİ: https://www.albaraka.com.tr/tr/bireysel/finansmanlar/ihtiyac/subesiz-umre-finansmani
  YENİ: https://www.albaraka.com.tr/tr/kampanyalar/detay/umre-finansmani-kampanyasi

Ücretsiz Ortak ATM Kampanyası
  ESKİ: https://www.albaraka.com.tr/tr/dijital-bankacilik/atm/ortak-atm-is-birlikleri
  YENİ: https://www.albaraka.com.tr/tr/kampanyalar/detay/ortak-atm-is-birlikleri

ALBARAKA TÜRK - FINAL AUDIT V2
Toplam kayıt        : 65
Finansman           : 17
Kampanya            : 48
Duplicate URL       : 0
Duplicate type/name : 0
Validation error    : 0
Final JSON          : /content/albaraka_turk_final.json

SONUÇ: ALBARAKA TÜRK 65/65 TAMAMEN BAŞARILI ✅


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# ALBARAKA TÜRK
# FINAL QUALITY PATCH V3
#
# Input:
#   /content/albaraka_turk_final.json
#
# Outputs:
#   /content/albaraka_turk_finansman_extracted_v3.json
#   /content/albaraka_turk_kampanya_extracted_v3.json
#   /content/albaraka_turk_final_quality_v3.json
#
# Amaç:
#   - Kesin semantik eksikleri düzelt
#   - Koşul başlık/gürültülerini temizle
#   - 65 kaydı baştan audit et
# ============================================================

import os
import re
import json
from collections import Counter
from datetime import datetime, date

from google.colab import files


# ============================================================
# PATH
# ============================================================

INPUT_CANDIDATES = [
    "/content/albaraka_turk_final.json",
    "/content/albaraka_turk_final (1).json",
]


INPUT_FILE = None

for candidate in INPUT_CANDIDATES:
    if os.path.exists(candidate):
        INPUT_FILE = candidate
        break


if INPUT_FILE is None:
    raise FileNotFoundError(
        "albaraka_turk_final.json bulunamadı."
    )


FINANCE_OUTPUT = (
    "/content/"
    "albaraka_turk_finansman_extracted_v3.json"
)

CAMPAIGN_OUTPUT = (
    "/content/"
    "albaraka_turk_kampanya_extracted_v3.json"
)

FINAL_OUTPUT = (
    "/content/"
    "albaraka_turk_final_quality_v3.json"
)


BANK_NAME = (
    "Albaraka Türk Katılım Bankası A.Ş."
)


TODAY = date(
    2026,
    8,
    23,
)


# ============================================================
# SCHEMA
# ============================================================

SCHEMA_KEYS = [
    "banka",
    "kayit_turu",
    "urun_adi",
    "urun_kategorisi",
    "kar_payi_orani",
    "finansman_orani",
    "finansman_tutari",
    "vade",
    "taksit_sayisi",
    "masraf_bilgisi",
    "kampanya_turu",
    "kampanya_avantaji",
    "kampanya_suresi",
    "hedef_kitle",
    "para_birimi",
    "kosullar",
    "kaynak_url",
    "ham_metin",
]


LIST_FIELDS = {
    "kar_payi_orani",
    "finansman_orani",
    "finansman_tutari",
    "vade",
    "taksit_sayisi",
    "masraf_bilgisi",
    "kampanya_avantaji",
    "hedef_kitle",
    "para_birimi",
    "kosullar",
}


SCALAR_FIELDS = (
    set(SCHEMA_KEYS)
    -
    LIST_FIELDS
)


# ============================================================
# HELPERS
# ============================================================

def clean_text(value):
    value = str(value or "")

    value = (
        value
        .replace("\xa0", " ")
        .replace("’", "'")
        .replace("‘", "'")
        .replace("–", "-")
        .replace("—", "-")
        .replace("\u00ad", "")
    )

    value = re.sub(
        r"[ \t]+",
        " ",
        value
    )

    value = re.sub(
        r"\n[ \t]+",
        "\n",
        value
    )

    value = re.sub(
        r"\n{3,}",
        "\n\n",
        value
    )

    return value.strip()


def normalize(value):
    return (
        clean_text(value)
        .replace("İ", "i")
        .replace("I", "ı")
        .casefold()
    )


def unique(values):
    result = []
    seen = set()

    for value in values:
        value = clean_text(value)

        if not value:
            continue

        key = normalize(value)

        if key in seen:
            continue

        seen.add(key)
        result.append(value)

    return result


# ============================================================
# LOAD
# ============================================================

with open(
    INPUT_FILE,
    "r",
    encoding="utf-8"
) as f:
    records = json.load(f)


if not isinstance(records, list):
    raise TypeError(
        "Final JSON direct list olmalı."
    )


print("=" * 125)
print(
    "ALBARAKA TÜRK - "
    "FINAL QUALITY PATCH V3"
)
print("=" * 125)

print(
    "Input:",
    INPUT_FILE
)

print(
    "Kayıt:",
    len(records)
)


by_name = {
    record["urun_adi"]: record
    for record in records
}


# ============================================================
# PATCH LOG
# ============================================================

patch_log = []


def log_patch(
    name,
    field,
    old,
    new,
):
    if old != new:
        patch_log.append({
            "urun_adi": name,
            "field": field,
            "old": old,
            "new": new,
        })


def set_field(
    name,
    field,
    new_value,
):
    record = by_name[name]

    old_value = record[field]

    log_patch(
        name,
        field,
        old_value,
        new_value,
    )

    record[field] = new_value


# ============================================================
# 1) TAKSİTLİO
#
# Ham metin:
# 150.000 TL'ye kadar
# 6 ay vade
# %2,99 kar oranı
# tahsis ücreti %0,5
# ============================================================

name = (
    "Taksitlio.com "
    "Alışveriş Finansmanı"
)

set_field(
    name,
    "kar_payi_orani",
    ["%2,99"],
)

set_field(
    name,
    "finansman_orani",
    [],
)

set_field(
    name,
    "finansman_tutari",
    ["150.000 TL"],
)

set_field(
    name,
    "vade",
    ["6 ay"],
)

set_field(
    name,
    "masraf_bilgisi",
    [
        (
            "Finansman tahsis ücreti, "
            "toplam finansman tutarının "
            "%0,5'i oranında tahsil edilir."
        )
    ],
)

set_field(
    name,
    "para_birimi",
    ["TL"],
)


# ============================================================
# 2) TOGG FİNANSMANI
#
# Product table:
#
# T10F V2      12 ay  1.000.000  %0
# T10F V2      48 ay  1.700.000  %2,99
# T10X V2      12 ay    800.000  %0
# T10X V2      48 ay  1.700.000  %2,99
# T10X 4MORE   10 ay  1.500.000  %0
# T10F 4MORE   36 ay  1.500.000  %3,05
#
# Ayrıca max finansman marjı %70.
# ============================================================

name = "Togg Finansmanı"


set_field(
    name,
    "kar_payi_orani",
    [
        "%0",
        "%2,99",
        "%3,05",
    ],
)

set_field(
    name,
    "finansman_orani",
    ["%70"],
)

set_field(
    name,
    "finansman_tutari",
    [
        "800.000 TL",
        "1.000.000 TL",
        "1.500.000 TL",
        "1.700.000 TL",
    ],
)

set_field(
    name,
    "vade",
    [
        "10 ay",
        "12 ay",
        "36 ay",
        "48 ay",
    ],
)

set_field(
    name,
    "para_birimi",
    ["TL"],
)


togg_conditions = [
    (
        "T10F V2 için 12 ay vadede "
        "1.000.000 TL finansman ve "
        "%0 aylık kâr oranı sunulur."
    ),
    (
        "T10F V2 için 48 ay vadede "
        "1.700.000 TL finansman ve "
        "%2,99 aylık kâr oranı sunulur."
    ),
    (
        "T10X V2 için 12 ay vadede "
        "800.000 TL finansman ve "
        "%0 aylık kâr oranı sunulur."
    ),
    (
        "T10X V2 için 48 ay vadede "
        "1.700.000 TL finansman ve "
        "%2,99 aylık kâr oranı sunulur."
    ),
    (
        "T10X V2 4MORE için 10 ay vadede "
        "1.500.000 TL finansman ve "
        "%0 aylık kâr oranı sunulur."
    ),
    (
        "T10F V2 4MORE için 36 ay vadede "
        "1.500.000 TL finansman ve "
        "%3,05 aylık kâr oranı sunulur."
    ),
    (
        "Proforma fatura tutarına göre "
        "finansman oranı en fazla %70'tir."
    ),
]


for condition in togg_conditions:
    if condition not in by_name[name]["kosullar"]:
        by_name[name]["kosullar"].append(
            condition
        )


# ============================================================
# 3) KONUT FİNANSMANI
#
# Bunlar kâr payı değil:
# ekspertiz değerine göre
# kullandırılabilecek finansman marjları.
# ============================================================

name = "Konut Finansmanı"


set_field(
    name,
    "finansman_orani",
    [
        "%90",
        "%80",
        "%70",
        "%60",
        "%50",
        "%40",
        "%30",
        "%22,5",
        "%20",
        "%17,5",
        "%15",
        "%12,5",
        "%10",
        "%7,5",
        "%5",
    ],
)

set_field(
    name,
    "para_birimi",
    ["TL"],
)


konut_conditions = [
    (
        "Standart konut alımında "
        "ekspertiz değeri, konut değer aralığı "
        "ve enerji sınıfına göre "
        "finansman oranı %20 ile %90 "
        "arasında değişmektedir."
    ),
    (
        "İkinci ve sonraki konut alımlarında "
        "ekspertiz değeri, konut değer aralığı "
        "ve enerji sınıfına göre "
        "finansman oranı %5 ile %22,5 "
        "arasında değişmektedir."
    ),
]


for condition in konut_conditions:
    if condition not in by_name[name]["kosullar"]:
        by_name[name]["kosullar"].append(
            condition
        )


# ============================================================
# 4) DENİZ TAŞITLARI
#
# Kaynakta:
# "Ödemelerinizi 36 kadar vadelendirebilir"
#
# Site cümlesinde "ay" düşmüş.
# Taşıt bağlamında 36 ay olarak normalize edilir.
# ============================================================

set_field(
    "Deniz Taşıtları Finansmanı",
    "vade",
    ["36 aya kadar"],
)


# ============================================================
# 5) TAŞIT KİRALAMA
# ============================================================

set_field(
    "Taşıt Kiralama Finansmanı",
    "vade",
    ["36 aya kadar"],
)


# ============================================================
# 6) 2B ARAZİ
# ============================================================

name = "2B Arazi Finansmanı"

set_field(
    name,
    "masraf_bilgisi",
    [
        "İpotek tesis ücreti alınmaz.",
        "Ekspertiz ücreti alınmaz.",
    ],
)


# ============================================================
# 7) DİJİTAL KATILMA HESABI
#
# 98/2 = kâr PAYLAŞIM oranı.
# Numeric kâr payı oranı değildir.
# ============================================================

name = (
    "Dijital Katılma Hesabı'na "
    "Özel Paylaşım Oranları!"
)

set_field(
    name,
    "kar_payi_orani",
    [],
)


# ============================================================
# 8) OTOPARK + VALE
#
# "%50 iade" avantajdır.
# "masraflarınızın yarısı cebinizde"
# masraf bilgisi değildir.
# ============================================================

name = (
    "Otopark ve Vale "
    "Harcamalarınıza %50 İade "
    "Albaraka'da!"
)

set_field(
    name,
    "masraf_bilgisi",
    [],
)


# ============================================================
# 9) albaFX
#
# İlk iki değer advantage / heading.
# Sadece gerçek MKK masraf açıklaması tutulur.
# ============================================================

name = (
    "albaFX'te Karma "
    "Düzey 1 Ücretsiz!"
)


old_fees = (
    by_name[
        name
    ][
        "masraf_bilgisi"
    ]
)


real_fees = [
    fee
    for fee in old_fees
    if (
        "MKK"
        in fee
        and
        (
            "ücret"
            in normalize(fee)
            or
            "komisyon"
            in normalize(fee)
        )
    )
]


set_field(
    name,
    "masraf_bilgisi",
    real_fees,
)


# ============================================================
# JET FİNANSMAN
#
# Kaynak kendi içinde tutarsız:
#
# Bir yerde:
# min 1.000 / max 60.000 TL
#
# Başka yerde vade tablosu:
# 50.001-100.000
# 100.001 ve üzeri
#
# Bu nedenle finansman_tutari BİLEREK boş.
# Tahmin yapılmaz.
# ============================================================

set_field(
    "Jet Finansman",
    "finansman_tutari",
    [],
)


# ============================================================
# CONDITION QUALITY CLEANER
#
# Sadece yüksek güvenli structural noise temizlenir.
# Ham metin ASLA değiştirilmez.
# ============================================================

EXACT_NOISE = {
    "kampanya başlangıç ve bitiş",
    "kampanya başlangıç ve bitiş tarihi",
    "kampanya başlangıç ve bitiş tarihleri",
    "kampanya başlangıç - bitiş tarihleri",

    "kampanyaya katılım adımları",
    "kampanyaya katılım adımları:",
    "kampanya katılım adımları",
    "kampanya katılım adımları:",
    "kampanyaya katılmak için:",
    "kampanyaya katılmak için",

    "kampanyadan kimler faydalanabilir?",
    "kampanyadan kimler faydalanabilir:",
    "kampanyadan kimler faydalanabilir",

    "ek kampanya detayları",
    "ek kampanya detayları:",
    "önemli hatırlatmalar ve koşullar",

    "tarihlerinde geçerlidir.",
    "tarihleri arasında geçerlidir.",

    "albaraka türk katılım bankası a.ş.",

    "müşteri ol",
    "hemen başvur",
}


def is_noise_condition(
    condition,
    product_name,
):

    c = clean_text(
        condition
    )

    n = normalize(
        c
    )

    product_n = normalize(
        product_name
    )


    if not c:
        return True


    # -----------------------------------------------
    # Exact structural noise
    # -----------------------------------------------

    if n in EXACT_NOISE:
        return True


    # -----------------------------------------------
    # Tek başına ürün/kampanya başlığı
    # -----------------------------------------------

    if n == product_n:
        return True


    # -----------------------------------------------
    # Generic campaign section headings
    # -----------------------------------------------

    heading_prefixes = [
        "kampanya başlangıç",
        "kampanya katılım adımları",
        "kampanyaya katılım adımları",
        "kampanyadan kimler faydalanabilir",
        "ek kampanya detayları",
    ]


    if (
        len(c) <= 90
        and
        any(
            n.startswith(prefix)
            for prefix in heading_prefixes
        )
    ):
        return True


    # -----------------------------------------------
    # Link-title contamination
    # e.g. "Sağlık Kampanyası | Albaraka Türk"
    # -----------------------------------------------

    if (
        "| albaraka türk"
        in n
        and
        len(c) <= 100
    ):
        return True


    # -----------------------------------------------
    # Pure section heading ending with colon
    # -----------------------------------------------

    if (
        c.endswith(":")
        and
        len(c) <= 75
        and
        not re.search(
            r"\d|%|TL",
            c,
            flags=re.I
        )
    ):
        return True


    # -----------------------------------------------
    # Common "Nedir?" headings
    # -----------------------------------------------

    if (
        len(c) <= 90
        and
        (
            n.endswith(" nedir?")
            or
            n.endswith(" nasıl başvurulur?")
            or
            n.endswith(" nasıl kullanılır?")
            or
            n.endswith(" avantajları nelerdir?")
        )
    ):
        return True


    # -----------------------------------------------
    # Broken tiny fragments
    # -----------------------------------------------

    if (
        len(c) < 12
        and
        not re.search(
            r"\d|%|TL",
            c,
            flags=re.I
        )
    ):
        return True


    return False


removed_conditions = []


for record in records:

    old_conditions = list(
        record[
            "kosullar"
        ]
    )

    new_conditions = []


    for condition in old_conditions:

        condition = clean_text(
            condition
        )


        if is_noise_condition(
            condition,
            record[
                "urun_adi"
            ],
        ):

            removed_conditions.append({
                "urun_adi":
                    record[
                        "urun_adi"
                    ],

                "condition":
                    condition,
            })

            continue


        new_conditions.append(
            condition
        )


    record[
        "kosullar"
    ] = unique(
        new_conditions
    )


# ============================================================
# FINAL VALIDATION
# ============================================================

errors = []
warnings = []


# ------------------------------------------------------------
# Counts
# ------------------------------------------------------------

finance_records = [
    r
    for r in records
    if r[
        "kayit_turu"
    ] == "finansman"
]


campaign_records = [
    r
    for r in records
    if r[
        "kayit_turu"
    ] == "kampanya"
]


if len(records) != 65:
    errors.append(
        f"Toplam kayıt {len(records)} != 65"
    )


if len(finance_records) != 17:
    errors.append(
        (
            "Finansman "
            f"{len(finance_records)} != 17"
        )
    )


if len(campaign_records) != 48:
    errors.append(
        (
            "Kampanya "
            f"{len(campaign_records)} != 48"
        )
    )


# ------------------------------------------------------------
# Schema + types
# ------------------------------------------------------------

for record in records:

    name = record[
        "urun_adi"
    ]


    if list(
        record.keys()
    ) != SCHEMA_KEYS:

        errors.append(
            (
                f"{name} -> "
                "schema/order hatası"
            )
        )


    for field in LIST_FIELDS:

        if not isinstance(
            record[field],
            list
        ):

            errors.append(
                (
                    f"{name} -> "
                    f"{field} list değil"
                )
            )


    for field in SCALAR_FIELDS:

        if not isinstance(
            record[field],
            str
        ):

            errors.append(
                (
                    f"{name} -> "
                    f"{field} string değil"
                )
            )


    if (
        record[
            "banka"
        ]
        != BANK_NAME
    ):

        errors.append(
            (
                f"{name} -> "
                "banka yanlış"
            )
        )


    if record[
        "kayit_turu"
    ] not in {
        "finansman",
        "kampanya",
    }:

        errors.append(
            (
                f"{name} -> "
                "kayit_turu yanlış"
            )
        )


    for field in [
        "urun_adi",
        "kaynak_url",
        "ham_metin",
    ]:

        if not record[
            field
        ]:

            errors.append(
                (
                    f"{name} -> "
                    f"{field} boş"
                )
            )


    if (
        "TRY"
        in record[
            "para_birimi"
        ]
    ):

        errors.append(
            (
                f"{name} -> "
                "TRY bulundu"
            )
        )


# ============================================================
# DUPLICATE AUDIT
# ============================================================

urls = [
    r[
        "kaynak_url"
    ].rstrip("/")
    for r in records
]


url_counter = Counter(
    urls
)


duplicate_urls = {
    url: count
    for url, count
    in url_counter.items()
    if count > 1
}


duplicate_url_count = sum(
    count - 1
    for count
    in duplicate_urls.values()
)


if duplicate_url_count:

    errors.append(
        (
            "Duplicate URL: "
            f"{duplicate_url_count}"
        )
    )


name_type_keys = [
    (
        r[
            "kayit_turu"
        ],
        r[
            "urun_adi"
        ],
    )
    for r in records
]


duplicate_name_type = (
    len(name_type_keys)
    -
    len(set(name_type_keys))
)


if duplicate_name_type:

    errors.append(
        (
            "Duplicate type/name: "
            f"{duplicate_name_type}"
        )
    )


# ============================================================
# CAMPAIGN DATE AUDIT
# ============================================================

expired = []
bad_period = []


for record in campaign_records:

    period = record[
        "kampanya_suresi"
    ]


    if not period:
        continue


    match = re.fullmatch(
        (
            r"(\d{2}\.\d{2}\.\d{4})"
            r"\s*-\s*"
            r"(\d{2}\.\d{2}\.\d{4})"
        ),
        period
    )


    if not match:

        bad_period.append(
            record[
                "urun_adi"
            ]
        )

        continue


    end_date = datetime.strptime(
        match.group(2),
        "%d.%m.%Y"
    ).date()


    if end_date < TODAY:

        expired.append(
            record[
                "urun_adi"
            ]
        )


if bad_period:
    errors.append(
        (
            "Süre format hatası: "
            f"{bad_period}"
        )
    )


if expired:
    errors.append(
        (
            "Süresi geçmiş kampanya: "
            f"{expired}"
        )
    )


# ============================================================
# CRITICAL SEMANTIC REQUIRES
# ============================================================

by_name = {
    record[
        "urun_adi"
    ]:
    record
    for record
    in records
}


def require(
    name,
    field,
    expected,
):

    actual = (
        by_name[
            name
        ][
            field
        ]
    )


    if actual != expected:

        errors.append(
            (
                f"{name} -> "
                f"{field}: "
                f"{actual} "
                f"!= {expected}"
            )
        )


# ------------------------------------------------------------
# Taksitlio
# ------------------------------------------------------------

require(
    "Taksitlio.com Alışveriş Finansmanı",
    "kar_payi_orani",
    ["%2,99"]
)

require(
    "Taksitlio.com Alışveriş Finansmanı",
    "finansman_tutari",
    ["150.000 TL"]
)

require(
    "Taksitlio.com Alışveriş Finansmanı",
    "vade",
    ["6 ay"]
)


# ------------------------------------------------------------
# Togg
# ------------------------------------------------------------

require(
    "Togg Finansmanı",
    "kar_payi_orani",
    [
        "%0",
        "%2,99",
        "%3,05",
    ]
)

require(
    "Togg Finansmanı",
    "finansman_orani",
    ["%70"]
)

require(
    "Togg Finansmanı",
    "finansman_tutari",
    [
        "800.000 TL",
        "1.000.000 TL",
        "1.500.000 TL",
        "1.700.000 TL",
    ]
)

require(
    "Togg Finansmanı",
    "vade",
    [
        "10 ay",
        "12 ay",
        "36 ay",
        "48 ay",
    ]
)


# ------------------------------------------------------------
# Konut
# ------------------------------------------------------------

require(
    "Konut Finansmanı",
    "finansman_orani",
    [
        "%90",
        "%80",
        "%70",
        "%60",
        "%50",
        "%40",
        "%30",
        "%22,5",
        "%20",
        "%17,5",
        "%15",
        "%12,5",
        "%10",
        "%7,5",
        "%5",
    ]
)


# ------------------------------------------------------------
# Deniz / Kiralama
# ------------------------------------------------------------

require(
    "Deniz Taşıtları Finansmanı",
    "vade",
    ["36 aya kadar"]
)

require(
    "Taşıt Kiralama Finansmanı",
    "vade",
    ["36 aya kadar"]
)


# ------------------------------------------------------------
# 2B
# ------------------------------------------------------------

require(
    "2B Arazi Finansmanı",
    "masraf_bilgisi",
    [
        "İpotek tesis ücreti alınmaz.",
        "Ekspertiz ücreti alınmaz.",
    ]
)


# ------------------------------------------------------------
# Digital Katılma
# ------------------------------------------------------------

require(
    (
        "Dijital Katılma Hesabı'na "
        "Özel Paylaşım Oranları!"
    ),
    "kar_payi_orani",
    []
)


# ------------------------------------------------------------
# Otopark vale
# ------------------------------------------------------------

require(
    (
        "Otopark ve Vale "
        "Harcamalarınıza %50 İade "
        "Albaraka'da!"
    ),
    "masraf_bilgisi",
    []
)


# ============================================================
# FINANCE/CAMPAIGN LEAK AUDIT
# ============================================================

for record in finance_records:

    name = record[
        "urun_adi"
    ]


    if record[
        "kampanya_turu"
    ]:

        errors.append(
            (
                f"{name} -> "
                "finansman kaydında "
                "kampanya_turu dolu"
            )
        )


    if record[
        "kampanya_avantaji"
    ]:

        errors.append(
            (
                f"{name} -> "
                "finansman kaydında "
                "kampanya_avantaji dolu"
            )
        )


    if record[
        "kampanya_suresi"
    ]:

        errors.append(
            (
                f"{name} -> "
                "finansman kaydında "
                "kampanya_suresi dolu"
            )
        )


# ============================================================
# KNOWN SOURCE CONFLICT
# ============================================================

jet = by_name[
    "Jet Finansman"
]


if (
    "60.000 TL"
    in jet[
        "ham_metin"
    ]
    and
    "100.001 TL"
    in jet[
        "ham_metin"
    ]
):

    warnings.append(
        (
            "Jet Finansman: kaynak sayfada "
            "maksimum 60.000 TL ifadesi ile "
            "100.001 TL ve üzeri vade bandı "
            "aynı anda bulundu. "
            "finansman_tutari bilerek boş bırakıldı."
        )
    )


# ============================================================
# REMAINING STRUCTURAL NOISE AUDIT
# ============================================================

remaining_noise = []


for record in records:

    for condition in record[
        "kosullar"
    ]:

        if is_noise_condition(
            condition,
            record[
                "urun_adi"
            ],
        ):

            remaining_noise.append(
                (
                    record[
                        "urun_adi"
                    ],
                    condition,
                )
            )


if remaining_noise:

    errors.append(
        (
            "Koşullarda structural noise "
            f"kaldı: {len(remaining_noise)}"
        )
    )


# ============================================================
# SAVE
# ============================================================

finance_records = [
    record
    for record in records
    if record[
        "kayit_turu"
    ] == "finansman"
]


campaign_records = [
    record
    for record in records
    if record[
        "kayit_turu"
    ] == "kampanya"
]


with open(
    FINANCE_OUTPUT,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        finance_records,
        f,
        ensure_ascii=False,
        indent=4
    )


with open(
    CAMPAIGN_OUTPUT,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        campaign_records,
        f,
        ensure_ascii=False,
        indent=4
    )


with open(
    FINAL_OUTPUT,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        records,
        f,
        ensure_ascii=False,
        indent=4
    )


# ============================================================
# AUDIT SUMMARY
# ============================================================

print()
print("=" * 125)
print(
    "ALBARAKA TÜRK - "
    "FINAL QUALITY AUDIT V3"
)
print("=" * 125)

print(
    "Toplam kayıt          :",
    len(records)
)

print(
    "Finansman             :",
    len(finance_records)
)

print(
    "Kampanya              :",
    len(campaign_records)
)

print(
    "Uygulanan field patch :",
    len(patch_log)
)

print(
    "Silinen koşul gürültü :",
    len(removed_conditions)
)

print(
    "Kalan structural noise:",
    len(remaining_noise)
)

print(
    "Duplicate URL         :",
    duplicate_url_count
)

print(
    "Duplicate type/name   :",
    duplicate_name_type
)

print(
    "Expired campaign      :",
    len(expired)
)

print(
    "Validation error      :",
    len(errors)
)

print(
    "Warning               :",
    len(warnings)
)

print()
print(
    "Finance JSON:",
    FINANCE_OUTPUT
)

print(
    "Campaign JSON:",
    CAMPAIGN_OUTPUT
)

print(
    "Final JSON   :",
    FINAL_OUTPUT
)


# ============================================================
# PATCH DETAIL
# ============================================================

print()
print("=" * 125)
print("PATCH DETAYLARI")
print("=" * 125)


for patch in patch_log:

    print()
    print(
        patch[
            "urun_adi"
        ]
    )

    print(
        "  Alan :",
        patch[
            "field"
        ]
    )

    print(
        "  Eski :",
        patch[
            "old"
        ]
    )

    print(
        "  Yeni :",
        patch[
            "new"
        ]
    )


# ============================================================
# CRITICAL FINAL VALUES
# ============================================================

print()
print("=" * 125)
print("KRİTİK FINAL DEĞERLER")
print("=" * 125)


for name in [
    "Konut Finansmanı",
    "Togg Finansmanı",
    "Deniz Taşıtları Finansmanı",
    "Taşıt Kiralama Finansmanı",
    "2B Arazi Finansmanı",
    "Jet Finansman",
    "Taksitlio.com Alışveriş Finansmanı",
    (
        "Dijital Katılma Hesabı'na "
        "Özel Paylaşım Oranları!"
    ),
    (
        "Otopark ve Vale "
        "Harcamalarınıza %50 İade "
        "Albaraka'da!"
    ),
    "albaFX'te Karma Düzey 1 Ücretsiz!",
]:

    r = by_name[
        name
    ]

    print()
    print(name)

    print(
        "  Kâr Payı :",
        r[
            "kar_payi_orani"
        ]
    )

    print(
        "  Fin.Oran :",
        r[
            "finansman_orani"
        ]
    )

    print(
        "  Fin.Tutar:",
        r[
            "finansman_tutari"
        ]
    )

    print(
        "  Vade     :",
        r[
            "vade"
        ]
    )

    print(
        "  Taksit   :",
        r[
            "taksit_sayisi"
        ]
    )

    print(
        "  Masraf   :",
        r[
            "masraf_bilgisi"
        ]
    )


# ============================================================
# WARNINGS
# ============================================================

if warnings:

    print()
    print("=" * 125)
    print("UYARILAR")
    print("=" * 125)

    for warning in warnings:
        print(
            "-",
            warning
        )


# ============================================================
# ERRORS
# ============================================================

if errors:

    print()
    print("=" * 125)
    print("HATALAR")
    print("=" * 125)

    for error in errors:
        print(
            "-",
            error
        )


print()
print("=" * 125)

if not errors:

    print(
        "SONUÇ: ALBARAKA TÜRK "
        "FINAL QUALITY V3 "
        "65/65 BAŞARILI ✅"
    )

else:

    print(
        "SONUÇ: ALBARAKA TÜRK "
        "FINAL QUALITY V3 "
        "KONTROL GEREKİYOR ❌"
    )

print("=" * 125)


# ============================================================
# DOWNLOAD
# ============================================================

files.download(
    FINANCE_OUTPUT
)

files.download(
    CAMPAIGN_OUTPUT
)

files.download(
    FINAL_OUTPUT
)

ALBARAKA TÜRK - FINAL QUALITY PATCH V3
Input: /content/albaraka_turk_final.json
Kayıt: 65

ALBARAKA TÜRK - FINAL QUALITY AUDIT V3
Toplam kayıt          : 65
Finansman             : 17
Kampanya              : 48
Uygulanan field patch : 13
Silinen koşul gürültü : 214
Kalan structural noise: 0
Duplicate URL         : 0
Duplicate type/name   : 0
Expired campaign      : 0
Validation error      : 0

Finance JSON: /content/albaraka_turk_finansman_extracted_v3.json
Campaign JSON: /content/albaraka_turk_kampanya_extracted_v3.json
Final JSON   : /content/albaraka_turk_final_quality_v3.json

PATCH DETAYLARI

Taksitlio.com Alışveriş Finansmanı
  Alan : vade
  Eski : []
  Yeni : ['6 ay']

Togg Finansmanı
  Alan : kar_payi_orani
  Eski : []
  Yeni : ['%0', '%2,99', '%3,05']

Togg Finansmanı
  Alan : finansman_tutari
  Eski : []
  Yeni : ['800.000 TL', '1.000.000 TL', '1.500.000 TL', '1.700.000 TL']

Togg Finansmanı
  Alan : vade
  Eski : ['48 aya kadar']
  Yeni : ['10 ay', '12 ay', '36 ay', '48 ay']

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# ALBARAKA TÜRK
# FINAL QUALITY PATCH V4 - FINAL CLOSE
#
# Input:
#   /content/albaraka_turk_final_quality_v3.json
#
# Outputs:
#   /content/albaraka_turk_finansman_extracted_v4.json
#   /content/albaraka_turk_kampanya_extracted_v4.json
#   /content/albaraka_turk_final_quality_v4.json
# ============================================================

import json
from collections import Counter
from datetime import datetime, date

from google.colab import files


# ============================================================
# PATHS
# ============================================================

INPUT_FILE = (
    "/content/"
    "albaraka_turk_final_quality_v3.json"
)

FINANCE_OUTPUT = (
    "/content/"
    "albaraka_turk_finansman_extracted_v4.json"
)

CAMPAIGN_OUTPUT = (
    "/content/"
    "albaraka_turk_kampanya_extracted_v4.json"
)

FINAL_OUTPUT = (
    "/content/"
    "albaraka_turk_final_quality_v4.json"
)


BANK_NAME = (
    "Albaraka Türk Katılım Bankası A.Ş."
)

TODAY = date(
    2026,
    8,
    23,
)


# ============================================================
# SCHEMA
# ============================================================

SCHEMA_KEYS = [
    "banka",
    "kayit_turu",
    "urun_adi",
    "urun_kategorisi",
    "kar_payi_orani",
    "finansman_orani",
    "finansman_tutari",
    "vade",
    "taksit_sayisi",
    "masraf_bilgisi",
    "kampanya_turu",
    "kampanya_avantaji",
    "kampanya_suresi",
    "hedef_kitle",
    "para_birimi",
    "kosullar",
    "kaynak_url",
    "ham_metin",
]


LIST_FIELDS = {
    "kar_payi_orani",
    "finansman_orani",
    "finansman_tutari",
    "vade",
    "taksit_sayisi",
    "masraf_bilgisi",
    "kampanya_avantaji",
    "hedef_kitle",
    "para_birimi",
    "kosullar",
}


SCALAR_FIELDS = (
    set(SCHEMA_KEYS)
    -
    LIST_FIELDS
)


# ============================================================
# LOAD
# ============================================================

with open(
    INPUT_FILE,
    "r",
    encoding="utf-8"
) as f:
    records = json.load(f)


if not isinstance(
    records,
    list
):
    raise TypeError(
        "Final JSON direct list olmalı."
    )


by_name = {
    r["urun_adi"]: r
    for r in records
}


print("=" * 125)
print(
    "ALBARAKA TÜRK - "
    "FINAL QUALITY PATCH V4"
)
print("=" * 125)

print(
    "Input:",
    INPUT_FILE
)

print(
    "Kayıt:",
    len(records)
)


# ============================================================
# PATCH LOGGER
# ============================================================

patches = []


def patch(
    name,
    field,
    value,
):

    old = by_name[
        name
    ][
        field
    ]

    if old != value:

        patches.append({
            "urun_adi":
                name,

            "field":
                field,

            "old":
                old,

            "new":
                value,
        })

        by_name[
            name
        ][
            field
        ] = value


# ============================================================
# 1) ALBARAKA'DA MASRAFLARA SON
# ============================================================

name = (
    "Albaraka'da "
    "Masraflara Son!"
)


patch(
    name,
    "masraf_bilgisi",
    [
        (
            "TL, Döviz veya Altın cari ve "
            "katılma hesaplarından hesap "
            "işletim ücreti alınmaz."
        ),
        (
            "Albaraka Mobil, İnternet Şube "
            "ve Albaraka ATM'lerinde yurt içi "
            "TL EFT, Havale ve FAST işlemleri "
            "ücretsizdir."
        ),
        (
            "İnternet ve mobil üzerinden "
            "SWIFT işleminde banka işlem "
            "ücreti alınmaz; kısa SWIFT "
            "masrafı 25 USD, uzun SWIFT "
            "masrafı 40 USD'dir."
        ),
        (
            "Şubeden SWIFT işlemi ücretsiz "
            "para transferi kapsamında değildir."
        ),
        (
            "Debit kart ve kredi kartı için "
            "aidat veya ücret alınmaz."
        ),
        (
            "Kredi kartı borç ödeme işlemleri "
            "limitsiz ve komisyonsuzdur."
        ),
    ],
)


patch(
    name,
    "para_birimi",
    [
        "TL",
        "USD",
    ],
)


# ============================================================
# 2) VADE FARKSIZ 140K - TARGET CLEAN
# ============================================================

name = (
    "Vade Farksız 140.000 TL'ye "
    "Varan Destek!"
)


patch(
    name,
    "hedef_kitle",
    [
        (
            "Albaraka Mobil üzerinden "
            "dijital olarak müşteri olanlar"
        ),
    ],
)


# ============================================================
# 3) AĞUSTOS FATURA - TARGET CLEAN
# ============================================================

name = (
    "Ağustos Ayına Özel "
    "Fatura Kampanyası"
)


patch(
    name,
    "hedef_kitle",
    [
        "Albaraka World kredi kartı sahipleri",
        (
            "1-31 Ağustos 2026 tarihleri arasında "
            "OFT2026 davet kodunu kullanarak "
            "görüntülü görüşme ile ilk kez "
            "Albaraka müşterisi olanlar"
        ),
        "Bireysel müşteriler",
    ],
)


# ============================================================
# 4) YURT DIŞI ÇIKIŞ HARCI - TARGET CLEAN
# ============================================================

name = (
    "Yurt Dışı Çıkış Harcı Kampanyası "
    "| 1.250 TL Worldpuan!"
)


patch(
    name,
    "hedef_kitle",
    [
        "Albaraka World kredi kartı sahipleri",
        (
            "1 Temmuz 2026 itibarıyla "
            "Albaraka Mobil üzerinden "
            "görüntülü görüşme ile yeni "
            "müşteri olanlar"
        ),
        "Bireysel müşteriler",
    ],
)


# ============================================================
# 5) DİJİTAL KATILMA HESABI - TARGET CLEAN
# ============================================================

name = (
    "Dijital Katılma Hesabı'na "
    "Özel Paylaşım Oranları!"
)


patch(
    name,
    "hedef_kitle",
    [
        (
            "Albaraka Mobil üzerinden görüntülü "
            "görüşme ile ilk kez müşteri olup "
            "TL cinsinden Katılma Hesabı açanlar"
        ),
        "Gerçek kişiler",
    ],
)


# ============================================================
# VALIDATION
# ============================================================

errors = []
warnings = []


finance_records = [
    r
    for r in records
    if r[
        "kayit_turu"
    ] == "finansman"
]


campaign_records = [
    r
    for r in records
    if r[
        "kayit_turu"
    ] == "kampanya"
]


# ============================================================
# COUNT
# ============================================================

if len(records) != 65:

    errors.append(
        (
            f"Toplam kayıt "
            f"{len(records)} != 65"
        )
    )


if len(
    finance_records
) != 17:

    errors.append(
        (
            f"Finansman "
            f"{len(finance_records)} != 17"
        )
    )


if len(
    campaign_records
) != 48:

    errors.append(
        (
            f"Kampanya "
            f"{len(campaign_records)} != 48"
        )
    )


# ============================================================
# SCHEMA / TYPE
# ============================================================

for record in records:

    name = record[
        "urun_adi"
    ]


    if list(
        record.keys()
    ) != SCHEMA_KEYS:

        errors.append(
            (
                f"{name} -> "
                "schema/order hatası"
            )
        )


    for field in LIST_FIELDS:

        if not isinstance(
            record[field],
            list
        ):

            errors.append(
                (
                    f"{name} -> "
                    f"{field} list değil"
                )
            )


    for field in SCALAR_FIELDS:

        if not isinstance(
            record[field],
            str
        ):

            errors.append(
                (
                    f"{name} -> "
                    f"{field} string değil"
                )
            )


    if (
        record[
            "banka"
        ]
        != BANK_NAME
    ):

        errors.append(
            (
                f"{name} -> "
                "banka yanlış"
            )
        )


    if record[
        "kayit_turu"
    ] not in {
        "finansman",
        "kampanya",
    }:

        errors.append(
            (
                f"{name} -> "
                "kayit_turu yanlış"
            )
        )


    for field in [
        "urun_adi",
        "kaynak_url",
        "ham_metin",
    ]:

        if not record[
            field
        ]:

            errors.append(
                (
                    f"{name} -> "
                    f"{field} boş"
                )
            )


    if (
        "TRY"
        in record[
            "para_birimi"
        ]
    ):

        errors.append(
            (
                f"{name} -> "
                "TRY bulundu"
            )
        )


# ============================================================
# DUPLICATES
# ============================================================

urls = [
    r[
        "kaynak_url"
    ].rstrip("/")
    for r in records
]


url_counter = Counter(
    urls
)


duplicate_urls = {
    url: count
    for url, count
    in url_counter.items()
    if count > 1
}


duplicate_url_count = sum(
    count - 1
    for count
    in duplicate_urls.values()
)


if duplicate_url_count:

    errors.append(
        (
            "Duplicate URL: "
            f"{duplicate_url_count}"
        )
    )


type_name_keys = [
    (
        r[
            "kayit_turu"
        ],
        r[
            "urun_adi"
        ],
    )
    for r in records
]


duplicate_type_name = (
    len(type_name_keys)
    -
    len(set(type_name_keys))
)


if duplicate_type_name:

    errors.append(
        (
            "Duplicate type/name: "
            f"{duplicate_type_name}"
        )
    )


# ============================================================
# CAMPAIGN DATE
# ============================================================

expired = []
bad_period = []


for record in campaign_records:

    period = record[
        "kampanya_suresi"
    ]


    if not period:
        continue


    try:

        start_text, end_text = [
            x.strip()
            for x
            in period.split(
                " - "
            )
        ]


        start_date = datetime.strptime(
            start_text,
            "%d.%m.%Y"
        ).date()


        end_date = datetime.strptime(
            end_text,
            "%d.%m.%Y"
        ).date()


        if start_date > end_date:

            bad_period.append(
                record[
                    "urun_adi"
                ]
            )


        if end_date < TODAY:

            expired.append(
                record[
                    "urun_adi"
                ]
            )


    except Exception:

        bad_period.append(
            record[
                "urun_adi"
            ]
        )


if bad_period:

    errors.append(
        (
            "Bad campaign period: "
            f"{bad_period}"
        )
    )


if expired:

    errors.append(
        (
            "Expired campaign: "
            f"{expired}"
        )
    )


# ============================================================
# CRITICAL V3 VALUES MUST SURVIVE
# ============================================================

def require(
    name,
    field,
    expected,
):

    actual = by_name[
        name
    ][
        field
    ]

    if actual != expected:

        errors.append(
            (
                f"{name} -> "
                f"{field}: "
                f"{actual} "
                f"!= {expected}"
            )
        )


# Taksitlio
require(
    "Taksitlio.com Alışveriş Finansmanı",
    "kar_payi_orani",
    ["%2,99"]
)

require(
    "Taksitlio.com Alışveriş Finansmanı",
    "finansman_tutari",
    ["150.000 TL"]
)

require(
    "Taksitlio.com Alışveriş Finansmanı",
    "vade",
    ["6 ay"]
)


# Togg
require(
    "Togg Finansmanı",
    "kar_payi_orani",
    [
        "%0",
        "%2,99",
        "%3,05",
    ]
)

require(
    "Togg Finansmanı",
    "finansman_orani",
    ["%70"]
)

require(
    "Togg Finansmanı",
    "finansman_tutari",
    [
        "800.000 TL",
        "1.000.000 TL",
        "1.500.000 TL",
        "1.700.000 TL",
    ]
)

require(
    "Togg Finansmanı",
    "vade",
    [
        "10 ay",
        "12 ay",
        "36 ay",
        "48 ay",
    ]
)


# Konut
require(
    "Konut Finansmanı",
    "finansman_orani",
    [
        "%90",
        "%80",
        "%70",
        "%60",
        "%50",
        "%40",
        "%30",
        "%22,5",
        "%20",
        "%17,5",
        "%15",
        "%12,5",
        "%10",
        "%7,5",
        "%5",
    ]
)


# Deniz + kiralama
require(
    "Deniz Taşıtları Finansmanı",
    "vade",
    ["36 aya kadar"]
)

require(
    "Taşıt Kiralama Finansmanı",
    "vade",
    ["36 aya kadar"]
)


# 2B
require(
    "2B Arazi Finansmanı",
    "masraf_bilgisi",
    [
        "İpotek tesis ücreti alınmaz.",
        "Ekspertiz ücreti alınmaz.",
    ]
)


# Digital katılma must NOT leak 98/2 to kar payı
require(
    (
        "Dijital Katılma Hesabı'na "
        "Özel Paylaşım Oranları!"
    ),
    "kar_payi_orani",
    []
)


# ============================================================
# V4 SPECIFIC CHECKS
# ============================================================

require(
    "Albaraka'da Masraflara Son!",
    "para_birimi",
    [
        "TL",
        "USD",
    ]
)


masraf = by_name[
    "Albaraka'da Masraflara Son!"
][
    "masraf_bilgisi"
]


if not any(
    "25 USD"
    in x
    and
    "40 USD"
    in x
    for x in masraf
):

    errors.append(
        (
            "Albaraka'da Masraflara Son -> "
            "SWIFT masraf bilgisi eksik"
        )
    )


if not any(
    "hesap işletim ücreti"
    in x.lower()
    for x in masraf
):

    errors.append(
        (
            "Albaraka'da Masraflara Son -> "
            "hesap işletim ücreti bilgisi eksik"
        )
    )


if not any(
    "aidat"
    in x.lower()
    for x in masraf
):

    errors.append(
        (
            "Albaraka'da Masraflara Son -> "
            "kart aidat bilgisi eksik"
        )
    )


# ============================================================
# TARGET QUALITY
# ============================================================

target_fragment_errors = []


for record in campaign_records:

    for target in record[
        "hedef_kitle"
    ]:

        target = target.strip()


        if (
            target.endswith(",")
            or
            target.endswith(";")
        ):

            target_fragment_errors.append(
                (
                    record[
                        "urun_adi"
                    ],
                    target,
                )
            )


if target_fragment_errors:

    errors.append(
        (
            "Yarım hedef_kitle "
            f"parçası: "
            f"{target_fragment_errors}"
        )
    )


# ============================================================
# FINANCE CAMPAIGN FIELD LEAK
# ============================================================

for record in finance_records:

    if record[
        "kampanya_turu"
    ]:

        errors.append(
            (
                f"{record['urun_adi']} -> "
                "finansman kaydında "
                "kampanya_turu dolu"
            )
        )


    if record[
        "kampanya_avantaji"
    ]:

        errors.append(
            (
                f"{record['urun_adi']} -> "
                "finansman kaydında "
                "kampanya_avantaji dolu"
            )
        )


    if record[
        "kampanya_suresi"
    ]:

        errors.append(
            (
                f"{record['urun_adi']} -> "
                "finansman kaydında "
                "kampanya_suresi dolu"
            )
        )


# ============================================================
# REDIRECT CAMPAIGNS
# ============================================================

for name in [
    "Togg Taşıt Finansmanı Kampanyası",
    "Umre Finansmanı Kampanyası",
    "Ücretsiz Ortak ATM Kampanyası",
]:

    if by_name[
        name
    ][
        "kampanya_suresi"
    ]:

        errors.append(
            (
                f"{name} -> "
                "redirect kampanya "
                "süresi boş olmalı"
            )
        )


# ============================================================
# KNOWN SOURCE CONFLICT
# ============================================================

jet = by_name[
    "Jet Finansman"
]


if (
    "60.000 TL"
    in jet[
        "ham_metin"
    ]
    and
    "100.001 TL"
    in jet[
        "ham_metin"
    ]
):

    warnings.append(
        (
            "Jet Finansman: resmi kaynak "
            "kendi içinde tutarsız. "
            "Maksimum 60.000 TL ifadesi "
            "ile 100.001 TL ve üzeri "
            "vade bandı aynı sayfada. "
            "finansman_tutari bilerek boş."
        )
    )


# ============================================================
# SAVE
# ============================================================

with open(
    FINANCE_OUTPUT,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        finance_records,
        f,
        ensure_ascii=False,
        indent=4
    )


with open(
    CAMPAIGN_OUTPUT,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        campaign_records,
        f,
        ensure_ascii=False,
        indent=4
    )


with open(
    FINAL_OUTPUT,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        records,
        f,
        ensure_ascii=False,
        indent=4
    )


# ============================================================
# PRINT
# ============================================================

print()
print("=" * 125)
print(
    "ALBARAKA TÜRK - "
    "FINAL QUALITY AUDIT V4"
)
print("=" * 125)

print(
    "Toplam kayıt        :",
    len(records)
)

print(
    "Finansman           :",
    len(finance_records)
)

print(
    "Kampanya            :",
    len(campaign_records)
)

print(
    "V4 patch            :",
    len(patches)
)

print(
    "Duplicate URL       :",
    duplicate_url_count
)

print(
    "Duplicate type/name :",
    duplicate_type_name
)

print(
    "Expired campaign    :",
    len(expired)
)

print(
    "Target fragment     :",
    len(target_fragment_errors)
)

print(
    "Validation error    :",
    len(errors)
)

print(
    "Warning             :",
    len(warnings)
)

print(
    "Final JSON          :",
    FINAL_OUTPUT
)


# ============================================================
# PATCH DETAILS
# ============================================================

print()
print("=" * 125)
print("V4 PATCH DETAYLARI")
print("=" * 125)


for p in patches:

    print()
    print(
        p[
            "urun_adi"
        ]
    )

    print(
        "  Alan:",
        p[
            "field"
        ]
    )

    print(
        "  Eski:",
        p[
            "old"
        ]
    )

    print(
        "  Yeni:",
        p[
            "new"
        ]
    )


# ============================================================
# WARNINGS
# ============================================================

if warnings:

    print()
    print("=" * 125)
    print("UYARILAR")
    print("=" * 125)

    for warning in warnings:
        print(
            "-",
            warning
        )


# ============================================================
# ERRORS
# ============================================================

if errors:

    print()
    print("=" * 125)
    print("HATALAR")
    print("=" * 125)

    for error in errors:
        print(
            "-",
            error
        )


print()
print("=" * 125)

if not errors:

    print(
        "SONUÇ: ALBARAKA TÜRK "
        "FINAL QUALITY V4 "
        "65/65 BAŞARILI ✅"
    )

else:

    print(
        "SONUÇ: ALBARAKA TÜRK "
        "FINAL QUALITY V4 "
        "KONTROL GEREKİYOR ❌"
    )

print("=" * 125)


# ============================================================
# DOWNLOAD
# ============================================================

files.download(
    FINANCE_OUTPUT
)

files.download(
    CAMPAIGN_OUTPUT
)

files.download(
    FINAL_OUTPUT
)

ALBARAKA TÜRK - FINAL QUALITY PATCH V4
Input: /content/albaraka_turk_final_quality_v3.json
Kayıt: 65

ALBARAKA TÜRK - FINAL QUALITY AUDIT V4
Toplam kayıt        : 65
Finansman           : 17
Kampanya            : 48
V4 patch            : 6
Duplicate URL       : 0
Duplicate type/name : 0
Expired campaign    : 0
Target fragment     : 0
Validation error    : 0
Final JSON          : /content/albaraka_turk_final_quality_v4.json

V4 PATCH DETAYLARI

Albaraka'da Masraflara Son!
  Alan: masraf_bilgisi
  Eski: ["Albaraka'da Masraflara Son!", 'Albaraka Türk olarak dijitalleşen dünyada müşterilerimizin finansal ihtiyaçlarına çözüm üretmenin tutkusuyla masrafsız bir bankacılık sunuyoruz.', 'Siz de görüntülü görüşme ile hemen Albarakalı olabilir, mobil şube ve internet şube üzerinden işlemlerinizi gerçekleştirebilir, masrafsız bankacılığın keyfini çıkarabilirsiniz.', "Albaraka Türk'te açılacak TL, Döviz veya Altın tüm cari ve katılma hesaplarından hiçbir koşul gerektirmeksizin hesap işletim ücreti 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# ALBARAKA TÜRK
# FINAL QUALITY V5 GENERATOR
#
# Input:
#   /content/albaraka_turk_final_quality_v4.json
#
# Outputs:
#   /content/albaraka_turk_finansman_extracted_v5.json
#   /content/albaraka_turk_kampanya_extracted_v5.json
#   /content/albaraka_turk_final_quality_v5.json
# ============================================================

import json
from collections import Counter
from datetime import datetime, date

from google.colab import files


# ============================================================
# PATHS
# ============================================================

INPUT_FILE = (
    "/content/"
    "albaraka_turk_final_quality_v4.json"
)

FINANCE_OUTPUT = (
    "/content/"
    "albaraka_turk_finansman_extracted_v5.json"
)

CAMPAIGN_OUTPUT = (
    "/content/"
    "albaraka_turk_kampanya_extracted_v5.json"
)

FINAL_OUTPUT = (
    "/content/"
    "albaraka_turk_final_quality_v5.json"
)


BANK_NAME = (
    "Albaraka Türk Katılım Bankası A.Ş."
)

TODAY = date(
    2026,
    8,
    23,
)


# ============================================================
# SCHEMA
# ============================================================

SCHEMA_KEYS = [
    "banka",
    "kayit_turu",
    "urun_adi",
    "urun_kategorisi",
    "kar_payi_orani",
    "finansman_orani",
    "finansman_tutari",
    "vade",
    "taksit_sayisi",
    "masraf_bilgisi",
    "kampanya_turu",
    "kampanya_avantaji",
    "kampanya_suresi",
    "hedef_kitle",
    "para_birimi",
    "kosullar",
    "kaynak_url",
    "ham_metin",
]


LIST_FIELDS = {
    "kar_payi_orani",
    "finansman_orani",
    "finansman_tutari",
    "vade",
    "taksit_sayisi",
    "masraf_bilgisi",
    "kampanya_avantaji",
    "hedef_kitle",
    "para_birimi",
    "kosullar",
}


SCALAR_FIELDS = (
    set(SCHEMA_KEYS)
    -
    LIST_FIELDS
)


# ============================================================
# LOAD
# ============================================================

with open(
    INPUT_FILE,
    "r",
    encoding="utf-8"
) as f:

    records = json.load(f)


if not isinstance(
    records,
    list
):

    raise TypeError(
        "JSON direct list olmalı."
    )


by_name = {
    record["urun_adi"]: record
    for record in records
}


print("=" * 125)
print(
    "ALBARAKA TÜRK - "
    "FINAL QUALITY V5 GENERATOR"
)
print("=" * 125)

print(
    "Input:",
    INPUT_FILE
)

print(
    "Kayıt:",
    len(records)
)


# ============================================================
# PATCH 1
# KONUT FİNANSMANI
# ============================================================

name = "Konut Finansmanı"


by_name[name][
    "masraf_bilgisi"
] = [
    (
        "Konut finansmanı kullanımında "
        "KKDF ve BSMV ödemelerinden "
        "muaf olunabilir."
    )
]


# ============================================================
# PATCH 2
# TAŞIT FİNANSMANI
# ============================================================

name = "Taşıt Finansmanı"


by_name[name][
    "masraf_bilgisi"
] = [
    (
        "Tüzel kişi taşıt finansmanı "
        "kullanımlarında KKDF "
        "muafiyeti uygulanır."
    )
]


# ============================================================
# PATCH 3
# KASKO TARGET FIX
# ============================================================

name = (
    "8 Taksit Fırsatıyla "
    "KASKO Zamanı!"
)


by_name[name][
    "hedef_kitle"
] = [
    (
        "1 Mart 2026 - 31 Aralık 2026 "
        "tarihleri arasında Albaraka Mobil "
        "üzerinden Sigorta İşlemleri "
        "alanından Kasko sigortası alan "
        "bireysel müşteriler"
    )
]


broken_kasko = (
    "tarihleri arasında Albaraka Mobil'den "
    "“Sigorta İşlemleri” alanından Kasko "
    "sigorta alımı gerçekleştiren "
    "müşteriler faydalanabilir."
)


by_name[name][
    "kosullar"
] = [
    condition
    for condition
    in by_name[name]["kosullar"]
    if condition != broken_kasko
]


complete_kasko = (
    "1 Mart 2026 - 31 Aralık 2026 "
    "tarihleri arasında Albaraka Mobil'den "
    "Sigorta İşlemleri alanından Kasko "
    "sigortası alımı gerçekleştiren "
    "müşteriler faydalanabilir."
)


if (
    complete_kasko
    not in by_name[name]["kosullar"]
):

    by_name[name][
        "kosullar"
    ].append(
        complete_kasko
    )


# ============================================================
# PATCH 4
# SATIRDAN SANATA TARGET FIX
# ============================================================

name = (
    "Satırdan Sanata, Ek %10 İndirim "
    "ve Vade Farksız 4 Taksit "
    "Albaraka'da!"
)


by_name[name][
    "hedef_kitle"
] = [
    (
        "Albaraka Mobil üzerinden alınan "
        "indirim kodunu "
        "albarakakultur.com.tr üzerindeki "
        "Hediye Çeki Kullan alanına giren "
        "bireysel müşteriler"
    )
]


broken_satir = (
    "alanına giren müşteriler "
    "faydalanabilir."
)


by_name[name][
    "kosullar"
] = [
    condition
    for condition
    in by_name[name]["kosullar"]
    if condition != broken_satir
]


complete_satir = (
    "Albaraka Mobil üzerinden alınan kodu "
    "albarakakultur.com.tr üzerindeki "
    "Hediye Çeki Kullan alanına giren "
    "müşteriler kampanyadan faydalanabilir."
)


if (
    complete_satir
    not in by_name[name]["kosullar"]
):

    by_name[name][
        "kosullar"
    ].append(
        complete_satir
    )


# ============================================================
# SPLIT
# ============================================================

finance_records = [
    record
    for record in records
    if record[
        "kayit_turu"
    ] == "finansman"
]


campaign_records = [
    record
    for record in records
    if record[
        "kayit_turu"
    ] == "kampanya"
]


# ============================================================
# VALIDATION
# ============================================================

errors = []
warnings = []


# ------------------------------------------------------------
# COUNTS
# ------------------------------------------------------------

if len(records) != 65:

    errors.append(
        (
            f"Toplam kayıt "
            f"{len(records)} != 65"
        )
    )


if len(
    finance_records
) != 17:

    errors.append(
        (
            f"Finansman "
            f"{len(finance_records)} != 17"
        )
    )


if len(
    campaign_records
) != 48:

    errors.append(
        (
            f"Kampanya "
            f"{len(campaign_records)} != 48"
        )
    )


# ============================================================
# SCHEMA + TYPE CHECK
# ============================================================

for record in records:

    name = record[
        "urun_adi"
    ]


    if list(
        record.keys()
    ) != SCHEMA_KEYS:

        errors.append(
            (
                f"{name} -> "
                "schema/order hatası"
            )
        )


    for field in LIST_FIELDS:

        if not isinstance(
            record[field],
            list
        ):

            errors.append(
                (
                    f"{name} -> "
                    f"{field} list değil"
                )
            )


    for field in SCALAR_FIELDS:

        if not isinstance(
            record[field],
            str
        ):

            errors.append(
                (
                    f"{name} -> "
                    f"{field} string değil"
                )
            )


    if (
        record[
            "banka"
        ]
        != BANK_NAME
    ):

        errors.append(
            (
                f"{name} -> "
                "banka yanlış"
            )
        )


    if record[
        "kayit_turu"
    ] not in {
        "finansman",
        "kampanya",
    }:

        errors.append(
            (
                f"{name} -> "
                "kayit_turu yanlış"
            )
        )


    for field in [
        "urun_adi",
        "kaynak_url",
        "ham_metin",
    ]:

        if not record[
            field
        ]:

            errors.append(
                (
                    f"{name} -> "
                    f"{field} boş"
                )
            )


    if (
        "TRY"
        in record[
            "para_birimi"
        ]
    ):

        errors.append(
            (
                f"{name} -> "
                "TRY bulundu"
            )
        )


# ============================================================
# DUPLICATE CHECK
# ============================================================

urls = [
    record[
        "kaynak_url"
    ].rstrip("/")
    for record
    in records
]


duplicate_url = (
    len(urls)
    -
    len(set(urls))
)


if duplicate_url:

    errors.append(
        (
            "Duplicate URL: "
            f"{duplicate_url}"
        )
    )


type_name_keys = [
    (
        record[
            "kayit_turu"
        ],
        record[
            "urun_adi"
        ],
    )
    for record
    in records
]


duplicate_type_name = (
    len(type_name_keys)
    -
    len(set(type_name_keys))
)


if duplicate_type_name:

    errors.append(
        (
            "Duplicate type/name: "
            f"{duplicate_type_name}"
        )
    )


# ============================================================
# CAMPAIGN DATE CHECK
# ============================================================

expired = []


for record in campaign_records:

    period = record[
        "kampanya_suresi"
    ]


    if not period:

        continue


    try:

        start_text, end_text = [
            x.strip()
            for x in period.split(
                " - "
            )
        ]


        end_date = datetime.strptime(
            end_text,
            "%d.%m.%Y"
        ).date()


        if end_date < TODAY:

            expired.append(
                record[
                    "urun_adi"
                ]
            )


    except Exception:

        errors.append(
            (
                f"{record['urun_adi']} -> "
                "kampanya_suresi format hatası"
            )
        )


if expired:

    errors.append(
        (
            "Expired campaign: "
            f"{expired}"
        )
    )


# ============================================================
# CRITICAL VALUES
# ============================================================

def require(
    name,
    field,
    expected,
):

    actual = (
        by_name[
            name
        ][
            field
        ]
    )


    if actual != expected:

        errors.append(
            (
                f"{name} -> "
                f"{field}: "
                f"{actual} "
                f"!= {expected}"
            )
        )


# ------------------------------------------------------------
# Togg
# ------------------------------------------------------------

require(
    "Togg Finansmanı",
    "kar_payi_orani",
    [
        "%0",
        "%2,99",
        "%3,05",
    ]
)


require(
    "Togg Finansmanı",
    "finansman_orani",
    ["%70"]
)


require(
    "Togg Finansmanı",
    "finansman_tutari",
    [
        "800.000 TL",
        "1.000.000 TL",
        "1.500.000 TL",
        "1.700.000 TL",
    ]
)


require(
    "Togg Finansmanı",
    "vade",
    [
        "10 ay",
        "12 ay",
        "36 ay",
        "48 ay",
    ]
)


# ------------------------------------------------------------
# Konut
# ------------------------------------------------------------

require(
    "Konut Finansmanı",
    "finansman_orani",
    [
        "%90",
        "%80",
        "%70",
        "%60",
        "%50",
        "%40",
        "%30",
        "%22,5",
        "%20",
        "%17,5",
        "%15",
        "%12,5",
        "%10",
        "%7,5",
        "%5",
    ]
)


require(
    "Konut Finansmanı",
    "masraf_bilgisi",
    [
        (
            "Konut finansmanı kullanımında "
            "KKDF ve BSMV ödemelerinden "
            "muaf olunabilir."
        )
    ]
)


# ------------------------------------------------------------
# Taşıt
# ------------------------------------------------------------

require(
    "Taşıt Finansmanı",
    "masraf_bilgisi",
    [
        (
            "Tüzel kişi taşıt finansmanı "
            "kullanımlarında KKDF "
            "muafiyeti uygulanır."
        )
    ]
)


# ------------------------------------------------------------
# Taksitlio
# ------------------------------------------------------------

require(
    "Taksitlio.com Alışveriş Finansmanı",
    "kar_payi_orani",
    ["%2,99"]
)


require(
    "Taksitlio.com Alışveriş Finansmanı",
    "finansman_tutari",
    ["150.000 TL"]
)


require(
    "Taksitlio.com Alışveriş Finansmanı",
    "vade",
    ["6 ay"]
)


# ------------------------------------------------------------
# Dijital Katılma
# ------------------------------------------------------------

require(
    (
        "Dijital Katılma Hesabı'na "
        "Özel Paylaşım Oranları!"
    ),
    "kar_payi_orani",
    []
)


# ============================================================
# BROKEN TARGET CHECK
# ============================================================

broken_targets = []


for record in campaign_records:

    for target in record[
        "hedef_kitle"
    ]:

        target = target.strip()


        if (
            target.startswith(
                "tarihleri arasında"
            )
            or
            target.startswith(
                "alanına giren"
            )
            or
            target.endswith(",")
            or
            target.endswith(";")
        ):

            broken_targets.append(
                (
                    record[
                        "urun_adi"
                    ],
                    target,
                )
            )


if broken_targets:

    errors.append(
        (
            "Broken hedef_kitle: "
            f"{broken_targets}"
        )
    )


# ============================================================
# FINANCE → CAMPAIGN LEAK CHECK
# ============================================================

for record in finance_records:

    name = record[
        "urun_adi"
    ]


    if record[
        "kampanya_turu"
    ]:

        errors.append(
            (
                f"{name} -> "
                "kampanya_turu dolu"
            )
        )


    if record[
        "kampanya_avantaji"
    ]:

        errors.append(
            (
                f"{name} -> "
                "kampanya_avantaji dolu"
            )
        )


    if record[
        "kampanya_suresi"
    ]:

        errors.append(
            (
                f"{name} -> "
                "kampanya_suresi dolu"
            )
        )


# ============================================================
# KNOWN SOURCE CONFLICT
# ============================================================

jet = by_name[
    "Jet Finansman"
]


if (
    "60.000 TL"
    in jet[
        "ham_metin"
    ]
    and
    "100.001 TL"
    in jet[
        "ham_metin"
    ]
):

    warnings.append(
        (
            "Jet Finansman: resmi kaynak "
            "kendi içinde tutarsız. "
            "Maksimum 60.000 TL ifadesi ile "
            "100.001 TL ve üzeri vade bandı "
            "aynı sayfada yer alıyor. "
            "finansman_tutari bilerek boş."
        )
    )


# ============================================================
# SAVE
# ============================================================

with open(
    FINANCE_OUTPUT,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        finance_records,
        f,
        ensure_ascii=False,
        indent=4
    )


with open(
    CAMPAIGN_OUTPUT,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        campaign_records,
        f,
        ensure_ascii=False,
        indent=4
    )


with open(
    FINAL_OUTPUT,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        records,
        f,
        ensure_ascii=False,
        indent=4
    )


# ============================================================
# FINAL REPORT
# ============================================================

print()
print("=" * 125)
print(
    "ALBARAKA TÜRK - "
    "FINAL QUALITY AUDIT V5"
)
print("=" * 125)

print(
    "Toplam kayıt        :",
    len(records)
)

print(
    "Finansman           :",
    len(finance_records)
)

print(
    "Kampanya            :",
    len(campaign_records)
)

print(
    "Duplicate URL       :",
    duplicate_url
)

print(
    "Duplicate type/name :",
    duplicate_type_name
)

print(
    "Expired campaign    :",
    len(expired)
)

print(
    "Broken hedef_kitle  :",
    len(broken_targets)
)

print(
    "Validation error    :",
    len(errors)
)

print(
    "Warning             :",
    len(warnings)
)

print(
    "Final JSON          :",
    FINAL_OUTPUT
)


if warnings:

    print()
    print("=" * 125)
    print("UYARILAR")
    print("=" * 125)

    for warning in warnings:

        print(
            "-",
            warning
        )


if errors:

    print()
    print("=" * 125)
    print("HATALAR")
    print("=" * 125)

    for error in errors:

        print(
            "-",
            error
        )


print()
print("=" * 125)

if not errors:

    print(
        "SONUÇ: ALBARAKA TÜRK "
        "FINAL QUALITY V5 "
        "65/65 BAŞARILI ✅"
    )

else:

    print(
        "SONUÇ: ALBARAKA TÜRK "
        "FINAL QUALITY V5 "
        "KONTROL GEREKİYOR ❌"
    )

print("=" * 125)


# ============================================================
# DOWNLOAD
# ============================================================

files.download(
    FINANCE_OUTPUT
)

files.download(
    CAMPAIGN_OUTPUT
)

files.download(
    FINAL_OUTPUT
)

ALBARAKA TÜRK - FINAL QUALITY V5 GENERATOR
Input: /content/albaraka_turk_final_quality_v4.json
Kayıt: 65

ALBARAKA TÜRK - FINAL QUALITY AUDIT V5
Toplam kayıt        : 65
Finansman           : 17
Kampanya            : 48
Duplicate URL       : 0
Duplicate type/name : 0
Expired campaign    : 0
Broken hedef_kitle  : 0
Validation error    : 0
Final JSON          : /content/albaraka_turk_final_quality_v5.json

UYARILAR
- Jet Finansman: resmi kaynak kendi içinde tutarsız. Maksimum 60.000 TL ifadesi ile 100.001 TL ve üzeri vade bandı aynı sayfada yer alıyor. finansman_tutari bilerek boş.

SONUÇ: ALBARAKA TÜRK FINAL QUALITY V5 65/65 BAŞARILI ✅


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>